In [0]:
# Updated copy: publishes Unity Catalog table and column comments for PREBORN outputs.

import re

def _norm_sql(x):
    return f"coalesce(nullif(trim(cast({x} as string)), ''), '~')"

def mid(grain, site, *part_sqls):
    site_lit = "'" + str(site).replace("'", "''") + "'"
    inner = ", ".join([f"'{grain}'", _norm_sql(site_lit)] + [_norm_sql(p) for p in part_sqls])
    return f"conv(substr(sha2(concat_ws('|', {inner}), 256), 1, 15), 16, 10)"

def assert_mint_parity(site):
    import hashlib
    k = f"pregnancy|{site}|TESTKEY"
    expect = int(hashlib.sha256(k.encode()).hexdigest()[:15], 16)
    got = spark.sql(f"SELECT {mid('pregnancy', site, chr(39)+'TESTKEY'+chr(39))} AS v").first()["v"]
    assert int(got) == expect, f"MINT PARITY BROKEN: {got} != {expect}"
    print("mint parity OK for site", site)


In [0]:
# === DDL: ensure_core_schema — ported verbatim from sp1_00_config (person/pregnancy/infant) ===
# NOTE: sp1_00_config only defined person/pregnancy/infant; the staged sp1_85 built the 4 domain
# tables + visit_occurrence via CREATE OR REPLACE (schema implicit). The consolidated build INSERTs
# with EXPLICIT column lists, so those tables' DDL is added HERE (Task 4) with column ORDER/TYPES
# matching the _core_persist / _core_visit SELECT output exactly.

def ensure_core_schema(schema):
    S = schema
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S}")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.person (
      person_id BIGINT, person_source_value STRING, person_role STRING,
      gender_source_value STRING, birth_datetime TIMESTAMP,
      race_concept_id BIGINT, race_source_value STRING, site_id STRING
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.pregnancy (
      person_id BIGINT, pregnancy_id BIGINT,
      pregnancy_start_date DATE, pregnancy_end_date DATE,
      gestational_length_in_day INT,
      pregnancy_outcome_concept_id BIGINT, pregnancy_mode_delivery_concept_id BIGINT,
      pregnancy_single_concept_id BIGINT, pregnancy_marital_status_concept_id BIGINT,
      pregnancy_number_fetuses INT, pregnancy_number_liveborn INT,
      prev_pregnancy_parity_concept_id BIGINT, prev_pregnancy_gravidity INT,
      prev_livebirth_number INT, prev_still_misc_number INT,
      pre_pregnancy_bmi DOUBLE, pregnancy_folic_concept_id BIGINT,
      pregnancy_outcome_source_value STRING, pregnancy_mode_delivery_source_value STRING,
      site_id STRING
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.infant (
      pregnancy_id BIGINT, infant_id BIGINT,
      birth_outcome_concept_id BIGINT, birth_weight DOUBLE,
      birth_con_malformation_concept_id BIGINT, birth_apgar INT,
      labour_delivery_id STRING, site_id STRING
    ) USING DELTA""")

    # --- domain fact tables (persisted by _core_persist from _all_facts) ---
    # Column ORDER/TYPES must match the persist SELECT output exactly:
    #   (<id>, person_id, <concept>, <date>, pregnancy_id, source_table,
    #    <source_value>, <source_concept_id>, multi_map_flag, [value_as_number, unit_source_value], site_id)
    # condition_occurrence / procedure_occurrence: with_value=False (no value_as_number/unit).
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.condition_occurrence (
      condition_occurrence_id BIGINT, person_id BIGINT, condition_concept_id BIGINT,
      condition_start_date DATE, pregnancy_id BIGINT, source_table STRING,
      condition_source_value STRING, condition_source_concept_id BIGINT,
      multi_map_flag INT, site_id STRING
    ) USING DELTA""")

    # measurement / observation: with_value=True -> value_as_number + unit_source_value.
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.measurement (
      measurement_id BIGINT, person_id BIGINT, measurement_concept_id BIGINT,
      measurement_date DATE, pregnancy_id BIGINT, source_table STRING,
      measurement_source_value STRING, measurement_source_concept_id BIGINT,
      multi_map_flag INT, value_as_number DOUBLE, unit_source_value STRING, site_id STRING
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.observation (
      observation_id BIGINT, person_id BIGINT, observation_concept_id BIGINT,
      observation_date DATE, pregnancy_id BIGINT, source_table STRING,
      observation_source_value STRING, observation_source_concept_id BIGINT,
      multi_map_flag INT, value_as_number DOUBLE, unit_source_value STRING, site_id STRING
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.procedure_occurrence (
      procedure_occurrence_id BIGINT, person_id BIGINT, procedure_concept_id BIGINT,
      procedure_date DATE, pregnancy_id BIGINT, source_table STRING,
      procedure_source_value STRING, procedure_source_concept_id BIGINT,
      multi_map_flag INT, site_id STRING
    ) USING DELTA""")

    # visit_occurrence from msd201 care-contacts (persisted by _core_visit).
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.visit_occurrence (
      visit_occurrence_id BIGINT, pregnancy_id BIGINT, visit_start_date DATE,
      visit_source_value STRING, site_id STRING
    ) USING DELTA""")


In [0]:
# === DDL: ensure_l3_schema — ported verbatim from sp2_00_config ===

def ensure_l3_schema(schema_l3):
    S3 = schema_l3
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S3}")

    # pregnancy-as-condition additions (union-compatible with <core>.condition_occurrence
    # on the columns a consumer needs; L3-owned ids on the pregnancy-condition grain)
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S3}.condition_occurrence (
      condition_occurrence_id BIGINT, person_id BIGINT, condition_concept_id BIGINT,
      condition_start_date DATE, condition_end_date DATE, condition_type_concept_id BIGINT, pregnancy_id BIGINT,
      condition_source_value STRING, site_id STRING) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S3}.fact_relationship (
      domain_concept_id_1 BIGINT, fact_id_1 BIGINT,
      domain_concept_id_2 BIGINT, fact_id_2 BIGINT,
      relationship_concept_id BIGINT, site_id STRING) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S3}.episode (
      episode_id BIGINT, person_id BIGINT, episode_concept_id BIGINT,
      episode_start_date DATE, episode_end_date DATE, episode_object_concept_id BIGINT,
      pregnancy_id BIGINT, site_id STRING) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S3}.episode_event (
      episode_id BIGINT, event_id BIGINT, episode_event_field_concept_id BIGINT,
      source_table STRING, site_id STRING) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S3}.person_race_l3 (
      person_id BIGINT, l3_race_concept_id BIGINT, nhs_letter STRING, site_id STRING) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S3}.observation (
      observation_id BIGINT, person_id BIGINT, observation_concept_id BIGINT,
      value_as_concept_id BIGINT, observation_date DATE,
      observation_type_concept_id BIGINT, observation_source_value STRING, site_id STRING) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S3}._l3_qa (
      check_name STRING, metric STRING, value BIGINT, detail STRING) USING DELTA""")


In [0]:
# === DDL: ensure_enrich_schema — ported verbatim from sp3_00_config ===
# Keeps the ADD COLUMNS schema-check pattern exactly (ADD COLUMN IF NOT EXISTS is INVALID Delta):
# check the existing schema and only ADD COLUMNS that are missing.

def ensure_enrich_schema(schema_enrich, core_schema):
    S = core_schema; E = schema_enrich

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {E}")

    # --- provenance markers on the common core (idempotent) ---
    # NOTE: Delta 'ALTER TABLE ... ADD COLUMNS' has no IF NOT EXISTS clause, so we
    # check the existing schema and only add columns that are missing.
    for tbl, cols in [
        ("pregnancy", ["pre_pregnancy_bmi_source STRING",
                       "pregnancy_marital_status_source_value STRING"]),
        ("infant",    ["birth_weight_source STRING", "birth_apgar_source STRING"]),
    ]:
        existing = {f.name.lower() for f in spark.table(f"{S}.{tbl}").schema.fields}
        for col in cols:
            cname = col.split()[0].lower()
            if cname not in existing:
                spark.sql(f"ALTER TABLE {S}.{tbl} ADD COLUMNS ({col})")

    # --- baby bridge (linkage; site_id, tag-exempt) ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {E}.baby_bridge (
      site_id STRING, minted_infant_id BIGINT, minted_pregnancy_id BIGINT,
      cerner_pregnancy_id BIGINT, cerner_baby_person_id BIGINT,
      cerner_pregnancy_child_id STRING, birth_order INT, msds_birth_order INT,
      link_method STRING, nnu_entity_id STRING, baby_nhs STRING
    ) USING DELTA""")

    # --- enrichment fact tables (site_id + enrichment_tag) ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {E}.measurement (
      measurement_id BIGINT, person_id BIGINT, measurement_concept_id BIGINT,
      measurement_date DATE, pregnancy_id BIGINT, infant_id BIGINT,
      source_table STRING, measurement_source_value STRING,
      measurement_source_concept_id BIGINT, value_as_number DOUBLE,
      unit_source_value STRING, enrichment_tag STRING, site_id STRING
    ) USING DELTA""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {E}.observation (
      observation_id BIGINT, person_id BIGINT, observation_concept_id BIGINT,
      observation_date DATE, pregnancy_id BIGINT, infant_id BIGINT,
      source_table STRING, observation_source_value STRING,
      observation_source_concept_id BIGINT, value_as_number DOUBLE,
      value_as_string STRING, enrichment_tag STRING, site_id STRING
    ) USING DELTA""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {E}.procedure_occurrence (
      procedure_occurrence_id BIGINT, person_id BIGINT, procedure_concept_id BIGINT,
      procedure_date DATE, pregnancy_id BIGINT, infant_id BIGINT,
      source_table STRING, procedure_source_value STRING,
      procedure_source_concept_id BIGINT, enrichment_tag STRING, site_id STRING
    ) USING DELTA""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {E}.condition_occurrence (
      condition_occurrence_id BIGINT, person_id BIGINT, condition_concept_id BIGINT,
      condition_start_date DATE, pregnancy_id BIGINT, infant_id BIGINT,
      source_table STRING, condition_source_value STRING,
      condition_source_concept_id BIGINT, enrichment_tag STRING, site_id STRING
    ) USING DELTA""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {E}.visit_detail (
      visit_detail_id BIGINT, person_id BIGINT, visit_detail_concept_id BIGINT,
      visit_detail_start_date DATE, visit_detail_end_date DATE,
      pregnancy_id BIGINT, infant_id BIGINT, source_table STRING,
      visit_detail_source_value STRING, enrichment_tag STRING, site_id STRING
    ) USING DELTA""")

    # --- QA tables (site_id, tag-exempt) ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {E}._gap_status (
      gap_id STRING, source STRING, disposition STRING, evidence STRING, site_id STRING
    ) USING DELTA""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {E}._enrich_qa (
      metric STRING, value STRING, site_id STRING
    ) USING DELTA""")


In [0]:
# === CROSSWALKS — value-level constants used by msds_core_transform / enrichment ===
# RACE_CASE: NHS Data Dictionary ethnic category (A–Z) -> standard OMOP Race concept.
# D–G are Mixed categories. OMOP has no standard Mixed race concept, so they must remain
# unmapped (0), with the exact NHS category preserved in race_source_value and the L3
# observation. S 'Any other ethnic group' and Z 'Not stated' also map to 0. Applied to BOTH
# mother and baby. A new site may pass race_case=<own CASE expr> after local verification.
RACE_CASE = """CASE upper(trim({col}))
  WHEN 'A' THEN 8527 WHEN 'B' THEN 8527 WHEN 'C' THEN 8527
  WHEN 'D' THEN 0 WHEN 'E' THEN 0 WHEN 'F' THEN 0 WHEN 'G' THEN 0
  WHEN 'H' THEN 8515 WHEN 'L' THEN 8515
  WHEN 'J' THEN 38003589 WHEN 'K' THEN 38003575 WHEN 'M' THEN 38003598
  WHEN 'N' THEN 38003600 WHEN 'P' THEN 38003598 WHEN 'R' THEN 38003579
  ELSE 0 END"""

# MARITAL_CASE: Cerner mill_person MARITAL_TYPE_CD -> OMOP marital-status concept, captured
# from sp3_20_fill_bmi_marital. Uses a {col} placeholder for the source column
# (e.g. MARITAL_CASE.format(col='mpr.MARITAL_TYPE_CD')).
MARITAL_CASE = """CASE {col}
  WHEN 309239 THEN 4053842
  WHEN 309237 THEN 4338692
  WHEN 309241 THEN 4143188
  WHEN 309236 THEN 4069297
  WHEN 309238 THEN 4027529
  ELSE 0 END"""


In [0]:
# === seed_mapping_spec — ported verbatim from sp1_60_mapping_spec_seed (S -> schema param) ===
# msds_omop_map is a site-INDEPENDENT lookup rebuilt each run, so the TRUNCATE is safe here.

def seed_mapping_spec(schema):
    S = schema

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.msds_omop_map (
      rule_id STRING, msds_table STRING, msds_field STRING, source_value STRING,
      source_vocabulary STRING, is_field_mapping INT, creation_type STRING,
      target_omop_table STRING, target_domain STRING,
      target_concept_id BIGINT, source_concept_id BIGINT,
      mapping_type STRING, output_grain STRING, output_role STRING,
      value_field STRING, unit_field STRING, unit_concept_id BIGINT,
      relationship_concept_id BIGINT, barts_convention_ref STRING,
      mireda_divergence STRING, notes STRING
    ) USING DELTA""")

    # FIELD-LEVEL default rules (source_value NULL): one per (table, code field) so EVERY coded
    # row matches a rule (LEFT-join-to-map safety) and routes to its OMOP domain. Concept
    # resolution is by OMOP vocabulary at emit time (target_concept_id NULL here = vocab-driven).
    # target_domain is the vocab domain we EXPECT (per-value routing still overrides at emit).
    spark.sql(f"TRUNCATE TABLE {S}.msds_omop_map")
    rows = [
     # diagnosis-style (code + scheme) -> Condition
     ("msd106_diag","msd106diagnosispreg","DIAG","SNOMED","condition_occurrence","Condition","one-per-diagnosis"),
     ("msd404_diag","msd404diagneo","DIAG","SNOMED","condition_occurrence","Condition","one-per-neonatal-diagnosis"),
     ("msd107_prevdiag","msd107medhistory","PREVDIAG","SNOMED","condition_occurrence","Condition","one-per-medhistory"),
     # findings/obs (value + unit) -> Measurement (numeric) ; obs axis
     ("msd109_obs","msd109findobsmother","OBSCODE","SNOMED","measurement","Measurement","one-per-observation"),
     ("msd109_finding","msd109findobsmother","FINDINGCODE","SNOMED","observation","Observation","one-per-finding"),
     # care-activity 3-axis tables (202/302/405): procedure / finding / obs axes
     ("msd202_proc","msd202careactivitypreg","PROCEDURECODE","SNOMED","procedure_occurrence","Procedure","one-per-activity"),
     ("msd202_finding","msd202careactivitypreg","FINDINGCODE","SNOMED","observation","Observation","one-per-activity"),
     ("msd202_obs","msd202careactivitypreg","OBSCODE","SNOMED","measurement","Measurement","one-per-activity"),
     ("msd302_proc","msd302careactlabdel","PROCEDURECODE","SNOMED","procedure_occurrence","Procedure","one-per-activity"),
     ("msd302_finding","msd302careactlabdel","FINDINGCODE","SNOMED","observation","Observation","one-per-activity"),
     ("msd302_obs","msd302careactlabdel","OBSCODE","SNOMED","measurement","Measurement","one-per-activity"),
     ("msd405_proc","msd405careactbaby","PROCEDURECODE","SNOMED","procedure_occurrence","Procedure","one-per-activity"),
     ("msd405_finding","msd405careactbaby","FINDINGCODE","SNOMED","observation","Observation","one-per-activity"),
     ("msd405_obs","msd405careactbaby","OBSCODE","SNOMED","measurement","Measurement","one-per-activity"),
    ]
    vals = ",\n".join([
      f"('{r[0]}','{r[1]}','{r[2]}',NULL,'{r[3]}',1,'M','{r[4]}','{r[5]}',NULL,NULL,'concept-lookup','{r[6]}',NULL,NULL,NULL,NULL,NULL,'barts:{r[4]}',NULL,'field-default rule; vocab-resolved')"
      for r in rows])
    spark.sql(f"INSERT INTO {S}.msds_omop_map VALUES\n{vals}")

    # CURATED value-level rules (creation_type 'C'): deprecated SNOMED codes the interpreter
    # CANNOT resolve (no valid standard, no 'Maps to', no replacement chain in the vocab),
    # each mapped by hand to the VERIFIED current standard concept (name+domain checked live
    # 2026-07-07). Value-level rules WIN over the field default (matched-CTE precedence in
    # 80_facts) and carry their own target table/domain. These are all Condition/Observation
    # clinical concepts, so they route to condition_occurrence / observation regardless of which
    # raw axis (proc/finding/obs) they arrive on — a headache is a Condition, not a measurement.
    # (target_omop_table, target_domain, target_concept_id, human-readable note)
    CURATED = [
      # obstetric — deprecated "tear during delivery" -> current perineal laceration standards
      ("199916005","condition_occurrence","Condition",197343, "First degree perineal tear during delivery -> First degree perineal laceration"),
      ("199925004","condition_occurrence","Condition",198492, "Second degree perineal tear during delivery -> Second degree perineal laceration"),
      ("199930000","condition_occurrence","Condition",193830, "Third degree perineal tear during delivery -> Third degree perineal laceration"),
      ("199607009","observation","Observation",4079844, "Intrauterine death - delivered -> Fetal death"),
      ("199049003","condition_occurrence","Condition",4273560, "Threatened premature labor - not delivered -> Premature labor"),
      ("199783004","condition_occurrence","Condition",72377,   "Shoulder dystocia - delivered -> Shoulder girdle dystocia"),
      # symptoms / complaints (C/O ...) -> current standard clinical concept
      ("272027003","condition_occurrence","Condition",378253,  "C/O a headache -> Headache"),
      ("272044004","condition_occurrence","Condition",441408,  "C/O vomiting -> Vomiting"),
      ("275570002","condition_occurrence","Condition",4129155, "Complaining of per vaginam bleeding -> Vaginal bleeding"),
      ("162415008","condition_occurrence","Condition",140214,  "C/O a rash -> Eruption"),
      ("268911002","condition_occurrence","Condition",140214,  "O/E rash present -> Eruption"),
      # administrative / safeguarding
      ("281399006","observation","Observation",4259518, "Did not attend -> Did not attend"),
      ("1077911000000105","observation","Observation",605010, "Safeguarding -> Safeguarding concern"),
      ("887041000000104","observation","Observation",605010, "Safeguarding issues -> Safeguarding concern"),
    ]
    cur_vals = []
    for r in rows:
        for code, tgt_tbl, tgt_dom, cid, note in CURATED:
            cur_vals.append(
              f"('{r[0]}__v{code}','{r[1]}','{r[2]}','{code}','SNOMED',0,'C',"
              f"'{tgt_tbl}','{tgt_dom}',{cid},NULL,'curated','{r[6]}',"
              f"NULL,NULL,NULL,NULL,NULL,'barts:{tgt_tbl}',NULL,"
              f"'curated (no vocab path): {note}')")
    spark.sql(f"INSERT INTO {S}.msds_omop_map VALUES\n" + ",\n".join(cur_vals))
    print("map rows", spark.table(f"{S}.msds_omop_map").count(),
          "curated", spark.sql(f"SELECT count(*) c FROM {S}.msds_omop_map WHERE creation_type='C'").first()["c"])


In [0]:
# === resolve_snomed — ported verbatim from sp1_70_resolve_snomed (S -> schema param) ===
# persist a SNOMED code -> (resolution) lookup once (§4.3 order). Materialized so each
# Stage-B emitter joins a small table, not the full vocab every run. Resolution surface per
# code: standard-as-is / Maps-to list / REPLACEMENT-CHAIN fallback (deprecated concepts with
# no Maps-to but a 'Concept replaced by'-family link whose target lands on a standard concept,
# directly or via its own Maps-to). Verified live 2026-07-07: recovers 128 dead codes (~2k rows)
# the plain Maps-to path returns 0 for.

def resolve_snomed(schema):
    S = schema

    spark.sql(f"""
    CREATE OR REPLACE TABLE {S}._resolved_snomed AS
    WITH src AS (
      SELECT concept_code, concept_id AS src_concept_id, standard_concept, domain_id AS src_domain,
             (invalid_reason IS NULL) AS is_valid
      FROM 4_prod.omop.concept WHERE vocabulary_id = 'SNOMED'
    ),
    m AS (
      SELECT cr.concept_id_1 AS from_id,
             array_sort(collect_set(CAST(cr.concept_id_2 AS BIGINT))) AS maps_to
      FROM 4_prod.omop.concept_relationship cr
      WHERE cr.relationship_id = 'Maps to' AND cr.invalid_reason IS NULL
      GROUP BY cr.concept_id_1
    ),
    -- replacement chain: pick ONE deterministic replacement per dead concept
    -- (relationship strength order, then lowest concept_id)
    rep AS (
      SELECT from_id, rep_id FROM (
        SELECT cr.concept_id_1 AS from_id, cr.concept_id_2 AS rep_id,
               row_number() OVER (PARTITION BY cr.concept_id_1 ORDER BY
                 CASE cr.relationship_id
                   WHEN 'Concept replaced by' THEN 0 WHEN 'Concept same_as to' THEN 1
                   WHEN 'Concept was_a to'    THEN 2 WHEN 'Concept poss_eq to' THEN 3
                   ELSE 4 END,
                 cr.concept_id_2) AS rk
        FROM 4_prod.omop.concept_relationship cr
        WHERE cr.relationship_id IN ('Concept replaced by','Concept same_as to',
                                     'Concept was_a to','Concept poss_eq to','Concept alt_to to')
          AND cr.invalid_reason IS NULL
      ) WHERE rk = 1
    ),
    -- land the replacement on a STANDARD concept: itself if standard+valid, else its min Maps-to
    rep_final AS (
      SELECT r.from_id,
             CASE WHEN c2.standard_concept = 'S' AND c2.invalid_reason IS NULL THEN r.rep_id
                  ELSE mt.min_maps END AS replaced_to,
             CASE WHEN c2.standard_concept = 'S' AND c2.invalid_reason IS NULL THEN c2.domain_id
                  ELSE cmt.domain_id END AS replaced_domain
      FROM rep r
      JOIN 4_prod.omop.concept c2 ON c2.concept_id = r.rep_id
      LEFT JOIN (
        SELECT concept_id_1, min(concept_id_2) AS min_maps
        FROM 4_prod.omop.concept_relationship
        WHERE relationship_id = 'Maps to' AND invalid_reason IS NULL
        GROUP BY concept_id_1
      ) mt ON mt.concept_id_1 = r.rep_id
      LEFT JOIN 4_prod.omop.concept cmt ON cmt.concept_id = mt.min_maps
    )
    SELECT s.concept_code, s.src_concept_id, s.standard_concept, s.src_domain, s.is_valid,
           m.maps_to, rf.replaced_to, rf.replaced_domain
    FROM src s
    LEFT JOIN m ON m.from_id = s.src_concept_id
    LEFT JOIN rep_final rf ON rf.from_id = s.src_concept_id
    """)
    n = spark.table(f"{S}._resolved_snomed").count()
    d = spark.sql(f"SELECT count(DISTINCT concept_code) c FROM {S}._resolved_snomed").first()["c"]
    r = spark.sql(f"SELECT count(*) c FROM {S}._resolved_snomed WHERE replaced_to IS NOT NULL AND size(coalesce(maps_to, array()))=0").first()["c"]
    print("resolved_snomed rows", n, "distinct codes", d, "replacement-only recoveries", r)


In [0]:
# === msds_core_transform — portable MSDS-only core (person/pregnancy/infant + facts/L3) ===
# Absorbs sp1_10/20/30 (part A: this build) and sp1_80/85 + sp2_10/20/30/40 (part B: Task 4).
# Site/schema parameterised; raw table names come via the `src` dict; every site-stamped CDM
# table uses DELETE-by-site + explicit-column INSERT (pool-safe) instead of TRUNCATE/OVERWRITE/
# CREATE OR REPLACE. Transient working tables keep CREATE OR REPLACE. Mint output is byte-identical
# to the staged sp1_* (m(...) is the site-bound rename of mid(...)).

def msds_core_transform(site, src, tgt, race_case=None):
    if race_case is None:
        race_case = RACE_CASE
    site_esc = str(site).replace("'", "''")
    tgt_l3 = tgt + "_l3"
    def m(grain, *p):              # local site-bound mint (ported SQL calls m(...) not mid(...))
        return mid(grain, site, *p)
    ensure_core_schema(tgt)
    ensure_l3_schema(tgt_l3)
    seed_mapping_spec(tgt)
    resolve_snomed(tgt)
    # --- part A: person / pregnancy / infant (this task) ---
    _core_person(site, site_esc, src, tgt, m, race_case)
    _core_pregnancy(site, site_esc, src, tgt, m)
    _core_infant(site, site_esc, src, tgt, m, race_case)
    # --- part B (Task 4): facts, persist, visit, L3 ---
    _core_facts(site, site_esc, src, tgt, m)
    _core_persist(site, site_esc, tgt, m)
    _core_visit(site, site_esc, src, tgt, m)
    _l3_additions(site, site_esc, tgt, tgt_l3, m)
    print("msds_core_transform done for", site, "->", tgt)


In [0]:
# === _core_person — ported from sp1_10_person_spine ===
# mothers -> _stg_mother + person(role='mother'). Demographics (birth_datetime, gender, race)
# derived from msd001; race via the Barts NHS ethnic-category A–Z crosswalk recovered from
# 4_prod.omop.person (D8: Barts convention binds). TRUNCATE -> DELETE-by-site; person INSERT
# carries an explicit column list (ensure_enrich_schema later adds provenance cols).

def _core_person(site, site_esc, src, tgt, m, race_case):
    spark.sql(f"DELETE FROM {tgt}.person WHERE site_id='{site_esc}'")
    print("step2: person rows for site cleared")

    spark.sql(f"""
    CREATE OR REPLACE TABLE {tgt}._stg_mother AS
    SELECT
      {m('person', 'ORGIDLPID', 'LPIDMOTHER')} AS person_id,
      ORGIDLPID, LPIDMOTHER,
      concat_ws('|', '{site_esc}', coalesce(nullif(trim(ORGIDLPID),''),'~'), LPIDMOTHER) AS person_source_value,
      NHSNUMBERMOTHER AS nhs_source_value,
      try_cast(PERSONBIRTHDATEMOTHER AS TIMESTAMP) AS birth_datetime,
      upper(trim(ETHNICCATEGORYMOTHER)) AS race_source_value,
      {race_case.format(col='ETHNICCATEGORYMOTHER')} AS race_concept_id
    FROM {src['msd001_motherdemo']}
    WHERE LPIDMOTHER IS NOT NULL
    """)
    print("step3: _stg_mother created")

    n = spark.table(f"{tgt}._stg_mother").count()
    d = spark.sql(f"SELECT count(DISTINCT person_id) c FROM {tgt}._stg_mother").first()["c"]
    assert n == d, f"mother person_id COLLISION: {n} rows vs {d} distinct"
    print("step4: mothers", n, "collision-free")

    spark.sql(f"""
    INSERT INTO {tgt}.person (person_id, person_source_value, person_role, gender_source_value, birth_datetime, race_concept_id, race_source_value, site_id)
    SELECT person_id, person_source_value, 'mother' AS person_role,
           CAST(NULL AS STRING) AS gender_source_value, birth_datetime,
           race_concept_id, race_source_value, '{site_esc}' AS site_id
    FROM {tgt}._stg_mother
    """)
    pc = spark.sql(f"SELECT count(*) c FROM {tgt}.person WHERE person_role='mother'").first()["c"]
    race = spark.sql(f"SELECT count(CASE WHEN race_concept_id<>0 THEN 1 END) r, count(birth_datetime) b, count(*) t FROM {tgt}.person WHERE person_role='mother'").first()
    print("step5: mother persons inserted:", pc, "| race-mapped", race['r'], "dob", race['b'], "of", race['t'])


In [0]:
# === _core_pregnancy — ported from sp1_20_pregnancy ===
# msd101pregbook (+ msd301/msd401 delivery rollup) -> pregnancy. Dedup: exclude PREGNANCYID='0'
# sentinel (2 junk rows); dedup real dup PREGNANCYIDs by ADC_UPDT. Delivery-derived cols
# (end_date, gestation, outcome, mode, single/fetuses/liveborn) come from the labour/baby tables
# relinked to PREGNANCYID via LABOURDELIVERYID (same hop as msd404). INSERT OVERWRITE ->
# DELETE-by-site + explicit-column INSERT.

def _core_pregnancy(site, site_esc, src, tgt, m):
    # --- national code-set decodes (Barts convention, verified live 2026-07-07) ---
    # FOLICACIDSUPPLEMENT 01/02 -> Yes(4188539), 03 -> No(4188540)  [prod-join-back, 0 disagreement]
    # PREGOUTCOME (NHS Data Dictionary): 01 live birth, 02 stillbirth, 03/04 stillbirth, 05 other
    #   -> mapped to the same standard concepts prod omop.infant uses (01 dominant & clean).
    # DELIVERYMETHODCODE (NHS DD): 0 spont vaginal, 1 vaginal-other, 2 low forceps, 3 forceps,
    #   4 ventouse, 5/6 breech, 7 elective CS, 8 emergency CS  [prod-join-back, clean].
    # end_date source = PERSONBIRTHDATETIMEBABY (100% populated); BABYBIRTHDATETIME is 100% NULL
    # in the singular family. BIRTHWEIGHT is also 100% NULL -> birth_weight is SP-3 (see 30_infant).
    FOLIC_YES, FOLIC_NO = 4188539, 4188540
    NULLIP, MULTIP = 4012561, 4102166      # parity Condition concepts (prod)
    SINGLE_YES, SINGLE_NO = 4188539, 4188540

    OUTCOME_CASE = """CASE trim(dr.primary_outcome)
      WHEN '01' THEN 45883764   -- Live birth
      WHEN '02' THEN 37017027   -- Antepartum stillbirth
      WHEN '03' THEN 45773593   -- Intrapartum stillbirth
      WHEN '04' THEN 443213     -- Stillbirth (unspecified)
      ELSE 0 END"""
    MODE_CASE = """CASE trim(dr.primary_mode)
      WHEN '0' THEN 45885338 WHEN '1' THEN 45885338   -- Spontaneous vaginal
      WHEN '2' THEN 4156948                            -- Low forceps
      WHEN '3' THEN 4217586                            -- Forceps
      WHEN '4' THEN 440790                             -- Ventouse / vacuum
      WHEN '5' THEN 4213387 WHEN '6' THEN 4213387      -- Breech
      WHEN '7' THEN 4075182                            -- Elective caesarean
      WHEN '8' THEN 4167089                            -- Emergency caesarean
      ELSE 0 END"""
    OUTCOME_SRC = """CASE trim(dr.primary_outcome)
      WHEN '01' THEN 'Live birth' WHEN '02' THEN 'Stillbirth - antepartum'
      WHEN '03' THEN 'Stillbirth - intrapartum' WHEN '04' THEN 'Stillbirth'
      WHEN '05' THEN 'Other' ELSE dr.primary_outcome END"""
    MODE_SRC = """CASE trim(dr.primary_mode)
      WHEN '0' THEN 'Spontaneous vaginal' WHEN '1' THEN 'Vaginal' WHEN '2' THEN 'Low forceps'
      WHEN '3' THEN 'Forceps' WHEN '4' THEN 'Ventouse' WHEN '5' THEN 'Breech' WHEN '6' THEN 'Breech'
      WHEN '7' THEN 'Elective caesarean' WHEN '8' THEN 'Emergency caesarean' ELSE dr.primary_mode END"""

    spark.sql(f"DELETE FROM {tgt}.pregnancy WHERE site_id='{site_esc}'")

    spark.sql(f"""
    INSERT INTO {tgt}.pregnancy (person_id, pregnancy_id, pregnancy_start_date, pregnancy_end_date, gestational_length_in_day, pregnancy_outcome_concept_id, pregnancy_mode_delivery_concept_id, pregnancy_single_concept_id, pregnancy_marital_status_concept_id, pregnancy_number_fetuses, pregnancy_number_liveborn, prev_pregnancy_parity_concept_id, prev_pregnancy_gravidity, prev_livebirth_number, prev_still_misc_number, pre_pregnancy_bmi, pregnancy_folic_concept_id, pregnancy_outcome_source_value, pregnancy_mode_delivery_source_value, site_id)
    WITH dd AS (
      SELECT * FROM (
        SELECT p.*, row_number() OVER (
          PARTITION BY PREGNANCYID
          ORDER BY CASE WHEN ADC_UPDT IS NULL THEN 1 ELSE 0 END, ADC_UPDT DESC) AS rn
        FROM {src['msd101_pregbook']} p
        WHERE PREGNANCYID IS NOT NULL AND trim(PREGNANCYID) <> '0'
      ) WHERE rn = 1
    ),
    -- one baby row per (pregnancy, baby), relinked via LABOURDELIVERYID -> PREGNANCYID
    babies AS (
      SELECT l.PREGNANCYID,
             b.LPIDBABY, b.BIRTHORDERMATERNITYSUS, b.LABOURDELIVERYID,
             try_cast(b.PERSONBIRTHDATETIMEBABY AS DATE) AS birth_date,
             try_cast(b.GESTATIONLENGTHBIRTH AS INT) AS gest_days,
             cast(b.PREGOUTCOME AS STRING) AS pregoutcome,
             cast(b.DELIVERYMETHODCODE AS STRING) AS deliverymethod,
             row_number() OVER (PARTITION BY l.PREGNANCYID
               ORDER BY CASE WHEN b.BIRTHORDERMATERNITYSUS IS NULL THEN 1 ELSE 0 END,
                        try_cast(b.BIRTHORDERMATERNITYSUS AS INT), b.LPIDBABY) AS _bord
      FROM {src['msd401_babydemo']} b
      JOIN {src['msd301_labdel']} l ON l.LABOURDELIVERYID = b.LABOURDELIVERYID
      WHERE l.PREGNANCYID IS NOT NULL
    ),
    -- roll babies up to one row per pregnancy: primary (lowest birth order) drives outcome/mode
    dr AS (
      SELECT PREGNANCYID,
             count(*) AS n_babies,
             count(CASE WHEN trim(pregoutcome)='01' THEN 1 END) AS n_liveborn,
             max(birth_date) AS end_date,
             max(gest_days) AS gest_days,
             max(CASE WHEN _bord=1 THEN pregoutcome END) AS primary_outcome,
             max(CASE WHEN _bord=1 THEN deliverymethod END) AS primary_mode
      FROM babies GROUP BY PREGNANCYID
    )
    SELECT
      m.person_id,
      {m('pregnancy', 'p.PREGNANCYID')} AS pregnancy_id,
      try_cast(p.LASTMENSTRUALPERIODDATE AS DATE) AS pregnancy_start_date,
      CASE WHEN dr.end_date < try_cast(p.LASTMENSTRUALPERIODDATE AS DATE)
           THEN CAST(NULL AS DATE) ELSE dr.end_date END AS pregnancy_end_date,
      CASE WHEN dr.gest_days BETWEEN 140 AND 320 THEN dr.gest_days ELSE NULL END AS gestational_length_in_day,
      {OUTCOME_CASE}     AS pregnancy_outcome_concept_id,
      {MODE_CASE}        AS pregnancy_mode_delivery_concept_id,
      CASE WHEN dr.n_babies IS NULL THEN CAST(NULL AS BIGINT)
           WHEN dr.n_babies = 1 THEN {SINGLE_YES} ELSE {SINGLE_NO} END AS pregnancy_single_concept_id,
      CAST(NULL AS BIGINT) AS pregnancy_marital_status_concept_id,   -- SP-3 (no MSDS source)
      dr.n_babies        AS pregnancy_number_fetuses,
      dr.n_liveborn      AS pregnancy_number_liveborn,
      -- parity: nulliparous if no prior live/still births, else multiparous
      CASE WHEN try_cast(p.PREVIOUSLIVEBIRTHS AS INT) IS NULL
                AND try_cast(p.PREVIOUSSTILLBIRTHS AS INT) IS NULL THEN CAST(NULL AS BIGINT)
           WHEN coalesce(try_cast(p.PREVIOUSLIVEBIRTHS AS INT),0)
              + coalesce(try_cast(p.PREVIOUSSTILLBIRTHS AS INT),0) = 0 THEN {NULLIP}
           ELSE {MULTIP} END AS prev_pregnancy_parity_concept_id,
      -- gravidity: total prior pregnancies = live + still + losses<24wk + (this one is excluded)
      CASE WHEN try_cast(p.PREVIOUSLIVEBIRTHS AS INT) IS NULL
                AND try_cast(p.PREVIOUSSTILLBIRTHS AS INT) IS NULL
                AND try_cast(p.PREVIOUSLOSSESLESSTHAN24WEEKS AS INT) IS NULL THEN CAST(NULL AS INT)
           ELSE coalesce(try_cast(p.PREVIOUSLIVEBIRTHS AS INT),0)
              + coalesce(try_cast(p.PREVIOUSSTILLBIRTHS AS INT),0)
              + coalesce(try_cast(p.PREVIOUSLOSSESLESSTHAN24WEEKS AS INT),0) END AS prev_pregnancy_gravidity,
      try_cast(p.PREVIOUSLIVEBIRTHS AS INT)   AS prev_livebirth_number,
      try_cast(p.PREVIOUSSTILLBIRTHS AS INT)  AS prev_still_misc_number,
      CAST(NULL AS DOUBLE) AS pre_pregnancy_bmi,                     -- SP-3 (PERSONHEIGHT/WEIGHT 100% null)
      CASE WHEN trim(p.FOLICACIDSUPPLEMENT) IN ('01','02') THEN {FOLIC_YES}
           WHEN trim(p.FOLICACIDSUPPLEMENT) = '03' THEN {FOLIC_NO}
           ELSE 0 END AS pregnancy_folic_concept_id,
      {OUTCOME_SRC} AS pregnancy_outcome_source_value,
      {MODE_SRC}    AS pregnancy_mode_delivery_source_value,
      '{site_esc}' AS site_id
    FROM dd p
    JOIN {tgt}._stg_mother m ON m.LPIDMOTHER = p.LPIDMOTHER
    LEFT JOIN dr ON dr.PREGNANCYID = p.PREGNANCYID
    """)
    n = spark.table(f"{tgt}.pregnancy").count()
    d = spark.sql(f"SELECT count(DISTINCT pregnancy_id) c FROM {tgt}.pregnancy").first()["c"]
    assert n == d, f"pregnancy_id COLLISION {n} vs {d}"
    orphan = spark.sql(f"""SELECT count(*) c FROM {tgt}.pregnancy pr
      LEFT JOIN {tgt}.person pe ON pr.person_id=pe.person_id AND pe.person_role='mother'
      WHERE pe.person_id IS NULL""").first()["c"]
    assert orphan == 0, f"{orphan} pregnancies with no mother"
    cov = spark.sql(f"""SELECT
      count(pregnancy_end_date) end_dt, count(gestational_length_in_day) gest,
      count(CASE WHEN pregnancy_outcome_concept_id<>0 THEN 1 END) outc,
      count(CASE WHEN pregnancy_mode_delivery_concept_id<>0 THEN 1 END) mode,
      count(pregnancy_single_concept_id) single, count(pregnancy_number_fetuses) fet,
      count(prev_pregnancy_parity_concept_id) par, count(prev_pregnancy_gravidity) grav
      FROM {tgt}.pregnancy""").first()
    print("pregnancy rows", n, "orphan", orphan)
    print("coverage:", cov.asDict())

    spark.sql(f"DROP TABLE IF EXISTS {tgt}._stg_mother")


In [0]:
# === _core_infant — ported from sp1_30_infant ===
# msd401babydemo per-baby -> infant + person(role='infant'). Runs AFTER _core_person (does NOT
# clear all person) AND _core_pregnancy (needs pregnancy table to validate the FK). Enriched:
# birth_outcome (PREGOUTCOME), congenital malformation flag (msd404 neonatal diagnosis resolving
# under SNOMED 'Congenital disease' 440508). birth_weight + birth_apgar stay NULL — BIRTHWEIGHT
# and APGARSCORE5 are 100% NULL in the singular family (SP-3 / Cerner). The birth_weight
# expression is kept so a future MSDS load that populates BIRTHWEIGHT works with no code change.
# CREATE OR REPLACE infant -> transient _stg_infant then DELETE-by-site + explicit-column INSERT.

def _core_infant(site, site_esc, src, tgt, m, race_case):
    OUTCOME_CASE = """CASE trim(b.PREGOUTCOME)
      WHEN '01' THEN 45883764 WHEN '02' THEN 37017027 WHEN '03' THEN 45773593
      WHEN '04' THEN 443213 ELSE 0 END"""

    # per-baby congenital-malformation flag: babies (LPIDBABY) with any msd404 neonatal diagnosis
    # whose resolved concept sits under SNOMED 'Congenital disease' (440508). Persisted as a small
    # Delta table (single CTAS — serverless-safe, avoids temp-view chains).
    spark.sql(f"""
    CREATE OR REPLACE TABLE {tgt}._cong_babies AS
    WITH diag_resolved AS (
      SELECT d.LPIDBABY,
        CASE WHEN r.standard_concept='S' AND r.is_valid
               THEN array(CAST(r.src_concept_id AS BIGINT))
             WHEN size(r.maps_to)>=1 THEN array_sort(array_distinct(r.maps_to))
             WHEN r.replaced_to IS NOT NULL
               THEN array(CAST(r.replaced_to AS BIGINT))
             ELSE array(CAST(0 AS BIGINT)) END AS cids
      FROM {src['msd404_diagneo']} d
      LEFT JOIN {tgt}._resolved_snomed r ON r.concept_code=d.DIAG
      WHERE d.LPIDBABY IS NOT NULL AND d.DIAG IS NOT NULL
    ),
    diag AS (
      SELECT DISTINCT LPIDBABY, explode(cids) AS cid FROM diag_resolved
    )
    SELECT DISTINCT diag.LPIDBABY
    FROM diag JOIN 4_prod.omop.concept_ancestor ca
      ON ca.descendant_concept_id=diag.cid AND ca.ancestor_concept_id=440508
    """)
    print("congenital babies:", spark.table(f"{tgt}._cong_babies").count())

    # identity triple (_idkind,_k1,_k2): primary=(ORGIDLOCALPATIENTIDBABY,LPIDBABY);
    # fallback (null LPIDBABY)=(LABOURDELIVERYID,BIRTHORDERMATERNITYSUS). Dedup on the FULL
    # triple (matches mint key). pregnancy_id = NULL unless the minted id exists in the built
    # pregnancy table (excludes the PREGNANCYID='0' sentinel + delivery rows whose PREGNANCYID
    # is not in msd101 booking -> no dangling FK).
    IDENT = """
      CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN 'lpid' ELSE 'fallback' END AS _idkind,
      CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN ORGIDLOCALPATIENTIDBABY ELSE LABOURDELIVERYID END AS _k1,
      CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN LPIDBABY ELSE cast(BIRTHORDERMATERNITYSUS as string) END AS _k2
    """

    spark.sql(f"""
    CREATE OR REPLACE TABLE {tgt}._stg_infant AS
    WITH ident AS (
      SELECT b.LABOURDELIVERYID, b.LPIDBABY, b.PREGOUTCOME,
             try_cast(b.BIRTHWEIGHT AS DOUBLE) AS BIRTHWEIGHT, {IDENT}
      FROM {src['msd401_babydemo']} b
    ),
    dedup AS (
      SELECT * FROM (
        SELECT i.*, row_number() OVER (
          PARTITION BY _idkind, _k1, _k2 ORDER BY LABOURDELIVERYID) AS rn
        FROM ident i
      ) WHERE rn = 1
    )
    SELECT
      pr.pregnancy_id                                             AS pregnancy_id,
      {m('infant', 'b._idkind', 'b._k1', 'b._k2')}              AS infant_id,
      {OUTCOME_CASE}                                              AS birth_outcome_concept_id,
      CASE WHEN b.BIRTHWEIGHT BETWEEN 100 AND 8000 THEN b.BIRTHWEIGHT ELSE NULL END AS birth_weight,
      CASE WHEN cg.LPIDBABY IS NOT NULL THEN 4079975 ELSE 0 END   AS birth_con_malformation_concept_id,
      CAST(NULL AS INT)    AS birth_apgar,
      b.LABOURDELIVERYID   AS labour_delivery_id,
      '{site_esc}' AS site_id
    FROM dedup b
    LEFT JOIN {src['msd301_labdel']} d301 ON d301.LABOURDELIVERYID = b.LABOURDELIVERYID
    LEFT JOIN {tgt}.pregnancy pr ON pr.pregnancy_id = {m('pregnancy', 'd301.PREGNANCYID')}
    LEFT JOIN {tgt}._cong_babies cg ON cg.LPIDBABY = b.LPIDBABY
    """)

    spark.sql(f"DELETE FROM {tgt}.infant WHERE site_id='{site_esc}'")
    spark.sql(f"""
    INSERT INTO {tgt}.infant (pregnancy_id, infant_id, birth_outcome_concept_id, birth_weight, birth_con_malformation_concept_id, birth_apgar, labour_delivery_id, site_id)
    SELECT pregnancy_id, infant_id, birth_outcome_concept_id, birth_weight, birth_con_malformation_concept_id, birth_apgar, labour_delivery_id, site_id
    FROM {tgt}._stg_infant
    """)
    n = spark.table(f"{tgt}.infant").count()
    d = spark.sql(f"SELECT count(DISTINCT infant_id) c FROM {tgt}.infant").first()["c"]
    assert n == d, f"infant_id COLLISION {n} vs {d}"
    # FK gate: no infant may reference a pregnancy_id absent from the pregnancy table
    fk = spark.sql(f"""SELECT count(*) c FROM {tgt}.infant i
      WHERE i.pregnancy_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tgt}.pregnancy pr WHERE pr.pregnancy_id=i.pregnancy_id)""").first()["c"]
    assert fk == 0, f"{fk} infants with dangling pregnancy_id"
    cov = spark.sql(f"""SELECT count(pregnancy_id) linked,
      count(CASE WHEN birth_outcome_concept_id<>0 THEN 1 END) outc,
      count(CASE WHEN birth_con_malformation_concept_id<>0 THEN 1 END) cong
      FROM {tgt}.infant""").first()
    print("infant rows", n, "collision-free | dangling-FK", fk, "| linked", cov['linked'], "outcome", cov['outc'], "congenital", cov['cong'])

    # infant persons (role='infant'): infant_id == person_id; sex + DOB + race from msd401
    spark.sql(f"""
    INSERT INTO {tgt}.person (person_id, person_source_value, person_role, gender_source_value, birth_datetime, race_concept_id, race_source_value, site_id)
    WITH ident AS (
      SELECT {IDENT},
        try_cast(PERSONBIRTHDATETIMEBABY AS TIMESTAMP) AS birth_datetime,
        cast(PERSONPHENSEX AS STRING) AS gender_source_value,
        upper(trim(ETHNICCATEGORYBABY)) AS race_source_value,
        {race_case.format(col='ETHNICCATEGORYBABY')} AS race_concept_id
      FROM {src['msd401_babydemo']}
    ),
    dedup AS (
      SELECT * FROM (
        SELECT i.*, row_number() OVER (PARTITION BY _idkind,_k1,_k2 ORDER BY _idkind) AS rn FROM ident i
      ) WHERE rn=1
    )
    SELECT
      {m('infant', '_idkind', '_k1', '_k2')} AS person_id,
      concat_ws('|','{site_esc}', _idkind, coalesce(nullif(trim(_k1),''),'~'), _k2) AS person_source_value,
      'infant' AS person_role,
      gender_source_value, birth_datetime,
      race_concept_id, race_source_value, '{site_esc}' AS site_id
    FROM dedup
    """)
    ic = spark.sql(f"SELECT count(*) c FROM {tgt}.person WHERE person_role='infant'").first()["c"]
    print("infant persons inserted:", ic)

    spark.sql(f"DROP TABLE IF EXISTS {tgt}._stg_infant")
    spark.sql(f"DROP TABLE IF EXISTS {tgt}._cong_babies")


In [0]:
# === _core_facts — ported from sp1_80_facts ===
# Decode the 7 long-format MSDS tables -> transient {tgt}._all_facts (INCREMENTAL append,
# serverless-safe): 15 small INSERT INTO statements, not one giant UNION CTAS. _all_facts has
# NO site_id (transient, rebuilt wholesale each run, consumed immediately by _core_persist),
# so it keeps DROP TABLE IF EXISTS + CREATE TABLE. Raw 4_prod.raw.<tbl> -> src[...] keys; the
# emit `msds_table` arg keeps the RAW MSDS table name (it is the msds_omop_map join key). The
# map/resolve joins read {tgt}.msds_omop_map / {tgt}._resolved_snomed.

def _core_facts(site, site_esc, src, tgt, m):
    spark.sql("DROP TABLE IF EXISTS " + tgt + "._all_facts")
    spark.sql("CREATE TABLE " + tgt + """._all_facts (
      source_table STRING, pregnancy_natural_key STRING, baby_natural_key STRING, careconid STRING, fact_natural_key STRING,
      target_omop_table STRING, concept_id BIGINT, multi_map_flag INT,
      source_value STRING, source_concept_id BIGINT, event_date DATE,
      value_as_number DOUBLE, unit_source_value STRING) USING DELTA""")

    # --- Step 0 (Task 2): ONE canonical care-contact relation — dedup the 25 raw msd201 dupes ONCE ---
    # Deterministic tie-break (no NEWID/random): NULL contact-datetime last (never ranks a NULL above a
    # non-null), then earliest CCONTACTDATETIME, then newest ADC_UPDT, then PREGNANCYID as a stable final
    # key. FROM_202 (msd202 relink) and the L3 episode_event read THIS table (not raw msd201) so
    # duplicate CARECONIDs cannot multiply downstream facts. NOTE (Task 4 Step 2): the legacy
    # `_core_visit` still reads raw msd201 and is NOT used by PREBORN — Task 7 builds visit_occurrence
    # from THIS deduped {tgt}._carecontact instead (so the dup CARECONIDs can't re-appear as dup visit
    # PKs). Deduped-away rows are
    # re-derivable; the _quarantine (reason dup_careconid) insert happens later where _quarantine exists.
    spark.sql(f"""
    CREATE OR REPLACE TABLE {tgt}._carecontact AS
    SELECT CARECONID, PREGNANCYID, CCONTACTDATETIME FROM (
      SELECT CARECONID, PREGNANCYID, CCONTACTDATETIME, row_number() OVER (
        PARTITION BY CARECONID
        ORDER BY CASE WHEN CCONTACTDATETIME IS NULL THEN 1 ELSE 0 END,
                 CCONTACTDATETIME,
                 CASE WHEN ADC_UPDT IS NULL THEN 1 ELSE 0 END, ADC_UPDT DESC,
                 PREGNANCYID) AS rn
      FROM {src['msd201_carecontactpreg']} WHERE CARECONID IS NOT NULL) WHERE rn = 1
    """)
    # FIX3: record the deduped-away CARECONIDs so _merge_to_cdm can quarantine them (reason dup_careconid).
    spark.sql(f"""
    CREATE OR REPLACE TABLE {tgt}._dropped_carecontact AS
    SELECT CARECONID FROM (
      SELECT CARECONID, row_number() OVER (
        PARTITION BY CARECONID
        ORDER BY CASE WHEN CCONTACTDATETIME IS NULL THEN 1 ELSE 0 END,
                 CCONTACTDATETIME,
                 CASE WHEN ADC_UPDT IS NULL THEN 1 ELSE 0 END, ADC_UPDT DESC,
                 PREGNANCYID) AS rn
      FROM {src['msd201_carecontactpreg']} WHERE CARECONID IS NOT NULL) WHERE rn > 1
    """)
    _cc_raw = spark.sql(f"SELECT count(*) c FROM {src['msd201_carecontactpreg']} WHERE CARECONID IS NOT NULL").first()["c"]
    _cc_keep = spark.table(f"{tgt}._carecontact").count()
    print("_carecontact rows", _cc_keep, "| dup_careconid dropped", _cc_raw - _cc_keep,
          "(recorded to {tgt}._dropped_carecontact; quarantined in _merge_to_cdm)")

    # --- Step 0 (Task 2): canonical baby-identity lookup (LPIDBABY -> resolved infant person_id) ---
    # SAME IDENT triple + dedup as _core_infant so infant_person_id == the minted infant PK. Only rows
    # with non-null LPIDBABY are kept (msd404/405 carry LPIDBABY); the merge (Task 6) LEFT JOINs
    # _all_facts.baby_natural_key -> _baby_id.LPIDBABY to route baby-keyed facts to the infant.
    _BABY_IDENT = """
      CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN 'lpid' ELSE 'fallback' END AS _idkind,
      CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN ORGIDLOCALPATIENTIDBABY ELSE LABOURDELIVERYID END AS _k1,
      CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN LPIDBABY ELSE cast(BIRTHORDERMATERNITYSUS as string) END AS _k2
    """
    spark.sql(f"""
    CREATE OR REPLACE TABLE {tgt}._baby_id AS
    WITH ident AS (
      SELECT LPIDBABY, {_BABY_IDENT} FROM {src['msd401_babydemo']}
    ),
    dedup AS (
      SELECT * FROM (
        SELECT i.*, row_number() OVER (PARTITION BY _idkind,_k1,_k2 ORDER BY _idkind) AS rn FROM ident i
      ) WHERE rn=1
    )
    SELECT LPIDBABY, {m('infant', '_idkind', '_k1', '_k2')} AS infant_person_id
    FROM dedup WHERE nullif(trim(LPIDBABY),'') IS NOT NULL
    """)
    # T6-carry-A (Task 7): collision guard — a single LPIDBABY must map to exactly one identity
    # triple (else the grain-routing join _all_facts.baby_natural_key=_baby_id.LPIDBABY fans out and
    # mis-routes/duplicates baby facts). Mirrors the pregnancy/infant collision asserts. 0 today at
    # Barts, but nothing else gates it and PREBORN is multi-site.
    _bn = spark.table(f"{tgt}._baby_id").count()
    _bd = spark.sql(f"SELECT count(DISTINCT LPIDBABY) c FROM {tgt}._baby_id").first()["c"]
    assert _bn == _bd, f"_baby_id LPIDBABY COLLISION {_bn} vs {_bd}"
    print("_baby_id rows", _bn, "collision-free")

    def emit(msds_table, code_col, date_expr, from_sql, row_id_cols, value_col=None, unit_col=None,
             baby_key_expr=None, careconid_expr=None, tie_break_cols=None):
        row_id = ", ".join(row_id_cols)
        fnk_cols = ", ".join(["m." + c for c in row_id_cols])
        val = ("try_cast(m." + value_col + " AS DOUBLE)") if value_col else "CAST(NULL AS DOUBLE)"
        unit = ("m." + unit_col) if unit_col else "CAST(NULL AS STRING)"
        baby_k = (baby_key_expr or "CAST(NULL AS STRING)")
        cc_k = (careconid_expr or "CAST(NULL AS STRING)")
        # _occ distinguishes true duplicate source rows. Its ORDER BY must include every
        # output-affecting field; ordering only by the partition columns makes IDs unstable.
        tie_cols = list(row_id_cols) + ["PREGNANCYID"] + list(tie_break_cols or [])
        if value_col:
            tie_cols.append(value_col)
        if unit_col:
            tie_cols.append(unit_col)
        tie_cols = list(dict.fromkeys(tie_cols))
        tie_order = ", ".join([f"coalesce(cast({c} as string),'~')" for c in tie_cols])
        filtered = ("SELECT * FROM (" + from_sql + ") src0 WHERE src0." + code_col +
                    " IS NOT NULL AND trim(src0." + code_col + ") <> ''")
        sql = (
          "INSERT INTO " + tgt + "._all_facts "
          "WITH occ AS ( SELECT src.*, row_number() OVER (PARTITION BY " + row_id +
            " ORDER BY " + tie_order + ") AS _occ FROM (" + filtered + ") src ), "
          "matched AS ( SELECT o.*, map.target_omop_table, map.target_domain, map.target_concept_id, "
            "row_number() OVER (PARTITION BY " + row_id + ", o._occ ORDER BY "
            "CASE WHEN map.source_value IS NOT NULL THEN 0 ELSE 1 END, map.rule_id) AS _rk "
            "FROM occ o LEFT JOIN " + tgt + ".msds_omop_map map "
            "ON map.msds_table='" + msds_table + "' AND map.msds_field='" + code_col + "' "
            "AND (map.source_value = o." + code_col + " OR map.source_value IS NULL) ), "
          "resolved AS ( SELECT m.*, r.src_concept_id AS resolved_source_concept_id, "
            "CASE WHEN m.target_concept_id IS NOT NULL THEN array(CAST(m.target_concept_id AS BIGINT)) "
            "WHEN r.standard_concept='S' AND r.is_valid THEN array(CAST(r.src_concept_id AS BIGINT)) "
            "WHEN size(r.maps_to) >= 1 THEN array_sort(array_distinct(r.maps_to)) "
            "WHEN r.replaced_to IS NOT NULL THEN array(CAST(r.replaced_to AS BIGINT)) "
            "ELSE array(CAST(0 AS BIGINT)) END AS resolved_ids "
            "FROM matched m LEFT JOIN " + tgt + "._resolved_snomed r "
            "ON r.concept_code = m." + code_col + " WHERE m._rk=1 ), "
          "expanded AS ( SELECT x.*, explode(x.resolved_ids) AS resolved_concept_id, "
            "size(x.resolved_ids) AS resolved_count FROM resolved x ) "
          "SELECT '" + msds_table + "' AS source_table, cast(m.PREGNANCYID as string) AS pregnancy_natural_key, "
            + baby_k + " AS baby_natural_key, " + cc_k + " AS careconid, "
            "concat_ws('#', " + fnk_cols + ", cast(m._occ AS string), "
              "cast(m.resolved_concept_id AS string)) AS fact_natural_key, "
            "CASE WHEN m.target_concept_id IS NOT NULL THEN m.target_omop_table "
              "WHEN m.resolved_concept_id = 0 THEN m.target_omop_table "
              "WHEN tc.domain_id='Condition' THEN 'condition_occurrence' "
              "WHEN tc.domain_id='Measurement' THEN 'measurement' "
              "WHEN tc.domain_id='Observation' THEN 'observation' "
              "WHEN tc.domain_id='Procedure' THEN 'procedure_occurrence' "
              "ELSE CAST(NULL AS STRING) END AS target_omop_table, "
            "m.resolved_concept_id AS concept_id, "
            "CASE WHEN m.resolved_count > 1 THEN 1 ELSE 0 END AS multi_map_flag, "
            "m." + code_col + " AS source_value, "
            "m.resolved_source_concept_id AS source_concept_id, "
            + date_expr + " AS event_date, " + val + " AS value_as_number, "
            + unit + " AS unit_source_value "
          "FROM expanded m LEFT JOIN 4_prod.omop.concept tc "
            "ON tc.concept_id=m.resolved_concept_id"
        )
        spark.sql(sql)
        print("emitted", msds_table, code_col)

    FROM_404 = (f"SELECT d.*, l.PREGNANCYID FROM {src['msd404_diagneo']} d "
      f"LEFT JOIN (SELECT LPIDBABY, max(LABOURDELIVERYID) AS LABOURDELIVERYID FROM {src['msd401_babydemo']} "
      "WHERE LPIDBABY IS NOT NULL GROUP BY LPIDBABY) b ON b.LPIDBABY=d.LPIDBABY "
      f"LEFT JOIN {src['msd301_labdel']} l ON l.LABOURDELIVERYID=b.LABOURDELIVERYID WHERE d.DIAG IS NOT NULL")
    FROM_405 = (f"SELECT a.*, l.PREGNANCYID FROM {src['msd405_careactbaby']} a "
      f"LEFT JOIN (SELECT LPIDBABY, max(LABOURDELIVERYID) AS LABOURDELIVERYID FROM {src['msd401_babydemo']} "
      "WHERE LPIDBABY IS NOT NULL GROUP BY LPIDBABY) b ON b.LPIDBABY=a.LPIDBABY "
      f"LEFT JOIN {src['msd301_labdel']} l ON l.LABOURDELIVERYID=b.LABOURDELIVERYID")
    FROM_202 = (f"SELECT a.*, c.PREGNANCYID, c.CCONTACTDATETIME FROM {src['msd202_careactivitypreg']} a "
      f"LEFT JOIN {tgt}._carecontact c ON c.CARECONID=a.CARECONID")
    FROM_302 = (f"SELECT a.*, l.PREGNANCYID FROM {src['msd302_careactlabdel']} a "
      f"LEFT JOIN {src['msd301_labdel']} l ON l.LABOURDELIVERYID=a.LABOURDELIVERYID")

    emit("msd106diagnosispreg","DIAG","try_cast(m.DIAGDATE AS DATE)",
      f"SELECT * FROM {src['msd106_diagnosispreg']} WHERE DIAG IS NOT NULL",["PREGNANCYID","DIAG","DIAGDATE"])
    emit("msd107medhistory","PREVDIAG","try_cast(m.DIAGDATE AS DATE)",
      f"SELECT * FROM {src['msd107_medhistory']} WHERE PREVDIAG IS NOT NULL",["PREGNANCYID","PREVDIAG","DIAGDATE"])
    emit("msd404diagneo","DIAG","try_cast(m.DIAGDATE AS DATE)",FROM_404,["LPIDBABY","DIAGSCHEME","DIAG","DIAGDATE"],baby_key_expr="m.LPIDBABY")
    emit("msd109findobsmother","OBSCODE","try_cast(m.OBSDATE AS DATE)",
      f"SELECT * FROM {src['msd109_findobsmother']} WHERE OBSCODE IS NOT NULL",["PREGNANCYID","OBSCODE","OBSDATE"],
      value_col="OBSVALUE",unit_col="UCUMUNIT")
    emit("msd109findobsmother","FINDINGCODE","try_cast(m.FINDINGDATE AS DATE)",
      f"SELECT * FROM {src['msd109_findobsmother']} WHERE FINDINGCODE IS NOT NULL",["PREGNANCYID","FINDINGCODE","FINDINGDATE"])
    emit("msd202careactivitypreg","PROCEDURECODE","try_cast(m.CCONTACTDATETIME AS DATE)",FROM_202,["CAREACTIDMOTHER","PROCEDURECODE"],careconid_expr="m.CARECONID",tie_break_cols=["CARECONID","CCONTACTDATETIME"])
    emit("msd202careactivitypreg","FINDINGCODE","try_cast(m.CCONTACTDATETIME AS DATE)",FROM_202,["CAREACTIDMOTHER","FINDINGCODE"],careconid_expr="m.CARECONID",tie_break_cols=["CARECONID","CCONTACTDATETIME"])
    emit("msd202careactivitypreg","OBSCODE","try_cast(m.CCONTACTDATETIME AS DATE)",FROM_202,["CAREACTIDMOTHER","OBSCODE"],value_col="OBSVALUE",unit_col="UCUMUNIT",careconid_expr="m.CARECONID",tie_break_cols=["CARECONID","CCONTACTDATETIME"])
    emit("msd302careactlabdel","PROCEDURECODE","try_cast(m.CLININTERDATETIMEMOTHER AS DATE)",FROM_302,["LABOURDELIVERYID","CLININTERDATETIMEMOTHER","CAREPROFLID","PROCEDURECODE"])
    emit("msd302careactlabdel","FINDINGCODE","try_cast(m.CLININTERDATETIMEMOTHER AS DATE)",FROM_302,["LABOURDELIVERYID","CLININTERDATETIMEMOTHER","CAREPROFLID","FINDINGCODE"])
    emit("msd302careactlabdel","OBSCODE","try_cast(m.CLININTERDATETIMEMOTHER AS DATE)",FROM_302,["LABOURDELIVERYID","CLININTERDATETIMEMOTHER","CAREPROFLID","OBSCODE"],value_col="OBSVALUE",unit_col="UCUMUNIT")
    emit("msd405careactbaby","PROCEDURECODE","try_cast(m.CLININTERDATETIMEBABY AS DATE)",FROM_405,["CAREACTIDBABY","PROCEDURECODE"],baby_key_expr="m.LPIDBABY",tie_break_cols=["LPIDBABY","CLININTERDATETIMEBABY"])
    emit("msd405careactbaby","FINDINGCODE","try_cast(m.CLININTERDATETIMEBABY AS DATE)",FROM_405,["CAREACTIDBABY","FINDINGCODE"],baby_key_expr="m.LPIDBABY",tie_break_cols=["LPIDBABY","CLININTERDATETIMEBABY"])
    emit("msd405careactbaby","OBSCODE","try_cast(m.CLININTERDATETIMEBABY AS DATE)",FROM_405,["CAREACTIDBABY","OBSCODE"],value_col="OBSVALUE",unit_col="UCUMUNIT",baby_key_expr="m.LPIDBABY",tie_break_cols=["LPIDBABY","CLININTERDATETIMEBABY"])

    tot = spark.table(tgt + "._all_facts").count()
    print("all_facts rows", tot)


In [0]:
# === _core_persist — ported from sp1_85_persist_domains (persist helper + 4 calls) ===
# _all_facts -> per-domain CDM tables with minted ids. CREATE OR REPLACE -> DELETE-by-site +
# explicit-column INSERT (pool-safe). Domain-table DDL lives in ensure_core_schema; the explicit
# column list here matches that DDL and the SELECT output order exactly.
# MID_FACT/MID_PREG use the site-bound m(...) (sp1_85's inline mid()).

def _core_persist(site, site_esc, tgt, m):
    MID_FACT = m("{g}", "f.source_table", "f.fact_natural_key")
    MID_PREG = m("pregnancy", "f.pregnancy_natural_key")

    def persist(target_table, grain, id_col, concept_col, date_col, src_val_col, src_con_col, with_value):
        fact_id = MID_FACT.replace("{g}", grain)
        preg_id = MID_PREG
        valcols = (", f.value_as_number, f.unit_source_value") if with_value else ""
        cols = [id_col, "person_id", concept_col, date_col, "pregnancy_id", "source_table",
                src_val_col, src_con_col, "multi_map_flag"]
        if with_value:
            cols += ["value_as_number", "unit_source_value"]
        cols += ["site_id"]
        spark.sql("DELETE FROM " + tgt + "." + target_table + " WHERE site_id='" + site_esc + "'")
        sql = (
          "INSERT INTO " + tgt + "." + target_table + " (" + ", ".join(cols) + ") "
          "SELECT " + fact_id + " AS " + id_col + ", pr.person_id, "
            "f.concept_id AS " + concept_col + ", f.event_date AS " + date_col + ", "
            + preg_id + " AS pregnancy_id, f.source_table, "
            "f.source_value AS " + src_val_col + ", f.source_concept_id AS " + src_con_col + ", "
            "f.multi_map_flag" + valcols + ", '" + site_esc + "' AS site_id "
          "FROM " + tgt + "._all_facts f "
          "JOIN " + tgt + ".pregnancy pr ON pr.pregnancy_id = " + preg_id + " "
          "WHERE f.target_omop_table = '" + target_table + "'"
        )
        spark.sql(sql)
        print(target_table, spark.table(tgt + "." + target_table).count())

    persist("condition_occurrence","condition","condition_occurrence_id","condition_concept_id",
            "condition_start_date","condition_source_value","condition_source_concept_id", False)
    persist("measurement","measurement","measurement_id","measurement_concept_id",
            "measurement_date","measurement_source_value","measurement_source_concept_id", True)
    persist("observation","observation","observation_id","observation_concept_id",
            "observation_date","observation_source_value","observation_source_concept_id", True)
    persist("procedure_occurrence","procedure","procedure_occurrence_id","procedure_concept_id",
            "procedure_date","procedure_source_value","procedure_source_concept_id", False)

    # PREBORN (Task 4 Step 2, BLOCKING): _all_facts is NOT dropped here. The Task 6 grain-routed
    # fact merge reads the widened {stg}._all_facts (baby_natural_key/careconid, added in Task 2)
    # for grain routing + careconid linkage; dropping it here would silently destroy those keys.
    # drop_working() drops the whole {pub}_stg schema (incl. _all_facts) AFTER the merge/publish.


In [0]:
# === _core_visit — ported from sp1_85_persist_domains (visit_occurrence block) ===
# msd201 care-contacts -> visit_occurrence (minted); pregnancy_id links to pregnancy.
# CREATE OR REPLACE -> DELETE-by-site + explicit-column INSERT (pool-safe).
# NOTE: the sp1_85 `fact_relationship` scaffold (CREATE TABLE IF NOT EXISTS on {tgt}) is
# INTENTIONALLY DROPPED here — L3 owns the real fact_relationship at {tgt}_l3.fact_relationship
# (built by _l3_additions), so creating a stray empty {tgt}.fact_relationship is avoided.

def _core_visit(site, site_esc, src, tgt, m):
    spark.sql(f"DELETE FROM {tgt}.visit_occurrence WHERE site_id='{site_esc}'")
    spark.sql(
      "INSERT INTO " + tgt + ".visit_occurrence (visit_occurrence_id, pregnancy_id, visit_start_date, visit_source_value, site_id) "
      "SELECT " + m("visit", "CARECONID") + " AS visit_occurrence_id, "
        + m("pregnancy", "PREGNANCYID") + " AS pregnancy_id, "
        "try_cast(CCONTACTDATETIME AS DATE) AS visit_start_date, "
        "CARECONID AS visit_source_value, '" + site_esc + "' AS site_id "
      "FROM " + src['msd201_carecontactpreg'] + " WHERE CARECONID IS NOT NULL")
    print("visit_occurrence", spark.table(tgt + ".visit_occurrence").count())


In [0]:
# === _l3_additions — ported from sp2_10/20/30/40 (in order) ===
# All four already use DELETE-by-site + INSERT and read only {tgt}/{tgt_l3} tables (no raw src).
# Reschema: S->tgt, S3->tgt_l3, mid->m, SITE_ESC->site_esc. The sp2_40 LOCAL RACE_CASE / OBS_CASE
# CASE blocks are the L3 3-bucket collapse — DISTINCT from the core RACE_CASE crosswalk — and are
# kept verbatim (NOT substituted with the core constant).

def _l3_additions(site, site_esc, tgt, tgt_l3, m):
    S = tgt; S3 = tgt_l3; SITE_ESC = site_esc

    # --- sp2_10_pregnancy_condition: pregnancy -> condition_occurrence (MIREDA concept 4299535) ---
    spark.sql(f"DELETE FROM {S3}.condition_occurrence WHERE site_id = '{SITE_ESC}'")
    spark.sql(f"""
    INSERT INTO {S3}.condition_occurrence
    SELECT
      {m('pregnancy-condition', 'pr.pregnancy_id')} AS condition_occurrence_id,
      pr.person_id,
      4299535 AS condition_concept_id,             -- OMOP standard concept 'Pregnancy'
      pr.pregnancy_start_date AS condition_start_date,
      CASE WHEN pr.pregnancy_end_date < pr.pregnancy_start_date THEN NULL
           ELSE pr.pregnancy_end_date END AS condition_end_date,
      32817 AS condition_type_concept_id,          -- OMOP Type Concept 'EHR' (MSDS is EHR-derived)
      pr.pregnancy_id,
      'MSDS pregnancy' AS condition_source_value,
      '{SITE_ESC}' AS site_id
    FROM {S}.pregnancy pr
    WHERE pr.site_id = '{SITE_ESC}'
    """)
    n = spark.table(f"{S3}.condition_occurrence").count()
    d = spark.sql(f"SELECT count(DISTINCT condition_occurrence_id) c FROM {S3}.condition_occurrence").first()["c"]
    assert n == d, f"pregnancy-condition id COLLISION {n} vs {d}"
    # conformance: all rows carry 4299535; end >= start where end is non-null
    bad_concept = spark.sql(f"SELECT count(*) c FROM {S3}.condition_occurrence WHERE condition_concept_id<>4299535").first()["c"]
    bad_dates = spark.sql(f"""SELECT count(*) c FROM {S3}.condition_occurrence
      WHERE condition_end_date IS NOT NULL AND condition_end_date < condition_start_date""").first()["c"]
    assert bad_concept == 0, f"{bad_concept} rows not 4299535"
    assert bad_dates == 0, f"{bad_dates} rows end<start"
    bad_type = spark.sql(f"SELECT count(*) c FROM {S3}.condition_occurrence WHERE condition_type_concept_id<>32817").first()["c"]
    assert bad_type == 0, f"{bad_type} rows missing EHR type concept"
    null_end = spark.sql(f"SELECT count(*) c FROM {S3}.condition_occurrence WHERE condition_end_date IS NULL").first()["c"]
    print("pregnancy-condition rows", n, "collision-free | null_end", null_end, f"({round(100*null_end/(n or 1),1)}% antenatal-only, expected ~30%)")

    # --- sp2_20_fact_relationship: mother<->child, twin, sibling (symmetric pairs) ---
    # All person-to-person; fact_id = SP-1 person_id; domain concept = person.person_id CDM field.
    PFLD = 1147026   # probed person.person_id CDM field concept
    MOTHER, CHILD, TWIN, SIBLING = 4248584, 4285883, 4013484, 4292398

    # base of (mother_person_id, child_person_id, pregnancy_id) — only linked infants.
    # site-scoped read (pool-safe): filter infant to THIS site so we never re-stamp foreign rows.
    spark.sql(f"""
    CREATE OR REPLACE TABLE {S3}._mc_base AS
    SELECT pr.person_id AS mother_id,
           cast(i.infant_id AS bigint) AS child_id,
           i.pregnancy_id
    FROM {S}.infant i
    JOIN {S}.pregnancy pr ON pr.pregnancy_id = i.pregnancy_id
    WHERE i.pregnancy_id IS NOT NULL AND i.site_id = '{SITE_ESC}'
    """)

    # mother<->child links (both directions) — per-site delete + insert (pooled table safe)
    spark.sql(f"DELETE FROM {S3}.fact_relationship WHERE site_id = '{SITE_ESC}'")
    spark.sql(f"""
    INSERT INTO {S3}.fact_relationship
    SELECT {PFLD}, mother_id, {PFLD}, child_id, {MOTHER}, '{SITE_ESC}' FROM {S3}._mc_base
    UNION ALL
    SELECT {PFLD}, child_id, {PFLD}, mother_id, {CHILD}, '{SITE_ESC}' FROM {S3}._mc_base
    """)

    # twins: infant pairs sharing a pregnancy (both directions, exclude self)
    spark.sql(f"""
    INSERT INTO {S3}.fact_relationship
    SELECT {PFLD}, a.child_id, {PFLD}, b.child_id, {TWIN}, '{SITE_ESC}'
    FROM {S3}._mc_base a JOIN {S3}._mc_base b
      ON a.pregnancy_id = b.pregnancy_id AND a.child_id <> b.child_id
    """)

    # siblings: infant pairs, same mother, DIFFERENT pregnancies (both directions)
    spark.sql(f"""
    INSERT INTO {S3}.fact_relationship
    SELECT {PFLD}, a.child_id, {PFLD}, b.child_id, {SIBLING}, '{SITE_ESC}'
    FROM {S3}._mc_base a JOIN {S3}._mc_base b
      ON a.mother_id = b.mother_id AND a.pregnancy_id <> b.pregnancy_id AND a.child_id <> b.child_id
    """)

    n = spark.table(f"{S3}.fact_relationship").count()
    by = spark.sql(f"SELECT relationship_concept_id, count(*) c FROM {S3}.fact_relationship GROUP BY 1 ORDER BY 1").collect()
    print("fact_relationship rows", n, "| by concept", [(r['relationship_concept_id'], r['c']) for r in by])

    # cleanup: drop the helper table so it does not pollute the L3 schema
    spark.sql(f"DROP TABLE IF EXISTS {S3}._mc_base")

    # --- sp2_30_episode: stage stable episode ids; finalize dates/events after enrichment ---
    # The final PREBORN episode rows and episode_event coverage are rebuilt in merge_l3 from
    # the fully merged core + enrichment tables. Staging keeps only stable ids/concepts.
    spark.sql(f"DELETE FROM {S3}.episode WHERE site_id = '{SITE_ESC}'")
    spark.sql(f"""
    INSERT INTO {S3}.episode
    SELECT
      {m('episode', 'pr.pregnancy_id')} AS episode_id,
      pr.person_id, 32533 AS episode_concept_id,
      pr.pregnancy_start_date AS episode_start_date,
      CASE WHEN pr.pregnancy_end_date < pr.pregnancy_start_date THEN NULL
           ELSE pr.pregnancy_end_date END AS episode_end_date,
      4299535 AS episode_object_concept_id,
      pr.pregnancy_id, '{SITE_ESC}' AS site_id
    FROM {S}.pregnancy pr
    WHERE pr.site_id = '{SITE_ESC}'
    """)
    en = spark.table(f"{S3}.episode").count()
    ed = spark.sql(f"SELECT count(DISTINCT episode_id) c FROM {S3}.episode").first()["c"]
    assert en == ed, f"episode_id COLLISION {en} vs {ed}"
    spark.sql(f"DELETE FROM {S3}.episode_event WHERE site_id = '{SITE_ESC}'")
    print("staged episode ids", en, "collision-free | final episode_event deferred until enrichment merge")

    # --- sp2_40_ethnicity_l3: MIREDA 3-bucket RACE collapse (L3 only) + finer NHS-letter OBSERVATION ---
    # RACE collapse: derive the 3 MIREDA buckets from the NHS ETHNIC LETTER (race_source_value),
    # NOT from race_concept_id. WHY letter-driven: SP-1 collapses the Mixed letters D-G into the
    # White concept 8527 (keeping the true letter in race_source_value). Deriving from
    # race_concept_id would therefore mis-bucket ~6.5k Mixed persons as White. MIREDA requires
    # Mixed -> RACE 0, so we key off the authoritative letter instead.
    # White  8527      <- A, B, C
    # Asian  8515      <- H, J, K, L, R
    # Black  38003598  <- M, N, P
    # RACE 0           <- D,E,F,G (Mixed), S (Other), Z (Not-stated), null / anything else
    # (LOCAL L3 3-bucket collapse — DISTINCT from the core RACE_CASE crosswalk.)
    RACE_CASE = """CASE upper(trim(race_source_value))
      WHEN 'A' THEN 8527 WHEN 'B' THEN 8527 WHEN 'C' THEN 8527
      WHEN 'H' THEN 8515 WHEN 'J' THEN 8515 WHEN 'K' THEN 8515 WHEN 'L' THEN 8515 WHEN 'R' THEN 8515
      WHEN 'M' THEN 38003598 WHEN 'N' THEN 38003598 WHEN 'P' THEN 38003598
      ELSE 0 END"""

    # MULTI-SITE SAFE: clear only THIS site then INSERT INTO (never INSERT OVERWRITE).
    spark.sql(f"DELETE FROM {S3}.person_race_l3 WHERE site_id = '{SITE_ESC}'")
    spark.sql(f"""
    INSERT INTO {S3}.person_race_l3
    SELECT person_id, {RACE_CASE} AS l3_race_concept_id,
           upper(trim(race_source_value)) AS nhs_letter, '{SITE_ESC}' AS site_id
    FROM {S}.person
    WHERE site_id = '{SITE_ESC}'
    """)

    # finer OBSERVATION: NHS Ethnic Category concept per letter (A..R). S/Z/other -> no row.
    OBS_CASE = """CASE upper(trim(race_source_value))
      WHEN 'A' THEN 700385  -- White British
      WHEN 'B' THEN 700386  -- White Irish
      WHEN 'C' THEN 700387  -- Other White
      WHEN 'D' THEN 700388  -- Mixed White & Black Caribbean
      WHEN 'E' THEN 700389  -- Mixed White & Black African
      WHEN 'F' THEN 700390  -- Mixed White & Asian
      WHEN 'G' THEN 700391  -- Other Mixed
      WHEN 'H' THEN 700362  -- Asian Indian
      WHEN 'J' THEN 700363  -- Asian Pakistani
      WHEN 'K' THEN 700364  -- Asian Bangladeshi
      WHEN 'L' THEN 700365  -- Other Asian
      WHEN 'M' THEN 700366  -- Black Caribbean
      WHEN 'N' THEN 700367  -- Black African
      WHEN 'P' THEN 700368  -- Other Black
      WHEN 'R' THEN 700369  -- Chinese
      ELSE NULL END"""

    spark.sql(f"DELETE FROM {S3}.observation WHERE site_id = '{SITE_ESC}'")
    spark.sql(f"""
    INSERT INTO {S3}.observation (
      observation_id, person_id, observation_concept_id, value_as_concept_id,
      observation_date, observation_type_concept_id, observation_source_value, site_id)
    SELECT
      {m('l3-ethnicity-obs', 'person_id')} AS observation_id,
      person_id, 0 AS observation_concept_id, {OBS_CASE} AS value_as_concept_id,
      -- ethnicity is a registration attribute; MSDS carries no event date for it, so NULL.
      CAST(NULL AS DATE) AS observation_date,
      32817 AS observation_type_concept_id,
      upper(trim(race_source_value)) AS observation_source_value, '{SITE_ESC}' AS site_id
    FROM {S}.person
    WHERE {OBS_CASE} IS NOT NULL AND site_id = '{SITE_ESC}'
    """)

    # verification scoped to THIS site so multi-site pooling doesn't confuse the counts.
    rn = spark.sql(f"SELECT count(*) c FROM {S3}.person_race_l3 WHERE site_id = '{SITE_ESC}'").first()["c"]
    on = spark.sql(f"SELECT count(*) c FROM {S3}.observation WHERE site_id = '{SITE_ESC}'").first()["c"]
    od = spark.sql(f"SELECT count(DISTINCT observation_id) c FROM {S3}.observation WHERE site_id = '{SITE_ESC}'").first()["c"]
    assert on == od, f"observation_id COLLISION {on} vs {od}"
    dist = spark.sql(f"SELECT l3_race_concept_id, count(*) c FROM {S3}.person_race_l3 WHERE site_id = '{SITE_ESC}' GROUP BY 1 ORDER BY 1").collect()
    print("person_race_l3", rn, "| observation", on, "collision-free | race dist",
          [(r['l3_race_concept_id'], r['c']) for r in dist])


In [0]:
print("_common loaded")


In [0]:
# === Unity Catalog documentation — table and column comments ===
# Mirrors the production OMOP Pipeline pattern: a table-comment map plus reusable
# column descriptions, materialised into Unity Catalog metadata for published tables.

PREBORN_TABLE_COMMENTS = {
    "person": "OMOP CDM PERSON records for pregnant people and infants in PREBORN, with source-site and maternal/infant role lineage.",
    "observation_period": "OMOP CDM OBSERVATION_PERIOD records defining the dates during which each PREBORN person is represented by available clinical data.",
    "condition_occurrence": "OMOP CDM CONDITION_OCCURRENCE records generated from PREBORN maternity and neonatal diagnosis data.",
    "procedure_occurrence": "OMOP CDM PROCEDURE_OCCURRENCE records generated from PREBORN maternity and neonatal care activities.",
    "measurement": "OMOP CDM MEASUREMENT records for structured numeric or coded maternity and neonatal measurements.",
    "observation": "OMOP CDM OBSERVATION records for clinical findings and other non-measurement facts in PREBORN.",
    "visit_occurrence": "OMOP CDM VISIT_OCCURRENCE records representing maternity care contacts linked to people and pregnancies.",
    "visit_detail": "OMOP CDM VISIT_DETAIL records representing detailed neonatal or enrichment encounters within a visit.",
    "episode": "OMOP CDM EPISODE records representing pregnancy episodes and their clinical date bounds.",
    "episode_event": "OMOP CDM EPISODE_EVENT bridge linking pregnancy episodes to their constituent clinical events.",
    "fact_relationship": "OMOP CDM FACT_RELATIONSHIP rows linking related PREBORN facts, including pregnancy and infant relationships.",
    "cdm_source": "OMOP CDM_SOURCE metadata describing the PREBORN CDM instance, ETL release and vocabulary versions.",
    "pregnancy": "PREBORN maternity extension containing one row per pregnancy with gestation, outcome, delivery and prior-pregnancy attributes.",
    "infant": "PREBORN maternity extension containing one row per infant with birth outcome, weight, Apgar and congenital-malformation attributes.",
    "concept": "Bundled OMOP CONCEPT vocabulary subset containing concepts used by PREBORN plus the concepts required for referential closure.",
    "concept_relationship": "Bundled OMOP CONCEPT_RELATIONSHIP rows whose source and target concepts are both included in the PREBORN vocabulary subset.",
    "concept_ancestor": "Bundled OMOP CONCEPT_ANCESTOR hierarchy rows needed to provide ancestry for concepts used by PREBORN.",
    "concept_synonym": "Bundled OMOP CONCEPT_SYNONYM rows for concepts included in the PREBORN vocabulary subset.",
    "vocabulary": "OMOP VOCABULARY metadata for the standardised vocabularies represented in the PREBORN concept bundle.",
    "domain": "OMOP DOMAIN metadata defining the clinical and administrative domains used by PREBORN concepts.",
    "concept_class": "OMOP CONCEPT_CLASS metadata defining concept classifications used by the bundled PREBORN vocabulary.",
    "relationship": "OMOP RELATIONSHIP metadata defining relationship types used by the bundled PREBORN vocabulary.",
    "_quarantine": "PREBORN data-quality quarantine containing records excluded from publication and the reason for exclusion.",
    "_publish_lock": "Operational lock table used to enforce a single PREBORN publisher or vocabulary rebuilder at a time.",
    "_publish_journal": "Durable PREBORN publish journal used to recover or roll back interrupted multi-table publications.",
}

PREBORN_COLUMN_DESCRIPTIONS = {
    "*": {
        "person_id": "PREBORN-minted identifier for the person represented by the record.",
        "gender_concept_id": "Foreign key to CONCEPT representing the person's gender.",
        "year_of_birth": "Four-digit year of birth derived from the available birth datetime.",
        "month_of_birth": "Month of birth, from 1 to 12, when available.",
        "day_of_birth": "Day of month of birth, from 1 to 31, when available.",
        "birth_datetime": "Recorded or derived date and time of birth.",
        "race_concept_id": "Foreign key to CONCEPT representing the person's race.",
        "ethnicity_concept_id": "Foreign key to CONCEPT representing the person's ethnicity.",
        "location_id": "Optional foreign key to the OMOP LOCATION table.",
        "provider_id": "Optional foreign key to the OMOP PROVIDER table.",
        "care_site_id": "Optional foreign key to the OMOP CARE_SITE table.",
        "person_source_value": "Source-system identifier retained for the person.",
        "gender_source_value": "Verbatim source value describing gender or sex.",
        "gender_source_concept_id": "Concept identifier corresponding to the source gender value, or 0 when unmapped.",
        "race_source_value": "Verbatim source value describing race or NHS ethnic category.",
        "race_source_concept_id": "Concept identifier corresponding to the source race value, or 0 when unmapped.",
        "ethnicity_source_value": "Verbatim source value describing ethnicity.",
        "ethnicity_source_concept_id": "Concept identifier corresponding to the source ethnicity value, or 0 when unmapped.",
        "site_id": "Source-site identifier used to partition and publish PREBORN data.",
        "person_role": "PREBORN role of the person, such as mother or infant.",

        "observation_period_id": "Unique identifier for the person's observation period.",
        "observation_period_start_date": "First date on which the person is represented in PREBORN.",
        "observation_period_end_date": "Last date on which the person is represented in PREBORN.",
        "period_type_concept_id": "Foreign key to CONCEPT describing how the observation period was derived.",

        "condition_occurrence_id": "Unique identifier for a condition occurrence.",
        "condition_concept_id": "Foreign key to the standard CONDITION concept represented by the record.",
        "condition_start_date": "Date on which the condition was recorded or inferred to start.",
        "condition_start_datetime": "Date and time on which the condition was recorded or inferred to start.",
        "condition_end_date": "Date on which the condition ended, when available.",
        "condition_end_datetime": "Date and time on which the condition ended, when available.",
        "condition_type_concept_id": "Foreign key to CONCEPT describing the provenance of the condition record.",
        "condition_status_concept_id": "Foreign key to CONCEPT describing the status of the condition.",
        "stop_reason": "Source or derived reason the condition ended.",
        "visit_occurrence_id": "Optional foreign key to the associated VISIT_OCCURRENCE.",
        "visit_detail_id": "Optional foreign key to the associated VISIT_DETAIL.",
        "condition_source_value": "Verbatim diagnosis or condition value from the source data.",
        "condition_source_concept_id": "Concept identifier corresponding to the source condition value, or 0 when unmapped.",
        "condition_status_source_value": "Verbatim source value describing condition status.",

        "procedure_occurrence_id": "Unique identifier for a procedure occurrence.",
        "procedure_concept_id": "Foreign key to the standard PROCEDURE concept represented by the record.",
        "procedure_date": "Date on which the procedure was performed.",
        "procedure_datetime": "Date and time on which the procedure was performed.",
        "procedure_end_date": "Date on which the procedure ended, when available.",
        "procedure_end_datetime": "Date and time on which the procedure ended, when available.",
        "procedure_type_concept_id": "Foreign key to CONCEPT describing the provenance of the procedure record.",
        "modifier_concept_id": "Foreign key to CONCEPT for an optional procedure modifier.",
        "quantity": "Number of procedures or units represented by the record.",
        "procedure_source_value": "Verbatim procedure value from the source data.",
        "procedure_source_concept_id": "Concept identifier corresponding to the source procedure value, or 0 when unmapped.",
        "modifier_source_value": "Verbatim source value for the procedure modifier.",

        "measurement_id": "Unique identifier for a measurement.",
        "measurement_concept_id": "Foreign key to the standard MEASUREMENT concept represented by the record.",
        "measurement_date": "Date on which the measurement was taken.",
        "measurement_datetime": "Date and time on which the measurement was taken.",
        "measurement_time": "Source measurement time when supplied separately from the date.",
        "measurement_type_concept_id": "Foreign key to CONCEPT describing the provenance of the measurement.",
        "operator_concept_id": "Foreign key to CONCEPT representing a comparison operator such as less than or greater than.",
        "value_as_number": "Numeric value of the measurement or observation.",
        "value_as_concept_id": "Foreign key to CONCEPT representing a coded result value.",
        "unit_concept_id": "Foreign key to CONCEPT representing the standard unit.",
        "range_low": "Lower bound of the source reference range.",
        "range_high": "Upper bound of the source reference range.",
        "measurement_source_value": "Verbatim measurement code or label from the source data.",
        "measurement_source_concept_id": "Concept identifier corresponding to the source measurement value, or 0 when unmapped.",
        "unit_source_value": "Verbatim unit supplied by the source data.",
        "unit_source_concept_id": "Concept identifier corresponding to the source unit, or 0 when unmapped.",
        "value_source_value": "Verbatim source representation of the result value.",
        "measurement_event_id": "Identifier of the event to which this measurement is linked.",
        "meas_event_field_concept_id": "Concept identifying the event table field referenced by measurement_event_id.",

        "observation_id": "Unique identifier for an observation.",
        "observation_concept_id": "Foreign key to the standard OBSERVATION concept represented by the record.",
        "observation_date": "Date on which the observation was recorded.",
        "observation_datetime": "Date and time on which the observation was recorded.",
        "observation_type_concept_id": "Foreign key to CONCEPT describing the provenance of the observation.",
        "value_as_string": "Text value of the observation when it is not represented numerically or as a concept.",
        "qualifier_concept_id": "Foreign key to CONCEPT qualifying the observation value.",
        "observation_source_value": "Verbatim observation or finding value from the source data.",
        "observation_source_concept_id": "Concept identifier corresponding to the source observation value, or 0 when unmapped.",
        "qualifier_source_value": "Verbatim source value for the observation qualifier.",
        "observation_event_id": "Identifier of the event to which this observation is linked.",
        "obs_event_field_concept_id": "Concept identifying the event table field referenced by observation_event_id.",

        "visit_concept_id": "Foreign key to CONCEPT representing the standard visit type.",
        "visit_start_date": "Date on which the visit started.",
        "visit_start_datetime": "Date and time on which the visit started.",
        "visit_end_date": "Date on which the visit ended.",
        "visit_end_datetime": "Date and time on which the visit ended.",
        "visit_type_concept_id": "Foreign key to CONCEPT describing the provenance of the visit.",
        "visit_source_value": "Verbatim care-contact or visit value from the source data.",
        "visit_source_concept_id": "Concept identifier corresponding to the source visit value, or 0 when unmapped.",
        "admitted_from_concept_id": "Foreign key to CONCEPT representing where the person was admitted from.",
        "admitted_from_source_value": "Verbatim source value describing where the person was admitted from.",
        "discharged_to_concept_id": "Foreign key to CONCEPT representing the discharge destination.",
        "discharged_to_source_value": "Verbatim source value describing the discharge destination.",
        "preceding_visit_occurrence_id": "Identifier of the immediately preceding visit occurrence for the person.",

        "visit_detail_id": "Unique identifier for a detailed visit segment.",
        "visit_detail_concept_id": "Foreign key to CONCEPT representing the standard visit-detail type.",
        "visit_detail_start_date": "Date on which the visit detail started.",
        "visit_detail_start_datetime": "Date and time on which the visit detail started.",
        "visit_detail_end_date": "Date on which the visit detail ended.",
        "visit_detail_end_datetime": "Date and time on which the visit detail ended.",
        "visit_detail_type_concept_id": "Foreign key to CONCEPT describing the provenance of the visit detail.",
        "visit_detail_source_value": "Verbatim source value describing the visit detail.",
        "visit_detail_source_concept_id": "Concept identifier corresponding to the source visit-detail value, or 0 when unmapped.",
        "preceding_visit_detail_id": "Identifier of the immediately preceding visit detail for the person.",
        "parent_visit_detail_id": "Identifier of the containing visit detail, when nested.",

        "episode_id": "Unique identifier for a clinical episode.",
        "episode_concept_id": "Foreign key to CONCEPT representing the standard episode type.",
        "episode_start_date": "Date on which the episode started.",
        "episode_start_datetime": "Date and time on which the episode started.",
        "episode_end_date": "Date on which the episode ended.",
        "episode_end_datetime": "Date and time on which the episode ended.",
        "episode_parent_id": "Identifier of a parent episode, when episodes are nested.",
        "episode_number": "Sequence number of the episode for the person.",
        "episode_object_concept_id": "Foreign key to CONCEPT identifying the domain represented by the episode.",
        "episode_type_concept_id": "Foreign key to CONCEPT describing how the episode was derived.",
        "episode_source_value": "Verbatim source value used to derive the episode.",
        "episode_source_concept_id": "Concept identifier corresponding to the source episode value, or 0 when unmapped.",
        "event_id": "Identifier of the clinical event linked to the episode.",
        "episode_event_field_concept_id": "Concept identifying the event-table field referenced by event_id.",

        "domain_concept_id_1": "Concept identifying the domain of the first related fact.",
        "fact_id_1": "Identifier of the first fact in the relationship.",
        "domain_concept_id_2": "Concept identifying the domain of the second related fact.",
        "fact_id_2": "Identifier of the second fact in the relationship.",
        "relationship_concept_id": "Foreign key to CONCEPT describing the relationship between the two facts.",

        "pregnancy_id": "PREBORN-minted identifier for the pregnancy linked to the record.",
        "pregnancy_start_date": "Estimated start date of the pregnancy.",
        "pregnancy_end_date": "Date on which the pregnancy ended.",
        "gestational_length_in_day": "Gestational length in completed days.",
        "pregnancy_outcome_concept_id": "Foreign key to CONCEPT representing the pregnancy outcome.",
        "pregnancy_mode_delivery_concept_id": "Foreign key to CONCEPT representing the mode of delivery.",
        "pregnancy_single_concept_id": "Foreign key to CONCEPT indicating singleton or multiple pregnancy status.",
        "pregnancy_marital_status_concept_id": "Foreign key to CONCEPT representing marital status during pregnancy.",
        "pregnancy_number_fetuses": "Number of fetuses recorded for the pregnancy.",
        "pregnancy_number_liveborn": "Number of liveborn infants recorded for the pregnancy.",
        "prev_pregnancy_parity_concept_id": "Concept representing previous pregnancy parity.",
        "prev_pregnancy_gravidity": "Number of previous pregnancies.",
        "prev_livebirth_number": "Number of previous live births.",
        "prev_still_misc_number": "Number of previous stillbirths or miscarriages.",
        "pre_pregnancy_bmi": "Body mass index recorded or derived before pregnancy.",
        "pregnancy_folic_concept_id": "Concept indicating folic-acid supplementation.",
        "pregnancy_outcome_source_value": "Verbatim source value for pregnancy outcome.",
        "pregnancy_mode_delivery_source_value": "Verbatim source value for mode of delivery.",
        "pre_pregnancy_bmi_source": "Source system or field used for the pre-pregnancy BMI.",
        "pregnancy_marital_status_source_value": "Verbatim source value for marital status during pregnancy.",

        "infant_id": "PREBORN-minted identifier for the infant and corresponding infant person.",
        "birth_outcome_concept_id": "Foreign key to CONCEPT representing the infant birth outcome.",
        "birth_weight": "Recorded birth weight, in grams where supplied by the source.",
        "birth_con_malformation_concept_id": "Concept indicating whether a congenital malformation was recorded.",
        "birth_apgar": "Recorded five-minute Apgar score.",
        "labour_delivery_id": "Source labour or delivery identifier linking the infant to delivery records.",
        "birth_weight_source": "Source system or field used for birth weight.",
        "birth_apgar_source": "Source system or field used for the Apgar score.",

        "source_table": "Name of the source table or transformation arm that produced the record.",
        "enrichment_tag": "Marker distinguishing enrichment-derived rows from common-core rows.",

        "cdm_source_name": "Name of the PREBORN CDM instance.",
        "cdm_source_abbreviation": "Short abbreviation for the PREBORN CDM instance.",
        "cdm_holder": "Organisation responsible for the PREBORN CDM instance.",
        "source_description": "Description of the source data represented by the CDM.",
        "source_documentation_reference": "Reference to documentation describing the source data.",
        "cdm_etl_reference": "Reference or version identifying the PREBORN ETL implementation.",
        "source_release_date": "Release date of the source data used by the ETL.",
        "cdm_release_date": "Release date of the generated PREBORN CDM.",
        "cdm_version": "OMOP Common Data Model version implemented by PREBORN.",
        "cdm_version_concept_id": "Concept identifier representing the OMOP CDM version.",
        "vocabulary_version": "Version of the OMOP vocabulary bundle loaded into PREBORN.",

        "concept_id": "Unique OMOP identifier for the concept.",
        "concept_name": "Unambiguous, human-readable name of the concept.",
        "domain_id": "Identifier of the OMOP domain to which the concept belongs.",
        "vocabulary_id": "Identifier of the source vocabulary from which the concept was drawn.",
        "concept_class_id": "Identifier of the class used to categorise the concept.",
        "standard_concept": "Flag indicating a Standard concept, Classification concept or source concept.",
        "concept_code": "Code assigned to the concept by its source vocabulary.",
        "valid_start_date": "Date on which the vocabulary record became valid.",
        "valid_end_date": "Date on which the vocabulary record ceased to be valid.",
        "invalid_reason": "Reason the vocabulary record was invalidated; null for a currently valid record.",
        "concept_id_1": "Identifier of the source concept in a directional concept relationship.",
        "concept_id_2": "Identifier of the target concept in a directional concept relationship.",
        "relationship_id": "Identifier of the relationship type.",
        "ancestor_concept_id": "Identifier of the ancestor concept in the OMOP hierarchy.",
        "descendant_concept_id": "Identifier of the descendant concept in the OMOP hierarchy.",
        "min_levels_of_separation": "Minimum number of hierarchy levels between ancestor and descendant.",
        "max_levels_of_separation": "Maximum number of hierarchy levels between ancestor and descendant.",
        "concept_synonym_name": "Alternative name or synonym for the concept.",
        "language_concept_id": "Concept identifier representing the synonym language.",
        "vocabulary_name": "Human-readable name of the vocabulary.",
        "vocabulary_reference": "Reference describing the vocabulary or its source.",
        "vocabulary_version": "Loaded version of the vocabulary.",
        "vocabulary_concept_id": "Concept identifier representing the vocabulary.",
        "domain_name": "Human-readable name of the OMOP domain.",
        "domain_concept_id": "Concept identifier representing the domain.",
        "concept_class_name": "Human-readable name of the concept class.",
        "concept_class_concept_id": "Concept identifier representing the concept class.",
        "relationship_name": "Human-readable name of the relationship type.",
        "is_hierarchical": "Flag indicating whether the relationship is hierarchical.",
        "defines_ancestry": "Flag indicating whether the relationship contributes to CONCEPT_ANCESTOR.",
        "reverse_relationship_id": "Identifier of the relationship used in the reverse direction.",

        "entity": "Type of entity placed in quarantine.",
        "entity_id": "Identifier of the quarantined entity.",
        "reason": "Machine-readable reason the record was quarantined.",
        "detail": "Human-readable diagnostic, publication or recovery detail.",

        "lock_name": "Name of the operational lock.",
        "token": "Unique token owned by the process holding the lock.",
        "purpose": "Description of the operation protected by the lock.",
        "acquired_at": "Timestamp at which the lock was acquired.",
        "publish_token": "Unique token identifying one publication attempt.",
        "table_name": "Name of the table included in the publication attempt.",
        "version_before": "Delta table version recorded before publication for rollback.",
        "state": "Current state of the publication journal entry.",
        "started_at": "Timestamp at which publication of the table began.",
        "finished_at": "Timestamp at which publication or recovery finished.",
    },
    "person": {
        "person_id": "Primary key for a PREBORN person; minted deterministically within the source site."
    },
    "pregnancy": {
        "person_id": "Identifier of the pregnant person linked to this pregnancy.",
        "pregnancy_id": "Primary key for a PREBORN pregnancy; minted deterministically within the source site."
    },
    "infant": {
        "pregnancy_id": "Identifier of the pregnancy that produced the infant.",
        "infant_id": "Primary key for the infant extension and identifier of the corresponding infant PERSON row."
    },
    "episode_event": {
        "episode_id": "Identifier of the pregnancy episode containing the linked event."
    },
    "_publish_journal": {
        "site_id": "Source site being published by this journal entry."
    },
}


def _preborn_sql_literal(value):
    return str(value).replace("'", "''")


def _preborn_quoted_relation(schema, table):
    parts = schema.split(".")
    if len(parts) != 2 or not all(parts):
        raise ValueError(f"expected catalog.schema, got {schema!r}")
    tick = chr(96)
    quote = lambda value: tick + str(value).replace(tick, tick + tick) + tick
    return ".".join([quote(parts[0]), quote(parts[1]), quote(table)])


def apply_preborn_comments(schema, tables=None):
    """Apply table and column comments to existing PREBORN publication tables.

    Metadata is compared first, so repeated calls only execute DDL for comments that
    are absent or have changed. Tables not present in the requested schema are skipped.
    """
    catalog, schema_name = schema.split(".", 1)
    requested = list(tables) if tables is not None else list(PREBORN_TABLE_COMMENTS)
    requested = list(dict.fromkeys(requested))
    if not requested:
        return {"table_comments_updated": 0, "column_comments_updated": 0, "tables_skipped": []}

    tick = chr(96)
    catalog_sql = tick + catalog.replace(tick, tick + tick) + tick
    schema_lit = _preborn_sql_literal(schema_name)
    table_values = ", ".join(f"'{_preborn_sql_literal(table)}'" for table in requested)

    table_rows = spark.sql(f"""
      SELECT table_name, comment AS table_comment
      FROM {catalog_sql}.information_schema.tables
      WHERE table_schema = '{schema_lit}' AND table_name IN ({table_values})
    """).collect()
    column_rows = spark.sql(f"""
      SELECT table_name, column_name, comment AS column_comment, ordinal_position
      FROM {catalog_sql}.information_schema.columns
      WHERE table_schema = '{schema_lit}' AND table_name IN ({table_values})
      ORDER BY table_name, ordinal_position
    """).collect()

    current_tables = {row["table_name"]: row["table_comment"] for row in table_rows}
    current_columns = {
        (row["table_name"], row["column_name"]): row["column_comment"]
        for row in column_rows
    }
    columns_by_table = {}
    for row in column_rows:
        columns_by_table.setdefault(row["table_name"], []).append(row["column_name"])

    table_updates = 0
    column_updates = 0
    skipped = []
    common_descriptions = PREBORN_COLUMN_DESCRIPTIONS["*"]

    for table in requested:
        if table not in current_tables:
            skipped.append(table)
            continue

        relation = _preborn_quoted_relation(schema, table)
        table_comment = PREBORN_TABLE_COMMENTS.get(
            table, f"PREBORN table containing {table.replace('_', ' ')} records."
        )
        if current_tables.get(table) != table_comment:
            spark.sql(
                f"COMMENT ON TABLE {relation} IS '{_preborn_sql_literal(table_comment)}'"
            )
            table_updates += 1

        overrides = PREBORN_COLUMN_DESCRIPTIONS.get(table, {})
        for column in columns_by_table.get(table, []):
            description = overrides.get(column) or common_descriptions.get(column)
            if description is None:
                description = (
                    f"PREBORN {table.replace('_', ' ')} attribute: "
                    f"{column.replace('_', ' ')}."
                )
            if current_columns.get((table, column)) == description:
                continue

            tick = chr(96)
            column_sql = tick + column.replace(tick, tick + tick) + tick
            spark.sql(
                f"ALTER TABLE {relation} ALTER COLUMN {column_sql} "
                f"COMMENT '{_preborn_sql_literal(description)}'"
            )
            column_updates += 1

    print(
        "apply_preborn_comments:",
        schema,
        "| table comments updated", table_updates,
        "| column comments updated", column_updates,
        "| missing tables skipped", skipped,
    )
    return {
        "table_comments_updated": table_updates,
        "column_comments_updated": column_updates,
        "tables_skipped": skipped,
    }


In [0]:
# === DDL: ensure_preborn_schema — single-schema OMOP CDM v5.4 SUBSET (Task 3) ===
# Standard CDM columns taken VERBATIM from /home/eatonb/sql_connections/OMOPCDM_sql_server_5.4_ddl.sql
# with SQL-Server -> Delta type translation: integer->BIGINT (minted ids exceed INT32),
# varchar(n)->STRING, datetime2/datetime->TIMESTAMP, date->DATE, float/Numeric->DOUBLE.
# Plus PREBORN LINEAGE columns: site_id (all clinical); pregnancy_id + source_table (facts);
# enrichment_tag (measurement/observation/visit_detail); person_role (person); pregnancy_id (episode).
# Maternity EXTENSIONS (pregnancy/infant) reuse ensure_core_schema DDL PLUS the ensure_enrich_schema
# *_source fill columns baked in here so limited-mode (no ensure_enrich_schema) has the same shape.
# All CREATE TABLE IF NOT EXISTS (idempotent). Vocab tables created empty (Task 9 fills them).

def ensure_preborn_schema(schema):
    S = schema
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S}")

    # --- person (v5.4: 18 cols) + lineage site_id, person_role ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.person (
      person_id BIGINT, gender_concept_id BIGINT, year_of_birth BIGINT,
      month_of_birth BIGINT, day_of_birth BIGINT, birth_datetime TIMESTAMP,
      race_concept_id BIGINT, ethnicity_concept_id BIGINT, location_id BIGINT,
      provider_id BIGINT, care_site_id BIGINT, person_source_value STRING,
      gender_source_value STRING, gender_source_concept_id BIGINT,
      race_source_value STRING, race_source_concept_id BIGINT,
      ethnicity_source_value STRING, ethnicity_source_concept_id BIGINT,
      site_id STRING, person_role STRING
    ) USING DELTA""")

    # --- observation_period (v5.4: 5 cols) + site_id ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.observation_period (
      observation_period_id BIGINT, person_id BIGINT,
      observation_period_start_date DATE, observation_period_end_date DATE,
      period_type_concept_id BIGINT, site_id STRING
    ) USING DELTA""")

    # --- condition_occurrence (v5.4: 16 cols) + site_id, pregnancy_id, source_table ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.condition_occurrence (
      condition_occurrence_id BIGINT, person_id BIGINT, condition_concept_id BIGINT,
      condition_start_date DATE, condition_start_datetime TIMESTAMP,
      condition_end_date DATE, condition_end_datetime TIMESTAMP,
      condition_type_concept_id BIGINT, condition_status_concept_id BIGINT,
      stop_reason STRING, provider_id BIGINT, visit_occurrence_id BIGINT,
      visit_detail_id BIGINT, condition_source_value STRING,
      condition_source_concept_id BIGINT, condition_status_source_value STRING,
      site_id STRING, pregnancy_id BIGINT, source_table STRING
    ) USING DELTA""")

    # --- procedure_occurrence (v5.4: 16 cols incl. procedure_end_date/datetime) + lineage ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.procedure_occurrence (
      procedure_occurrence_id BIGINT, person_id BIGINT, procedure_concept_id BIGINT,
      procedure_date DATE, procedure_datetime TIMESTAMP,
      procedure_end_date DATE, procedure_end_datetime TIMESTAMP,
      procedure_type_concept_id BIGINT, modifier_concept_id BIGINT, quantity BIGINT,
      provider_id BIGINT, visit_occurrence_id BIGINT, visit_detail_id BIGINT,
      procedure_source_value STRING, procedure_source_concept_id BIGINT,
      modifier_source_value STRING,
      site_id STRING, pregnancy_id BIGINT, source_table STRING
    ) USING DELTA""")

    # --- measurement (v5.4: 23 cols incl. unit_source_concept_id, measurement_event_id,
    #     meas_event_field_concept_id) + site_id, pregnancy_id, enrichment_tag, source_table ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.measurement (
      measurement_id BIGINT, person_id BIGINT, measurement_concept_id BIGINT,
      measurement_date DATE, measurement_datetime TIMESTAMP, measurement_time STRING,
      measurement_type_concept_id BIGINT, operator_concept_id BIGINT,
      value_as_number DOUBLE, value_as_concept_id BIGINT, unit_concept_id BIGINT,
      range_low DOUBLE, range_high DOUBLE, provider_id BIGINT,
      visit_occurrence_id BIGINT, visit_detail_id BIGINT,
      measurement_source_value STRING, measurement_source_concept_id BIGINT,
      unit_source_value STRING, unit_source_concept_id BIGINT, value_source_value STRING,
      measurement_event_id BIGINT, meas_event_field_concept_id BIGINT,
      site_id STRING, pregnancy_id BIGINT, enrichment_tag STRING, source_table STRING
    ) USING DELTA""")

    # --- observation (v5.4: 21 cols incl. value_source_value, observation_event_id,
    #     obs_event_field_concept_id) + site_id, pregnancy_id, enrichment_tag, source_table ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.observation (
      observation_id BIGINT, person_id BIGINT, observation_concept_id BIGINT,
      observation_date DATE, observation_datetime TIMESTAMP,
      observation_type_concept_id BIGINT, value_as_number DOUBLE,
      value_as_string STRING, value_as_concept_id BIGINT, qualifier_concept_id BIGINT,
      unit_concept_id BIGINT, provider_id BIGINT, visit_occurrence_id BIGINT,
      visit_detail_id BIGINT, observation_source_value STRING,
      observation_source_concept_id BIGINT, unit_source_value STRING,
      qualifier_source_value STRING, value_source_value STRING,
      observation_event_id BIGINT, obs_event_field_concept_id BIGINT,
      site_id STRING, pregnancy_id BIGINT, enrichment_tag STRING, source_table STRING
    ) USING DELTA""")

    # --- visit_occurrence (v5.4: 17 cols; admitted_from_*/discharged_to_*) + site_id, pregnancy_id ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.visit_occurrence (
      visit_occurrence_id BIGINT, person_id BIGINT, visit_concept_id BIGINT,
      visit_start_date DATE, visit_start_datetime TIMESTAMP,
      visit_end_date DATE, visit_end_datetime TIMESTAMP, visit_type_concept_id BIGINT,
      provider_id BIGINT, care_site_id BIGINT, visit_source_value STRING,
      visit_source_concept_id BIGINT, admitted_from_concept_id BIGINT,
      admitted_from_source_value STRING, discharged_to_concept_id BIGINT,
      discharged_to_source_value STRING, preceding_visit_occurrence_id BIGINT,
      site_id STRING, pregnancy_id BIGINT
    ) USING DELTA""")

    # --- visit_detail (v5.4: 19 cols; admitted_from_*/discharged_to_*, parent_visit_detail_id)
    #     + site_id, pregnancy_id, enrichment_tag, source_table ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.visit_detail (
      visit_detail_id BIGINT, person_id BIGINT, visit_detail_concept_id BIGINT,
      visit_detail_start_date DATE, visit_detail_start_datetime TIMESTAMP,
      visit_detail_end_date DATE, visit_detail_end_datetime TIMESTAMP,
      visit_detail_type_concept_id BIGINT, provider_id BIGINT, care_site_id BIGINT,
      visit_detail_source_value STRING, visit_detail_source_concept_id BIGINT,
      admitted_from_concept_id BIGINT, admitted_from_source_value STRING,
      discharged_to_source_value STRING, discharged_to_concept_id BIGINT,
      preceding_visit_detail_id BIGINT, parent_visit_detail_id BIGINT,
      visit_occurrence_id BIGINT,
      site_id STRING, pregnancy_id BIGINT, enrichment_tag STRING, source_table STRING
    ) USING DELTA""")

    # --- episode (v5.4: 13 cols) + pregnancy_id, site_id ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.episode (
      episode_id BIGINT, person_id BIGINT, episode_concept_id BIGINT,
      episode_start_date DATE, episode_start_datetime TIMESTAMP,
      episode_end_date DATE, episode_end_datetime TIMESTAMP, episode_parent_id BIGINT,
      episode_number BIGINT, episode_object_concept_id BIGINT,
      episode_type_concept_id BIGINT, episode_source_value STRING,
      episode_source_concept_id BIGINT, pregnancy_id BIGINT, site_id STRING
    ) USING DELTA""")

    # --- episode_event (v5.4: 3 cols) + source_table, site_id ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.episode_event (
      episode_id BIGINT, event_id BIGINT, episode_event_field_concept_id BIGINT,
      source_table STRING, site_id STRING
    ) USING DELTA""")

    # --- fact_relationship (v5.4: 5 cols) + site_id ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.fact_relationship (
      domain_concept_id_1 BIGINT, fact_id_1 BIGINT, domain_concept_id_2 BIGINT,
      fact_id_2 BIGINT, relationship_concept_id BIGINT, site_id STRING
    ) USING DELTA""")

    # --- cdm_source (v5.4: 11 cols) ---
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.cdm_source (
      cdm_source_name STRING, cdm_source_abbreviation STRING, cdm_holder STRING,
      source_description STRING, source_documentation_reference STRING,
      cdm_etl_reference STRING, source_release_date DATE, cdm_release_date DATE,
      cdm_version STRING, cdm_version_concept_id BIGINT, vocabulary_version STRING
    ) USING DELTA""")

    # === Maternity EXTENSIONS (verbatim from ensure_core_schema + ensure_enrich ADD COLUMNS) ===
    # pregnancy: core 20 cols + enrich fill/source (pre_pregnancy_bmi_source,
    # pregnancy_marital_status_source_value) baked in so limited mode has the same shape.
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.pregnancy (
      person_id BIGINT, pregnancy_id BIGINT,
      pregnancy_start_date DATE, pregnancy_end_date DATE,
      gestational_length_in_day INT,
      pregnancy_outcome_concept_id BIGINT, pregnancy_mode_delivery_concept_id BIGINT,
      pregnancy_single_concept_id BIGINT, pregnancy_marital_status_concept_id BIGINT,
      pregnancy_number_fetuses INT, pregnancy_number_liveborn INT,
      prev_pregnancy_parity_concept_id BIGINT, prev_pregnancy_gravidity INT,
      prev_livebirth_number INT, prev_still_misc_number INT,
      pre_pregnancy_bmi DOUBLE, pregnancy_folic_concept_id BIGINT,
      pregnancy_outcome_source_value STRING, pregnancy_mode_delivery_source_value STRING,
      site_id STRING,
      pre_pregnancy_bmi_source STRING, pregnancy_marital_status_source_value STRING
    ) USING DELTA""")

    # infant: core 8 cols + enrich fill/source (birth_weight_source, birth_apgar_source).
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.infant (
      pregnancy_id BIGINT, infant_id BIGINT,
      birth_outcome_concept_id BIGINT, birth_weight DOUBLE,
      birth_con_malformation_concept_id BIGINT, birth_apgar INT,
      labour_delivery_id STRING, site_id STRING,
      birth_weight_source STRING, birth_apgar_source STRING
    ) USING DELTA""")

    # === VOCAB tables (columns exact per canonical DDL; Task 9 fills them; created empty) ===
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.concept (
      concept_id BIGINT, concept_name STRING, domain_id STRING, vocabulary_id STRING,
      concept_class_id STRING, standard_concept STRING, concept_code STRING,
      valid_start_date DATE, valid_end_date DATE, invalid_reason STRING
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.concept_relationship (
      concept_id_1 BIGINT, concept_id_2 BIGINT, relationship_id STRING,
      valid_start_date DATE, valid_end_date DATE, invalid_reason STRING
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.concept_ancestor (
      ancestor_concept_id BIGINT, descendant_concept_id BIGINT,
      min_levels_of_separation BIGINT, max_levels_of_separation BIGINT
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.concept_synonym (
      concept_id BIGINT, concept_synonym_name STRING, language_concept_id BIGINT
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.vocabulary (
      vocabulary_id STRING, vocabulary_name STRING, vocabulary_reference STRING,
      vocabulary_version STRING, vocabulary_concept_id BIGINT
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.domain (
      domain_id STRING, domain_name STRING, domain_concept_id BIGINT
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.concept_class (
      concept_class_id STRING, concept_class_name STRING, concept_class_concept_id BIGINT
    ) USING DELTA""")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}.relationship (
      relationship_id STRING, relationship_name STRING, is_hierarchical STRING,
      defines_ancestry STRING, reverse_relationship_id STRING, relationship_concept_id BIGINT
    ) USING DELTA""")

    # === quarantine (Tasks 6-7) ===
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {S}._quarantine (
      site_id STRING, entity STRING, entity_id BIGINT, reason STRING, detail STRING
    ) USING DELTA""")



In [0]:
# === Task 5-9 STUBS — real bodies land in their own tasks; defined here so _common loads and
# build_cdm's call graph resolves. Signatures are FIXED (build_cdm calls them by these arities). ===

def _build_barts_enrichment(site, src, cerner, stg):  # TASK 5
    # Barts Cerner/BadgerNet enrichment: server-side reschema'd port of maternity_omop/full_omop
    # (sp1_40 cerner_bridge + sp3_10 baby_bridge + sp3_20/30 fills + sp3_40 cerner_maternal + sp3_50 badgernet).
    # CORE->stg, ENRICH->stg+'_enrich', CERNER[..]->cerner[..], SRC[..]->src[..]. Logic UNCHANGED; asserts kept.
    site_esc = site.replace("'", "''")
    def m(grain, *p):
        return mid(grain, site, *p)

    ensure_enrich_schema(stg + "_enrich", stg)


    # (1) sp1_40_cerner_bridge — minted <-> Cerner crosswalk (artifact for enrichment; NOT applied)
    # INT (mat_pregnancy.PREGNANCY_ID) vs STRING (msd101.PREGNANCYID) -> explicit CAST (correction #12)
    spark.sql(f"""
    CREATE OR REPLACE TABLE {stg}.cerner_bridge AS
    WITH dd AS (
      SELECT * FROM (
        SELECT p.PREGNANCYID, p.LPIDMOTHER,
          row_number() OVER (PARTITION BY p.PREGNANCYID
            ORDER BY CASE WHEN p.ADC_UPDT IS NULL THEN 1 ELSE 0 END, p.ADC_UPDT DESC) AS rn
        FROM {src["msd101_pregbook"]} p
        WHERE p.PREGNANCYID IS NOT NULL AND trim(p.PREGNANCYID) <> '0'
      ) WHERE rn = 1
    )
    SELECT
      {m('pregnancy', 'p.PREGNANCYID')} AS minted_pregnancy_id,
      p.PREGNANCYID AS msds_pregnancy_id,
      mp.PERSON_ID  AS cerner_person_id,
      mp.PREGNANCY_ID AS cerner_pregnancy_id
    FROM dd p
    LEFT JOIN {cerner["mat_pregnancy"]} mp
      ON CAST(mp.PREGNANCY_ID AS STRING) = p.PREGNANCYID
    """)
    tot = spark.table(f"{stg}.cerner_bridge").count()
    matched = spark.sql(f"SELECT count(*) c FROM {stg}.cerner_bridge WHERE cerner_person_id IS NOT NULL").first()["c"]
    print("bridge rows", tot, "cerner-matched", matched,
          "match_pct", round(matched/(tot or 1)*100,1))


    # (2) sp3_10_baby_bridge — infant <-> mat_birth (per-baby) <-> nnu_episodes.
    # Multiple births are linked by the recorded birth order on BOTH systems. A cryptographic
    # infant_id is deliberately never used as an ordering proxy. Ambiguous multiples remain
    # unlinked rather than receiving another infant's weight, Apgar, NHS number, or NNU record.
    spark.sql(f"DELETE FROM {stg}_enrich.baby_bridge WHERE site_id='{site_esc}'")
    spark.sql(f"""
    INSERT INTO {stg}_enrich.baby_bridge (
      site_id, minted_infant_id, minted_pregnancy_id, cerner_pregnancy_id,
      cerner_baby_person_id, cerner_pregnancy_child_id, birth_order,
      msds_birth_order, link_method, nnu_entity_id, baby_nhs)
    WITH raw_ident AS (
      SELECT b.*,
        CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN 'lpid' ELSE 'fallback' END AS _idkind,
        CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL
             THEN ORGIDLOCALPATIENTIDBABY ELSE LABOURDELIVERYID END AS _k1,
        CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL
             THEN LPIDBABY ELSE cast(BIRTHORDERMATERNITYSUS as string) END AS _k2
      FROM {src["msd401_babydemo"]} b
    ),
    raw_inf AS (
      SELECT infant_id, msds_birth_order FROM (
        SELECT {m('infant', 'x._idkind', 'x._k1', 'x._k2')} AS infant_id,
               try_cast(x.BIRTHORDERMATERNITYSUS AS INT) AS msds_birth_order,
               row_number() OVER (
                 PARTITION BY x._idkind, x._k1, x._k2
                 ORDER BY x.LABOURDELIVERYID,
                          CASE WHEN x.ADC_UPDT IS NULL THEN 1 ELSE 0 END,
                          x.ADC_UPDT DESC) AS rn
        FROM raw_ident x
      ) WHERE rn=1
    ),
    inf AS (
      SELECT i.infant_id, i.pregnancy_id, b.cerner_pregnancy_id,
             ri.msds_birth_order,
             count(*) OVER (PARTITION BY i.pregnancy_id) AS infant_count
      FROM {stg}.infant i
      JOIN {stg}.cerner_bridge b ON b.minted_pregnancy_id = i.pregnancy_id
      LEFT JOIN raw_inf ri ON ri.infant_id=i.infant_id
      WHERE i.site_id='{site_esc}'
    ),
    mb0 AS (
      SELECT PREGNANCY_ID, BABY_PERSON_ID,
             cast(PREGNANCY_CHILD_ID as string) AS child_id,
             try_cast(BIRTH_ODR_NBR as int) AS birth_order,
             regexp_replace(NHS, '[^0-9]', '') AS baby_nhs,
             count(*) OVER (PARTITION BY PREGNANCY_ID) AS mb_count,
             count(*) OVER (PARTITION BY PREGNANCY_ID, try_cast(BIRTH_ODR_NBR as int)) AS order_count,
             row_number() OVER (
               PARTITION BY PREGNANCY_ID, try_cast(BIRTH_ODR_NBR as int)
               ORDER BY cast(PREGNANCY_CHILD_SEQ_ID as bigint), BABY_PERSON_ID) AS order_rn
      FROM {cerner["mat_birth"]}
      WHERE DELETE_IND IS NULL OR cast(DELETE_IND as string) IN ('0','0.0')
    ),
    mb AS (
      SELECT * FROM mb0 WHERE order_rn=1
    ),
    nn AS (
      SELECT regexp_replace(NationalIDBaby, '[^0-9]', '') AS baby_nhs,
             max(EntityID) AS nnu_entity_id
      FROM {cerner["nnu_episodes"]}
      WHERE NationalIDBaby IS NOT NULL
      GROUP BY regexp_replace(NationalIDBaby, '[^0-9]', '')
    )
    SELECT '{site_esc}' AS site_id, inf.infant_id, inf.pregnancy_id,
           inf.cerner_pregnancy_id, mb.BABY_PERSON_ID, mb.child_id, mb.birth_order,
           inf.msds_birth_order,
           CASE WHEN inf.msds_birth_order IS NOT NULL
                     AND mb.birth_order=inf.msds_birth_order THEN 'birth_order'
                WHEN inf.infant_count=1 AND mb.mb_count=1 THEN 'singleton'
           END AS link_method,
           nn.nnu_entity_id, mb.baby_nhs
    FROM inf
    LEFT JOIN mb
      ON mb.PREGNANCY_ID=inf.cerner_pregnancy_id
     AND (
       (inf.msds_birth_order IS NOT NULL
        AND mb.birth_order=inf.msds_birth_order
        AND mb.order_count=1)
       OR (inf.infant_count=1 AND mb.mb_count=1)
     )
    LEFT JOIN nn ON nn.baby_nhs=mb.baby_nhs AND mb.baby_nhs<>''
    """)

    tot = spark.sql(f"SELECT count(*) c FROM {stg}_enrich.baby_bridge WHERE site_id='{site_esc}'").first()["c"]
    mb_hit = spark.sql(f"""SELECT count(*) c FROM {stg}_enrich.baby_bridge
      WHERE site_id='{site_esc}' AND cerner_baby_person_id IS NOT NULL""").first()["c"]
    nn_hit = spark.sql(f"""SELECT count(*) c FROM {stg}_enrich.baby_bridge
      WHERE site_id='{site_esc}' AND nnu_entity_id IS NOT NULL""").first()["c"]
    d = spark.sql(f"""SELECT count(DISTINCT minted_infant_id) c
      FROM {stg}_enrich.baby_bridge WHERE site_id='{site_esc}'""").first()["c"]
    bad_order = spark.sql(f"""SELECT count(*) c FROM {stg}_enrich.baby_bridge
      WHERE site_id='{site_esc}' AND link_method='birth_order'
        AND msds_birth_order <> birth_order""").first()["c"]
    if tot != d:
        raise RuntimeError(f"baby_bridge NOT unique per infant: {tot} rows vs {d} infants")
    if bad_order:
        raise RuntimeError(f"baby_bridge has {bad_order} cross-child birth-order links")
    print(f"baby_bridge {tot} infants | safely linked mat_birth {mb_hit} "
          f"({round(100*mb_hit/(tot or 1),1)}%) | nnu {nn_hit} "
          f"({round(100*nn_hit/(tot or 1),1)}%)")

    # (3) sp3_20_fill_bmi_marital — idempotent clear-then-refill of pregnancy BMI + marital.
    # (a) RESET this site's fills so stale/invalid source rows cannot leave carryover
    spark.sql(f"""UPDATE {stg}.pregnancy
      SET pre_pregnancy_bmi = NULL, pre_pregnancy_bmi_source = NULL,
          pregnancy_marital_status_concept_id = 0, pregnancy_marital_status_source_value = NULL
      WHERE site_id='{site_esc}'""")

    # (b) REFILL from current Cerner source via the bridge.
    spark.sql(f"""
    MERGE INTO {stg}.pregnancy AS t
    USING (
      SELECT b.minted_pregnancy_id AS pid,
             CASE WHEN mp.WT_BOOKING_KG > 0 AND mp.HT_BOOKING_CM > 0
                       AND (mp.WT_BOOKING_KG / pow(mp.HT_BOOKING_CM/100.0, 2)) BETWEEN 10 AND 80
                  THEN mp.WT_BOOKING_KG / pow(mp.HT_BOOKING_CM/100.0, 2) END AS bmi_calc,
             CASE WHEN try_cast(mp.BMI_BOOKING_DESC as double) BETWEEN 10 AND 80
                  THEN try_cast(mp.BMI_BOOKING_DESC as double) END AS bmi_desc,
             CASE WHEN mpr.MARITAL_TYPE_CD = 309239 THEN 4053842
                  WHEN mpr.MARITAL_TYPE_CD = 309237 THEN 4338692
                  WHEN mpr.MARITAL_TYPE_CD = 309241 THEN 4143188
                  WHEN mpr.MARITAL_TYPE_CD = 309236 THEN 4069297
                  WHEN mpr.MARITAL_TYPE_CD = 309238 THEN 4027529
                  ELSE 0 END AS marital_cid,
             cast(mpr.MARITAL_TYPE_CD as string) AS marital_src
      FROM {stg}.cerner_bridge b
      JOIN {cerner["mat_pregnancy"]} mp ON mp.PREGNANCY_ID = b.cerner_pregnancy_id
      LEFT JOIN {cerner["mill_person"]} mpr ON mpr.PERSON_ID = mp.PERSON_ID
    ) AS s
    ON t.pregnancy_id = s.pid AND t.site_id = '{site_esc}'
    WHEN MATCHED THEN UPDATE SET
      t.pre_pregnancy_bmi = coalesce(s.bmi_calc, s.bmi_desc),
      t.pre_pregnancy_bmi_source = CASE WHEN s.bmi_calc IS NOT NULL THEN 'cerner_mat_pregnancy_htwt'
                                        WHEN s.bmi_desc IS NOT NULL THEN 'cerner_mat_pregnancy_bmidesc' END,
      t.pregnancy_marital_status_concept_id = s.marital_cid,
      t.pregnancy_marital_status_source_value = s.marital_src
    """)

    tot = spark.sql(f"SELECT count(*) c FROM {stg}.pregnancy WHERE site_id='{site_esc}'").first()["c"]
    bmi = spark.sql(f"SELECT count(pre_pregnancy_bmi) c FROM {stg}.pregnancy WHERE site_id='{site_esc}'").first()["c"]
    mar = spark.sql(f"SELECT count(*) c FROM {stg}.pregnancy WHERE site_id='{site_esc}' AND pregnancy_marital_status_concept_id<>0").first()["c"]
    bad = spark.sql(f"""SELECT count(*) c FROM {stg}.pregnancy
      WHERE site_id='{site_esc}' AND pre_pregnancy_bmi IS NOT NULL AND pre_pregnancy_bmi_source IS NULL""").first()["c"]
    assert bad == 0, f"{bad} BMI values without provenance marker"
    oor = spark.sql(f"""SELECT count(*) c FROM {stg}.pregnancy
      WHERE site_id='{site_esc}' AND pre_pregnancy_bmi IS NOT NULL
        AND (pre_pregnancy_bmi < 10 OR pre_pregnancy_bmi > 80)""").first()["c"]
    assert oor == 0, f"{oor} BMI out of range"
    print(f"pregnancy {tot} | bmi {bmi} ({round(100*bmi/(tot or 1),1)}%) | marital-coded {mar} ({round(100*mar/(tot or 1),1)}%)")


    # (4) sp3_30_fill_infant — idempotent clear-then-refill of infant birth_weight + birth_apgar.
    # NOTE: mat_birth.BIRTH_WT is a STRING with a unit suffix e.g. '4122.00 g' (NOT numeric),
    # so parse the leading numeric with regexp_extract + try_cast before range-filtering.
    # (a) RESET
    spark.sql(f"""UPDATE {stg}.infant
      SET birth_weight = NULL, birth_weight_source = NULL,
          birth_apgar = NULL, birth_apgar_source = NULL
      WHERE site_id='{site_esc}'""")

    # (b) REFILL via baby_bridge -> mat_birth on the CHILD KEY, deduped to one row per infant.
    spark.sql(f"""
    MERGE INTO {stg}.infant AS t
    USING (
      SELECT iid, bw, ap FROM (
        SELECT bb.minted_infant_id AS iid,
               CASE WHEN try_cast(regexp_extract(mb.BIRTH_WT, '([0-9.]+)', 1) as double) BETWEEN 100 AND 8000
                    THEN try_cast(regexp_extract(mb.BIRTH_WT, '([0-9.]+)', 1) as double) END AS bw,
               CASE WHEN try_cast(mb.APGAR_5MIN as int) BETWEEN 0 AND 10
                    THEN try_cast(mb.APGAR_5MIN as int) END AS ap,
               row_number() OVER (
                 PARTITION BY bb.minted_infant_id
                 ORDER BY CASE WHEN try_cast(regexp_extract(mb.BIRTH_WT, '([0-9.]+)', 1) as double) BETWEEN 100 AND 8000 THEN 0 ELSE 1 END,
                          try_cast(regexp_extract(mb.BIRTH_WT, '([0-9.]+)', 1) as double) DESC NULLS LAST) AS rk
        FROM {stg}_enrich.baby_bridge bb
        JOIN {cerner["mat_birth"]} mb
          ON mb.PREGNANCY_ID = bb.cerner_pregnancy_id
         AND cast(mb.PREGNANCY_CHILD_ID as string) = bb.cerner_pregnancy_child_id
        WHERE bb.site_id='{site_esc}' AND bb.cerner_pregnancy_child_id IS NOT NULL
      ) WHERE rk = 1
    ) AS s
    ON t.infant_id = s.iid AND t.site_id = '{site_esc}'
    WHEN MATCHED THEN UPDATE SET
      t.birth_weight = s.bw,
      t.birth_weight_source = CASE WHEN s.bw IS NOT NULL THEN 'cerner_mat_birth' END,
      t.birth_apgar = s.ap,
      t.birth_apgar_source = CASE WHEN s.ap IS NOT NULL THEN 'cerner_mat_birth' END
    """)

    tot = spark.sql(f"SELECT count(*) c FROM {stg}.infant WHERE site_id='{site_esc}'").first()["c"]
    bw = spark.sql(f"SELECT count(birth_weight) c FROM {stg}.infant WHERE site_id='{site_esc}'").first()["c"]
    ap = spark.sql(f"SELECT count(birth_apgar) c FROM {stg}.infant WHERE site_id='{site_esc}'").first()["c"]
    for col, src in [("birth_weight","birth_weight_source"),("birth_apgar","birth_apgar_source")]:
        bad = spark.sql(f"""SELECT count(*) c FROM {stg}.infant
          WHERE site_id='{site_esc}' AND {col} IS NOT NULL AND {src} IS NULL""").first()["c"]
        assert bad == 0, f"{bad} {col} without provenance"
    print(f"infant {tot} | birth_weight {bw} ({round(100*bw/(tot or 1),1)}%) | apgar {ap} ({round(100*ap/(tot or 1),1)}%)")


    # (5) sp3_40_cerner_maternal — Cerner mat_pregnancy enrichment facts (observation + measurement).
    # NOTE: mp.* re-introduces PREGNANCY_ID/PERSON_ID which collide (case-insensitively)
    # with the explicit pregnancy_id/person_id aliases -> AMBIGUOUS_REFERENCE. Drop them
    # from the wildcard; the aliased minted/person ids are what we use downstream.
    BASE = f"""
      SELECT b.minted_pregnancy_id AS pregnancy_id, pr.person_id,
             b.cerner_pregnancy_id AS cpid,
             to_date(mp.FIRST_ANTENATAL_ASSESSMENT_DT_TM) AS clin_date,
             mp.* EXCEPT (PREGNANCY_ID, PERSON_ID)
      FROM {stg}.cerner_bridge b
      JOIN {stg}.pregnancy pr ON pr.pregnancy_id = b.minted_pregnancy_id AND pr.site_id='{site_esc}'
      JOIN {cerner["mat_pregnancy"]} mp ON mp.PREGNANCY_ID = b.cerner_pregnancy_id
    """

    OBS_FIELDS = [
      ("HIV_SCREEN_DESC","HIV screen"), ("HEPB_ANTIGEN_SCREEN_DESC","HepB antigen screen"),
      ("SYPHILIS_SCREEN_DESC","Syphilis screen"), ("RUBELLA_IGG_SCREEN_DESC","Rubella IgG screen"),
      ("HAEMAGLOBINOPATHY_SCREEN_DESC","Haemoglobinopathy screen"),
      ("DOWNS_SERUM_SCREEN_DESC","Downs serum screen"),
      ("SMOKE_BOOKING_DESC","Smoking at booking"), ("SMOKING_STATUS_DEL_DESC","Smoking at delivery"),
      ("LAST_METHOD_CONTRACEPT_DESC","Last contraception method"),
      ("ACCOMMODATION_DESC","Accommodation status"), ("EMPLOY_STATUS_DESC","Employment status"),
      ("SUPPORT_STATUS_DESC","Support status"), ("EDUCATION_DESC","Education status"),
      ("PLANNED_DEL_LOC_PREG_DESC","Planned delivery location"),
      ("LAB_ONSET_METHOD_DESC","Labour onset method"), ("AUGMENTATION_DESC","Labour augmentation"),
      ("EPISIOTOMY_DESC","Episiotomy"), ("PERINEAL_TRAUMA_DESC","Perineal trauma"),
      ("ANALGESIA_LAB_DESC","Labour analgesia"), ("ANAESTHESIA_DEL_DESC","Delivery anaesthesia"),
    ]
    MEAS_FIELDS = [
      ("LAB_TOTAL","Total labour minutes",None), ("LAB_STAGE1_TOTAL","Stage 1 minutes",None),
      ("LAB_STAGE2_TOTAL","Stage 2 minutes",None), ("LAB_STAGE3_TOTAL","Stage 3 minutes",None),
      ("TOTAL_BLOOD_LOSS","Total blood loss ml","ml"),
    ]

    def emit_obs():
        return " UNION ALL ".join(f"""
          SELECT {m('enrich-cerner-obs','cpid',f"'{col}'",'clin_date')} AS observation_id,
                 person_id, 0 AS observation_concept_id, clin_date AS observation_date,
                 pregnancy_id, CAST(NULL AS BIGINT) AS infant_id,
                 'mat_pregnancy' AS source_table, '{lab}: ' || cast({col} as string) AS observation_source_value,
                 0 AS observation_source_concept_id, CAST(NULL AS DOUBLE) AS value_as_number,
                 cast({col} as string) AS value_as_string, 'enrichment' AS enrichment_tag,
                 '{site_esc}' AS site_id
          FROM base WHERE nullif(trim(cast({col} as string)),'') IS NOT NULL""" for col, lab in OBS_FIELDS)

    def emit_meas():
        sels = []
        for col, lab, unit in MEAS_FIELDS:
            u = f"'{unit}'" if unit else "CAST(NULL AS STRING)"
            sels.append(f"""
              SELECT {m('enrich-cerner-meas','cpid',f"'{col}'",'clin_date')} AS measurement_id,
                     person_id, 0 AS measurement_concept_id, clin_date AS measurement_date,
                     pregnancy_id, CAST(NULL AS BIGINT) AS infant_id, 'mat_pregnancy' AS source_table,
                     '{lab}' AS measurement_source_value, 0 AS measurement_source_concept_id,
                     try_cast({col} as double) AS value_as_number, {u} AS unit_source_value,
                     'enrichment' AS enrichment_tag, '{site_esc}' AS site_id
              FROM base WHERE try_cast({col} as double) IS NOT NULL""")
        return " UNION ALL ".join(sels)

    spark.sql(f"CREATE OR REPLACE TEMP VIEW base AS {BASE}")
    for tgt, sqlbody, idcol in [("observation", emit_obs(), "observation_id"),
                                ("measurement", emit_meas(), "measurement_id")]:
        spark.sql(f"DELETE FROM {stg}_enrich.{tgt} WHERE site_id='{site_esc}' AND source_table='mat_pregnancy'")
        spark.sql(f"INSERT INTO {stg}_enrich.{tgt} {sqlbody}")
        n = spark.sql(f"SELECT count(*) c FROM {stg}_enrich.{tgt} WHERE site_id='{site_esc}' AND source_table='mat_pregnancy'").first()["c"]
        d = spark.sql(f"SELECT count(DISTINCT {idcol}) c FROM {stg}_enrich.{tgt} WHERE site_id='{site_esc}' AND source_table='mat_pregnancy'").first()["c"]
        assert n == d, f"{tgt} id COLLISION {n} vs {d}"
        print(tgt, "mat_pregnancy rows", n, "collision-free")


    # (6) sp3_50_badgernet_neonatal — G15: nnu_episodes + nnu_routineexamination + nnu_nccmds -> OMOP.
    # NOTE 1: serverless chokes on chained TEMP VIEWs (INTERNAL_ERROR), so link/base are inlined as
    #   CTEs in every INSERT rather than CREATE TEMP VIEW.
    # NOTE 2: baby_bridge maps 22 nnu_entity_ids to TWO distinct minted_infant_ids each (twins sharing
    #   one neonatal record). person_id = infant_id, so each twin legitimately gets its own copy of the
    #   shared nnu fact. The natural key therefore includes infant_id (the person), otherwise the two
    #   twins collide on the same (eid, field, dt) key. Including infant_id keeps them distinct without
    #   dropping any legitimate row.

    # link CTE: infants with an nnu entity id, at this site
    LINK_CTE = f"""link AS (
        SELECT bb.minted_infant_id AS infant_id, bb.minted_pregnancy_id AS pregnancy_id, bb.nnu_entity_id AS eid
        FROM {stg}_enrich.baby_bridge bb
        JOIN {stg}.infant i ON i.infant_id = bb.minted_infant_id AND i.site_id='{site_esc}'
        WHERE bb.site_id='{site_esc}' AND bb.nnu_entity_id IS NOT NULL
      )"""

    # base CTEs (per source). Guarded against AMBIGUOUS_REFERENCE by verifying the nnu tables have no
    # infant_id/pregnancy_id/eid columns (they don't); <tbl>.* is safe.
    BASE_EP = f"""base_ep AS (
        SELECT l.infant_id, l.pregnancy_id, l.eid, to_date(ne.AdmitTime) AS dt, ne.*
        FROM link l JOIN {cerner["nnu_episodes"]} ne ON ne.EntityID = l.eid
      )"""
    BASE_EX = f"""base_ex AS (
        SELECT l.infant_id, l.pregnancy_id, l.eid, to_date(rx.DateOfExamination) AS dt, rx.*
        FROM link l JOIN {cerner["nnu_routineexamination"]} rx ON rx.EntityID = l.eid
      )"""

    EP_MEAS = [("BirthHeadCircumference","Birth head circumference","cm"),
               ("AdmitHeadCircumference","Admission head circumference","cm"),
               ("AdmitWeight","Admission weight","g"),
               ("AdmitTemperature","Admission temperature","Cel"),
               ("BirthLength","Birth length","cm")]
    EP_OBS = [("HeadScanFirstResult","First cranial USS result"),
              ("HeadScanLastResult","Last cranial USS result"),
              ("FinalNNUOutcome","Final NNU outcome"),
              ("Seizures","Seizures"), ("HIEGrade","HIE grade")]
    EX_MEAS = [("HeadCircumference","NIPE head circumference","cm")]
    EX_OBS = [("Heart","NIPE heart"),("Femoral","NIPE femoral pulses"),("Hips","NIPE hips"),
              ("RedReflex","NIPE red reflex"),("Tone","NIPE tone"),("Spine","NIPE spine"),
              ("Overall","NIPE overall")]

    def emit_nnu_meas(view, src, fields):
        return " UNION ALL ".join(f"""
          SELECT {m('enrich-nnu-meas','infant_id','eid',f"'{src}:{c}'",'dt')} AS measurement_id,
                 infant_id AS person_id, 0 AS measurement_concept_id, dt AS measurement_date,
                 pregnancy_id, infant_id, '{src}' AS source_table,
                 '{lab}' AS measurement_source_value, 0 AS measurement_source_concept_id,
                 try_cast({c} as double) AS value_as_number, '{u}' AS unit_source_value,
                 'enrichment' AS enrichment_tag, '{site_esc}' AS site_id
          FROM {view} WHERE try_cast({c} as double) IS NOT NULL""" for c, lab, u in fields)

    def emit_nnu_obs(view, src, fields):
        return " UNION ALL ".join(f"""
          SELECT {m('enrich-nnu-obs','infant_id','eid',f"'{src}:{c}'",'dt')} AS observation_id,
                 infant_id AS person_id, 0 AS observation_concept_id, dt AS observation_date,
                 pregnancy_id, infant_id, '{src}' AS source_table,
                 '{lab}: ' || cast({c} as string) AS observation_source_value,
                 0 AS observation_source_concept_id, CAST(NULL AS DOUBLE) AS value_as_number,
                 cast({c} as string) AS value_as_string, 'enrichment' AS enrichment_tag, '{site_esc}' AS site_id
          FROM {view} WHERE nullif(trim(cast({c} as string)),'') IS NOT NULL""" for c, lab in fields)

    # ARM 3: nnu_nccmds -> visit_detail
    # nnu_nccmds is a DAILY critical-care record (one row per ActivityDate); CriticalCareStartDate/
    # DischargeDate/UnitFunction are constant across an episode's daily rows. Collapse to one
    # visit_detail per episode via SELECT DISTINCT on the episode-level columns. Discharge date is
    # added to the natural key because 29 episodes share (eid,start,func) with distinct discharge dates.
    VD_CONCEPT = 0
    NCCMDS_BODY = f"""
      SELECT {m('enrich-nnu-visitdetail','l.infant_id','nc.eid','nc.cc_start','nc.cc_end','nc.cc_func')} AS visit_detail_id,
             l.infant_id AS person_id, {VD_CONCEPT} AS visit_detail_concept_id,
             nc.cc_start AS visit_detail_start_date,
             nc.cc_end AS visit_detail_end_date,
             l.pregnancy_id, l.infant_id, 'nnu_nccmds' AS source_table,
             'CriticalCareUnitFunction=' || cast(nc.cc_func as string) AS visit_detail_source_value,
             'enrichment' AS enrichment_tag, '{site_esc}' AS site_id
      FROM link l JOIN nc ON nc.eid = l.eid
    """
    NC_CTE = f"""nc AS (
        SELECT DISTINCT EntityID AS eid,
               to_date(CriticalCareStartDate) AS cc_start,
               to_date(CriticalCareDischargeDate) AS cc_end,
               CriticalCareUnitFunction AS cc_func
        FROM {cerner["nnu_nccmds"]} WHERE CriticalCareStartDate IS NOT NULL
      )"""

    # Each INSERT is fully self-contained: WITH link + base CTE(s), then the body.
    def with_(ctes, body): return f"WITH {', '.join(ctes)}\n{body}"

    plan_ = [
      ("measurement", "nnu_episodes",           with_([LINK_CTE, BASE_EP], emit_nnu_meas("base_ep","nnu_episodes",EP_MEAS)), "measurement_id"),
      ("observation", "nnu_episodes",           with_([LINK_CTE, BASE_EP], emit_nnu_obs ("base_ep","nnu_episodes",EP_OBS)),  "observation_id"),
      ("measurement", "nnu_routineexamination", with_([LINK_CTE, BASE_EX], emit_nnu_meas("base_ex","nnu_routineexamination",EX_MEAS)), "measurement_id"),
      ("observation", "nnu_routineexamination", with_([LINK_CTE, BASE_EX], emit_nnu_obs ("base_ex","nnu_routineexamination",EX_OBS)),  "observation_id"),
      ("visit_detail","nnu_nccmds",             with_([LINK_CTE, NC_CTE], NCCMDS_BODY), "visit_detail_id"),
    ]
    for tgt, src, body, idcol in plan_:
        spark.sql(f"DELETE FROM {stg}_enrich.{tgt} WHERE site_id='{site_esc}' AND source_table='{src}'")
        spark.sql(f"INSERT INTO {stg}_enrich.{tgt} {body}")
        n = spark.sql(f"SELECT count(*) c FROM {stg}_enrich.{tgt} WHERE site_id='{site_esc}' AND source_table='{src}'").first()["c"]
        d = spark.sql(f"SELECT count(DISTINCT {idcol}) c FROM {stg}_enrich.{tgt} WHERE site_id='{site_esc}' AND source_table='{src}'").first()["c"]
        assert n == d, f"{tgt}/{src} id COLLISION {n} vs {d}"
        print(f"{tgt} <- {src}: {n} rows collision-free")




# === conform_person — eligibility gate + v5.4 person projection (Task 6 Step 1) ===
# Establish the ELIGIBLE person set FIRST (birth_datetime NOT NULL). Quarantine null-DOB persons
# (idempotent: clear this site+reason before INSERT). Rebuild {schema}.person from eligibles only,
# projecting the FULL v5.4 person column set. gender_concept_id: mother=8532 (Female); else map
# PERSONPHENSEX (verified 4_prod.raw.msd401babydemo: '1' male / '2' female / '9'/'X'/'0' -> UNKNOWN)
# 1->8507 (MALE), 2->8532 (FEMALE), else 8551 (UNKNOWN). ethnicity_concept_id=0 (no MSDS source);
# location/provider/care_site + source_concept_ids NULL/0. Keeps site_id + person_role lineage.
# EVERY downstream merge filters person_id to {stg}._eligible so quarantining a person cascades.

def conform_person(schema, site, stg):
    S = schema
    site_esc = str(site).replace("'", "''")

    # eligible person set (survives conformance) -> staging helper read by all merges
    spark.sql(f"""CREATE OR REPLACE TABLE {stg}._eligible AS
      SELECT person_id FROM {stg}.person
      WHERE site_id='{site_esc}' AND birth_datetime IS NOT NULL""")

    # quarantine null-DOB persons (idempotent: clear this site+reason first)
    spark.sql(f"DELETE FROM {S}._quarantine WHERE site_id='{site_esc}' AND reason='null_birth_year'")
    spark.sql(f"""INSERT INTO {S}._quarantine (site_id, entity, entity_id, reason, detail)
      SELECT '{site_esc}','person',person_id,'null_birth_year',person_source_value
      FROM {stg}.person WHERE site_id='{site_esc}' AND birth_datetime IS NULL""")

    # rebuild published person from eligibles only (full v5.4 column set)
    spark.sql(f"DELETE FROM {S}.person WHERE site_id='{site_esc}'")
    spark.sql(f"""
    INSERT INTO {S}.person (person_id, gender_concept_id, year_of_birth, month_of_birth,
      day_of_birth, birth_datetime, race_concept_id, ethnicity_concept_id, location_id,
      provider_id, care_site_id, person_source_value, gender_source_value,
      gender_source_concept_id, race_source_value, race_source_concept_id,
      ethnicity_source_value, ethnicity_source_concept_id, site_id, person_role)
    SELECT person_id,
      CASE WHEN person_role='mother' THEN 8532
           WHEN gender_source_value='1' THEN 8507
           WHEN gender_source_value='2' THEN 8532
           ELSE 8551 END AS gender_concept_id,
      year(birth_datetime)  AS year_of_birth,
      month(birth_datetime) AS month_of_birth,
      day(birth_datetime)   AS day_of_birth,
      birth_datetime,
      race_concept_id, 0 AS ethnicity_concept_id,
      CAST(NULL AS BIGINT) AS location_id, CAST(NULL AS BIGINT) AS provider_id,
      CAST(NULL AS BIGINT) AS care_site_id,
      person_source_value,
      CASE WHEN person_role='mother' THEN 'mother' ELSE gender_source_value END AS gender_source_value,
      0 AS gender_source_concept_id, race_source_value, 0 AS race_source_concept_id,
      CAST(NULL AS STRING) AS ethnicity_source_value, 0 AS ethnicity_source_concept_id,
      '{site_esc}' AS site_id, person_role
    FROM {stg}.person WHERE site_id='{site_esc}' AND birth_datetime IS NOT NULL
    """)
    tot = spark.sql(f"SELECT count(*) c FROM {stg}.person WHERE site_id='{site_esc}'").first()["c"]
    elig = spark.table(f"{stg}._eligible").count()
    q = spark.sql(f"SELECT count(*) c FROM {S}._quarantine WHERE site_id='{site_esc}' AND reason='null_birth_year'").first()["c"]
    gd = spark.sql(f"""SELECT
      count(CASE WHEN gender_concept_id=8507 THEN 1 END) male,
      count(CASE WHEN gender_concept_id=8532 THEN 1 END) female,
      count(CASE WHEN gender_concept_id=8551 THEN 1 END) unknown
      FROM {S}.person WHERE site_id='{site_esc}'""").first()
    print(f"conform_person: staged {tot} | eligible {elig} | quarantined(null_birth_year) {q} | "
          f"published gender M={gd['male']} F={gd['female']} U={gd['unknown']}")


# === _merge_to_cdm — grain-routed, eligibility-filtered fact merges (Task 6 Steps 2-3) ===
# Re-derives the domain rows from the WIDENED {stg}._all_facts (baby_natural_key/careconid +
# source_table) NOT the mother-attributed {stg}.condition_occurrence, so baby-keyed facts route to
# the infant. GRAIN ROUTING: person_id = infant (via {stg}._baby_id) for msd404/msd405, else the
# pregnancy's mother (via {stg}.pregnancy). Baby-keyed rows whose LPIDBABY does not resolve are
# QUARANTINED (unresolved_infant), NOT mother-attributed. Any routed person not in {stg}._eligible
# is quarantined (orphan_after_quarantine). Facts with an underivable event date are quarantined
# (null_event_date). fact ids are re-minted on the SAME grain as _core_persist so ids stay stable:
#   fact id = mid(<grain>, site, source_table, fact_natural_key); pregnancy_id = mid('pregnancy',
#   site, pregnancy_natural_key); visit_occurrence_id = mid('visit', site, careconid) when present.
# Enrichment measurement/observation rows carry their OWN person_id (nnu->infant, mat->mother),
# routed by that id (still eligibility-filtered); condition/procedure have no enrichment rows.
# TASK 7 extends this function with visit_occurrence / visit_detail / episode / episode_event /
# fact_relationship / pregnancy / infant merges (see markers below).

def _merge_to_cdm(schema, site, stg, include_enrichment):
    S = schema; E = stg + "_enrich"
    site_esc = str(site).replace("'", "''")
    def m(grain, *p):
        return mid(grain, site, *p)

    PREG_ID   = m('pregnancy', 'f.pregnancy_natural_key')
    IS_BABY   = "f.source_table IN ('msd404diagneo','msd405careactbaby')"
    BABY_JOIN = f"LEFT JOIN {stg}._baby_id bl ON bl.LPIDBABY = f.baby_natural_key"
    PREG_JOIN = f"LEFT JOIN {stg}.pregnancy pr ON pr.pregnancy_id = {PREG_ID}"
    ROUTED    = f"CASE WHEN {IS_BABY} THEN bl.infant_person_id ELSE pr.person_id END"
    VISIT_ID  = (f"CASE WHEN f.careconid IS NOT NULL THEN {m('visit','f.careconid')} "
                 "ELSE CAST(NULL AS BIGINT) END")   # TASK 7 Step 3: visit link at merge time
    EVENT_ED  = "coalesce(f.event_date, pr.pregnancy_start_date)"  # null-date fix -> pregnancy date
    QID       = m('quarantine-fact', 'q.source_table', 'q.fact_natural_key')

    # ---- (A) quarantine passes over ALL facts, ONCE (reasons mutually exclusive; per-reason
    #          idempotent clear). Done here (not per-domain) so a reason's DELETE-by-site does not
    #          wipe another domain's rows. Predicates mirror the merge's routing/eligibility exactly.
    BASE_Q = (f"SELECT f.source_table, f.fact_natural_key, f.target_omop_table, "
              f"{ROUTED} AS rp, {EVENT_ED} AS ed, "
              f"bl.infant_person_id AS bpid, ({IS_BABY}) AS is_baby "
              f"FROM {stg}._all_facts f {BABY_JOIN} {PREG_JOIN}")
    ELIG_RP = f"EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id = q.rp)"

    def _quar(reason, where):
        spark.sql(f"DELETE FROM {S}._quarantine WHERE site_id='{site_esc}' AND reason='{reason}'")
        spark.sql(f"""INSERT INTO {S}._quarantine (site_id, entity, entity_id, reason, detail)
          SELECT '{site_esc}' AS site_id, 'fact' AS entity, {QID} AS entity_id,
                 '{reason}' AS reason, concat_ws('|', q.source_table, q.fact_natural_key) AS detail
          FROM ({BASE_Q}) q WHERE {where}""")

    _quar("unsupported_domain", "q.target_omop_table IS NULL")
    _quar("unresolved_infant",
          "q.target_omop_table IS NOT NULL AND q.is_baby AND q.bpid IS NULL")
    _quar("orphan_after_quarantine",
          f"q.target_omop_table IS NOT NULL "
          f"AND NOT(q.is_baby AND q.bpid IS NULL) "
          f"AND (q.rp IS NULL OR NOT {ELIG_RP})")
    # null_event_date: passes person routing + eligibility but has no derivable date (core arm)
    spark.sql(f"DELETE FROM {S}._quarantine WHERE site_id='{site_esc}' AND reason='null_event_date'")
    spark.sql(f"""INSERT INTO {S}._quarantine (site_id, entity, entity_id, reason, detail)
      SELECT '{site_esc}','fact', {QID}, 'null_event_date', concat_ws('|', q.source_table, q.fact_natural_key)
      FROM ({BASE_Q}) q
      WHERE q.target_omop_table IS NOT NULL
        AND NOT(q.is_baby AND q.bpid IS NULL)
        AND q.rp IS NOT NULL AND {ELIG_RP} AND q.ed IS NULL""")

    # ---- (B) core-domain fact merge. Routed CTE exposes rp/vid/ed/pid; final SELECT re-mints the
    #          fact id on <grain> and keeps only rows whose routed person is eligible + has a date.
    def _routed_cte(t):
        return (
          "WITH r AS ( SELECT f.source_table, f.fact_natural_key, f.pregnancy_natural_key, "
          "f.concept_id, f.source_value, f.source_concept_id, f.value_as_number, f.unit_source_value, "
          f"{ROUTED} AS rp, {VISIT_ID} AS vid, {EVENT_ED} AS ed, pr.pregnancy_id AS pid "  # FIX1: joined (nullable) pregnancy_id lineage, not re-minted (baby-keyed facts w/o msd101 preg -> NULL, CDM-valid)
          f"FROM {stg}._all_facts f {BABY_JOIN} {PREG_JOIN} "
          f"WHERE f.target_omop_table = '{t}' ) ")
    KEEP = ("WHERE r.rp IS NOT NULL AND r.ed IS NOT NULL "
            f"AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id = r.rp)")

    # condition_occurrence (no enrichment rows)
    FID = m('condition', 'r.source_table', 'r.fact_natural_key')
    spark.sql(f"DELETE FROM {S}.condition_occurrence WHERE site_id='{site_esc}'")
    spark.sql(f"""{_routed_cte('condition_occurrence')}
    INSERT INTO {S}.condition_occurrence (condition_occurrence_id, person_id, condition_concept_id,
      condition_start_date, condition_start_datetime, condition_end_date, condition_end_datetime,
      condition_type_concept_id, condition_status_concept_id, stop_reason, provider_id,
      visit_occurrence_id, visit_detail_id, condition_source_value, condition_source_concept_id,
      condition_status_source_value, site_id, pregnancy_id, source_table)
    SELECT {FID} AS condition_occurrence_id, r.rp, r.concept_id AS condition_concept_id,
      r.ed AS condition_start_date, CAST(r.ed AS TIMESTAMP) AS condition_start_datetime,
      CAST(NULL AS DATE), CAST(NULL AS TIMESTAMP), 32817 AS condition_type_concept_id,
      CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS BIGINT),
      r.vid AS visit_occurrence_id, CAST(NULL AS BIGINT), r.source_value AS condition_source_value,
      r.source_concept_id AS condition_source_concept_id, CAST(NULL AS STRING),
      '{site_esc}' AS site_id, r.pid AS pregnancy_id, r.source_table
    FROM r {KEEP}""")
    print("condition_occurrence", spark.sql(f"SELECT count(*) c FROM {S}.condition_occurrence WHERE site_id='{site_esc}'").first()["c"])

    # procedure_occurrence (no enrichment rows)
    FID = m('procedure', 'r.source_table', 'r.fact_natural_key')
    spark.sql(f"DELETE FROM {S}.procedure_occurrence WHERE site_id='{site_esc}'")
    spark.sql(f"""{_routed_cte('procedure_occurrence')}
    INSERT INTO {S}.procedure_occurrence (procedure_occurrence_id, person_id, procedure_concept_id,
      procedure_date, procedure_datetime, procedure_end_date, procedure_end_datetime,
      procedure_type_concept_id, modifier_concept_id, quantity, provider_id, visit_occurrence_id,
      visit_detail_id, procedure_source_value, procedure_source_concept_id, modifier_source_value,
      site_id, pregnancy_id, source_table)
    SELECT {FID} AS procedure_occurrence_id, r.rp, r.concept_id AS procedure_concept_id,
      r.ed AS procedure_date, CAST(r.ed AS TIMESTAMP) AS procedure_datetime,
      CAST(NULL AS DATE), CAST(NULL AS TIMESTAMP), 32817 AS procedure_type_concept_id,
      CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      r.vid AS visit_occurrence_id, CAST(NULL AS BIGINT), r.source_value AS procedure_source_value,
      r.source_concept_id AS procedure_source_concept_id, CAST(NULL AS STRING),
      '{site_esc}' AS site_id, r.pid AS pregnancy_id, r.source_table
    FROM r {KEEP}""")
    print("procedure_occurrence", spark.sql(f"SELECT count(*) c FROM {S}.procedure_occurrence WHERE site_id='{site_esc}'").first()["c"])

    # measurement (core + enrichment)
    FID = m('measurement', 'r.source_table', 'r.fact_natural_key')
    spark.sql(f"DELETE FROM {S}.measurement WHERE site_id='{site_esc}'")
    spark.sql(f"""{_routed_cte('measurement')}
    INSERT INTO {S}.measurement (measurement_id, person_id, measurement_concept_id, measurement_date,
      measurement_datetime, measurement_time, measurement_type_concept_id, operator_concept_id,
      value_as_number, value_as_concept_id, unit_concept_id, range_low, range_high, provider_id,
      visit_occurrence_id, visit_detail_id, measurement_source_value, measurement_source_concept_id,
      unit_source_value, unit_source_concept_id, value_source_value, measurement_event_id,
      meas_event_field_concept_id, site_id, pregnancy_id, enrichment_tag, source_table)
    SELECT {FID} AS measurement_id, r.rp, r.concept_id AS measurement_concept_id, r.ed AS measurement_date,
      CAST(r.ed AS TIMESTAMP) AS measurement_datetime, CAST(NULL AS STRING), 32817 AS measurement_type_concept_id,
      CAST(NULL AS BIGINT), r.value_as_number, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      CAST(NULL AS DOUBLE), CAST(NULL AS DOUBLE), CAST(NULL AS BIGINT),
      r.vid AS visit_occurrence_id, CAST(NULL AS BIGINT), r.source_value AS measurement_source_value,
      r.source_concept_id AS measurement_source_concept_id, r.unit_source_value, CAST(NULL AS BIGINT),
      CAST(NULL AS STRING), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      '{site_esc}' AS site_id, r.pid AS pregnancy_id, CAST(NULL AS STRING) AS enrichment_tag, r.source_table
    FROM r {KEEP}""")
    if include_enrichment:
        spark.sql(f"""
        INSERT INTO {S}.measurement (measurement_id, person_id, measurement_concept_id, measurement_date,
          measurement_datetime, measurement_time, measurement_type_concept_id, operator_concept_id,
          value_as_number, value_as_concept_id, unit_concept_id, range_low, range_high, provider_id,
          visit_occurrence_id, visit_detail_id, measurement_source_value, measurement_source_concept_id,
          unit_source_value, unit_source_concept_id, value_source_value, measurement_event_id,
          meas_event_field_concept_id, site_id, pregnancy_id, enrichment_tag, source_table)
        SELECT e.measurement_id, e.person_id, e.measurement_concept_id,
          coalesce(e.measurement_date, pr.pregnancy_start_date) AS measurement_date,
          CAST(coalesce(e.measurement_date, pr.pregnancy_start_date) AS TIMESTAMP), CAST(NULL AS STRING),
          32817 AS measurement_type_concept_id, CAST(NULL AS BIGINT), e.value_as_number,
          CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS DOUBLE), CAST(NULL AS DOUBLE),
          CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
          e.measurement_source_value, e.measurement_source_concept_id, e.unit_source_value,
          CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
          '{site_esc}' AS site_id, e.pregnancy_id, e.enrichment_tag, e.source_table
        FROM {E}.measurement e
        LEFT JOIN {stg}.pregnancy pr ON pr.pregnancy_id = e.pregnancy_id
        WHERE e.site_id='{site_esc}'
          AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id = e.person_id)
          AND coalesce(e.measurement_date, pr.pregnancy_start_date) IS NOT NULL""")
        # enrichment measurement quarantine (orphan person / undatable) — appended (no per-reason clear)
        spark.sql(f"""INSERT INTO {S}._quarantine (site_id, entity, entity_id, reason, detail)
          SELECT '{site_esc}','fact', e.measurement_id,
            CASE WHEN NOT EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id=e.person_id)
                 THEN 'orphan_after_quarantine' ELSE 'null_event_date' END,
            concat_ws('|', e.source_table, cast(e.measurement_id as string))
          FROM {E}.measurement e
          LEFT JOIN {stg}.pregnancy pr ON pr.pregnancy_id = e.pregnancy_id
          WHERE e.site_id='{site_esc}'
            AND ( NOT EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id=e.person_id)
                  OR coalesce(e.measurement_date, pr.pregnancy_start_date) IS NULL )""")
    print("measurement", spark.sql(f"SELECT count(*) c FROM {S}.measurement WHERE site_id='{site_esc}'").first()["c"])

    # observation (core + enrichment)
    FID = m('observation', 'r.source_table', 'r.fact_natural_key')
    spark.sql(f"DELETE FROM {S}.observation WHERE site_id='{site_esc}'")
    spark.sql(f"""{_routed_cte('observation')}
    INSERT INTO {S}.observation (observation_id, person_id, observation_concept_id, observation_date,
      observation_datetime, observation_type_concept_id, value_as_number, value_as_string,
      value_as_concept_id, qualifier_concept_id, unit_concept_id, provider_id, visit_occurrence_id,
      visit_detail_id, observation_source_value, observation_source_concept_id, unit_source_value,
      qualifier_source_value, value_source_value, observation_event_id, obs_event_field_concept_id,
      site_id, pregnancy_id, enrichment_tag, source_table)
    SELECT {FID} AS observation_id, r.rp, r.concept_id AS observation_concept_id, r.ed AS observation_date,
      CAST(r.ed AS TIMESTAMP) AS observation_datetime, 32817 AS observation_type_concept_id,
      r.value_as_number, CAST(NULL AS STRING) AS value_as_string, CAST(NULL AS BIGINT),
      CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      r.vid AS visit_occurrence_id, CAST(NULL AS BIGINT), r.source_value AS observation_source_value,
      r.source_concept_id AS observation_source_concept_id, r.unit_source_value, CAST(NULL AS STRING),
      CAST(NULL AS STRING), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      '{site_esc}' AS site_id, r.pid AS pregnancy_id, CAST(NULL AS STRING) AS enrichment_tag, r.source_table
    FROM r {KEEP}""")
    if include_enrichment:
        spark.sql(f"""
        INSERT INTO {S}.observation (observation_id, person_id, observation_concept_id, observation_date,
          observation_datetime, observation_type_concept_id, value_as_number, value_as_string,
          value_as_concept_id, qualifier_concept_id, unit_concept_id, provider_id, visit_occurrence_id,
          visit_detail_id, observation_source_value, observation_source_concept_id, unit_source_value,
          qualifier_source_value, value_source_value, observation_event_id, obs_event_field_concept_id,
          site_id, pregnancy_id, enrichment_tag, source_table)
        SELECT e.observation_id, e.person_id, e.observation_concept_id,
          coalesce(e.observation_date, pr.pregnancy_start_date) AS observation_date,
          CAST(coalesce(e.observation_date, pr.pregnancy_start_date) AS TIMESTAMP),
          32817 AS observation_type_concept_id, e.value_as_number, e.value_as_string,
          CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
          CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), e.observation_source_value,
          e.observation_source_concept_id, CAST(NULL AS STRING), CAST(NULL AS STRING),
          CAST(NULL AS STRING), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
          '{site_esc}' AS site_id, e.pregnancy_id, e.enrichment_tag, e.source_table
        FROM {E}.observation e
        LEFT JOIN {stg}.pregnancy pr ON pr.pregnancy_id = e.pregnancy_id
        WHERE e.site_id='{site_esc}'
          AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id = e.person_id)
          AND coalesce(e.observation_date, pr.pregnancy_start_date) IS NOT NULL""")
        spark.sql(f"""INSERT INTO {S}._quarantine (site_id, entity, entity_id, reason, detail)
          SELECT '{site_esc}','fact', e.observation_id,
            CASE WHEN NOT EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id=e.person_id)
                 THEN 'orphan_after_quarantine' ELSE 'null_event_date' END,
            concat_ws('|', e.source_table, cast(e.observation_id as string))
          FROM {E}.observation e
          LEFT JOIN {stg}.pregnancy pr ON pr.pregnancy_id = e.pregnancy_id
          WHERE e.site_id='{site_esc}'
            AND ( NOT EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id=e.person_id)
                  OR coalesce(e.observation_date, pr.pregnancy_start_date) IS NULL )""")
    print("observation", spark.sql(f"SELECT count(*) c FROM {S}.observation WHERE site_id='{site_esc}'").first()["c"])

    # FIX3: record deduped-away CARECONIDs (from _core_facts Step 0) as dup_careconid quarantine.
    if spark.catalog.tableExists(f"{stg}._dropped_carecontact"):
        spark.sql(f"DELETE FROM {S}._quarantine WHERE site_id='{site_esc}' AND reason='dup_careconid'")
        spark.sql(f"""INSERT INTO {S}._quarantine (site_id, entity, entity_id, reason, detail)
          SELECT '{site_esc}', 'care_contact', {m('visit','d.CARECONID')} AS entity_id,
                 'dup_careconid', d.CARECONID
          FROM {stg}._dropped_carecontact d""")

    qc = spark.sql(f"""SELECT reason, count(*) c FROM {S}._quarantine
      WHERE site_id='{site_esc}' GROUP BY reason ORDER BY reason""").collect()
    print("quarantine by reason:", [(r['reason'], r['c']) for r in qc])

    # ---- TASK 7: visit_occurrence (from {stg}._carecontact), nnu parent visit_occurrence +
    #      visit_detail linkage, episode/episode_event/fact_relationship merge (from {stg}_l3.*),
    #      and pregnancy/infant extension merge (explicit projection, eligibility-filtered).
    build_visits(schema, site, stg)
    merge_visit_detail(schema, site, stg, include_enrichment)
    merge_l3(schema, site, stg)          # episode, episode_event, fact_relationship (+ L3 ethnicity obs date fix)
    merge_extensions(schema, site, stg)  # pregnancy, infant (explicit projection, limited-mode-safe)


# === build_visits — visit_occurrence from the canonical {stg}._carecontact (Task 7 Step 1) ===
# One visit per canonical CARECONID (already deduped in _core_facts Step 0, so no re-dedup here —
# finding #7). id = mid('visit', site, CARECONID) — the SAME grain/key the Task 6 fact merge uses for
# visit_occurrence_id (VISIT_ID = m('visit','f.careconid')), so msd202 facts link 1:1 to a real visit.
# person_id = the pregnancy's mother (join minted pregnancy_id -> {stg}.pregnancy), eligibility-filtered
# and NOT NULL (CDM requires visit person_id). visit_concept_id 9202 (Outpatient Visit — verified live
# in 4_prod.omop.concept: domain Visit, standard S). type 32817 (EHR). start = CCONTACTDATETIME::date,
# end = start (care-contact is same-day). DELETE-by-site + INSERT.

def build_visits(schema, site, stg):
    S = schema
    site_esc = str(site).replace("'", "''")
    def m(grain, *p):
        return mid(grain, site, *p)
    VID  = m('visit', 'cc.CARECONID')
    PREG = m('pregnancy', 'cc.PREGNANCYID')
    spark.sql(f"DELETE FROM {S}.visit_occurrence WHERE site_id='{site_esc}'")
    spark.sql(f"""
    INSERT INTO {S}.visit_occurrence (visit_occurrence_id, person_id, visit_concept_id,
      visit_start_date, visit_start_datetime, visit_end_date, visit_end_datetime,
      visit_type_concept_id, provider_id, care_site_id, visit_source_value,
      visit_source_concept_id, admitted_from_concept_id, admitted_from_source_value,
      discharged_to_concept_id, discharged_to_source_value, preceding_visit_occurrence_id,
      site_id, pregnancy_id)
    SELECT {VID} AS visit_occurrence_id, pr.person_id, 9202 AS visit_concept_id,
      CAST(cc.CCONTACTDATETIME AS DATE) AS visit_start_date,
      CAST(cc.CCONTACTDATETIME AS TIMESTAMP) AS visit_start_datetime,
      CAST(cc.CCONTACTDATETIME AS DATE) AS visit_end_date,
      CAST(cc.CCONTACTDATETIME AS TIMESTAMP) AS visit_end_datetime,
      32817 AS visit_type_concept_id, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      cc.CARECONID AS visit_source_value, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      CAST(NULL AS STRING), CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS BIGINT),
      '{site_esc}' AS site_id, {PREG} AS pregnancy_id
    FROM {stg}._carecontact cc
    JOIN {stg}.pregnancy pr ON pr.pregnancy_id = {PREG}
    WHERE EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id = pr.person_id)
    """)
    n = spark.sql(f"SELECT count(*) c FROM {S}.visit_occurrence WHERE site_id='{site_esc}'").first()["c"]
    d = spark.sql(f"SELECT count(DISTINCT visit_occurrence_id) c FROM {S}.visit_occurrence WHERE site_id='{site_esc}'").first()["c"]
    assert n == d, f"visit_occurrence_id COLLISION {n} vs {d}"
    print("visit_occurrence", n, "collision-free (Outpatient 9202)")


# === merge_visit_detail — nnu parent visit_occurrence + visit_detail (Task 7 Step 2) ===
# Enrichment-only (nnu). For each {stg}_enrich.visit_detail (one per nnu critical-care episode), mint a
# PARENT visit_occurrence keyed deterministically on grain 'nnu-visit' from the visit_detail_id. NOTE:
# the visit_detail_id is itself mid('enrich-nnu-visitdetail', site, infant_id, eid, cc_start, cc_end,
# cc_func) (sp3_50), but eid is NOT carried on the enrich.visit_detail row — so we mint the parent from
# the visit_detail_id, which is a lossless 1:1 deterministic proxy for that same 5-part key. Parent
# visit_concept_id 9201 (Inpatient Visit — verified live: domain Visit, standard S) / type 32817;
# start/end = the nnu episode dates (= visit_detail start/end); person_id = the infant. APPENDED into
# visit_occurrence (build_visits already did the per-site DELETE). Then visit_detail merged with
# visit_detail_concept_id 0 (proprietary), type 32817, non-null visit_occurrence_id = minted parent.
# Eligibility-filter person_id (infant).

def merge_visit_detail(schema, site, stg, include_enrichment):
    S = schema; E = stg + "_enrich"
    site_esc = str(site).replace("'", "''")
    def m(grain, *p):
        return mid(grain, site, *p)
    spark.sql(f"DELETE FROM {S}.visit_detail WHERE site_id='{site_esc}'")
    if not include_enrichment:
        print("visit_detail 0 (no enrichment)")
        return
    PVID = m('nnu-visit', 'vd.visit_detail_id')
    # parent visit_occurrence (append; per-site delete already done by build_visits)
    spark.sql(f"""
    INSERT INTO {S}.visit_occurrence (visit_occurrence_id, person_id, visit_concept_id,
      visit_start_date, visit_start_datetime, visit_end_date, visit_end_datetime,
      visit_type_concept_id, provider_id, care_site_id, visit_source_value,
      visit_source_concept_id, admitted_from_concept_id, admitted_from_source_value,
      discharged_to_concept_id, discharged_to_source_value, preceding_visit_occurrence_id,
      site_id, pregnancy_id)
    SELECT {PVID} AS visit_occurrence_id, vd.person_id, 9201 AS visit_concept_id,
      vd.visit_detail_start_date AS visit_start_date,
      CAST(vd.visit_detail_start_date AS TIMESTAMP) AS visit_start_datetime,
      coalesce(vd.visit_detail_end_date, vd.visit_detail_start_date) AS visit_end_date,  -- FIX2: 54 open nnu episodes have NULL end -> same-anchor fallback
      CAST(coalesce(vd.visit_detail_end_date, vd.visit_detail_start_date) AS TIMESTAMP) AS visit_end_datetime,
      32817 AS visit_type_concept_id, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      vd.visit_detail_source_value AS visit_source_value, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      CAST(NULL AS STRING), CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS BIGINT),
      '{site_esc}' AS site_id, vd.pregnancy_id
    FROM {E}.visit_detail vd
    WHERE vd.site_id='{site_esc}'
      AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id = vd.person_id)
    """)
    # visit_detail merged with non-null parent visit_occurrence_id
    spark.sql(f"""
    INSERT INTO {S}.visit_detail (visit_detail_id, person_id, visit_detail_concept_id,
      visit_detail_start_date, visit_detail_start_datetime, visit_detail_end_date,
      visit_detail_end_datetime, visit_detail_type_concept_id, provider_id, care_site_id,
      visit_detail_source_value, visit_detail_source_concept_id, admitted_from_concept_id,
      admitted_from_source_value, discharged_to_source_value, discharged_to_concept_id,
      preceding_visit_detail_id, parent_visit_detail_id, visit_occurrence_id,
      site_id, pregnancy_id, enrichment_tag, source_table)
    SELECT vd.visit_detail_id, vd.person_id, 0 AS visit_detail_concept_id,
      vd.visit_detail_start_date, CAST(vd.visit_detail_start_date AS TIMESTAMP),
      coalesce(vd.visit_detail_end_date, vd.visit_detail_start_date), CAST(coalesce(vd.visit_detail_end_date, vd.visit_detail_start_date) AS TIMESTAMP),  -- FIX2: same-anchor fallback
      32817 AS visit_detail_type_concept_id, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      vd.visit_detail_source_value, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS BIGINT),
      CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), {PVID} AS visit_occurrence_id,
      '{site_esc}' AS site_id, vd.pregnancy_id, vd.enrichment_tag, vd.source_table
    FROM {E}.visit_detail vd
    WHERE vd.site_id='{site_esc}'
      AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id = vd.person_id)
    """)
    n = spark.sql(f"SELECT count(*) c FROM {S}.visit_detail WHERE site_id='{site_esc}'").first()["c"]
    d = spark.sql(f"SELECT count(DISTINCT visit_detail_id) c FROM {S}.visit_detail WHERE site_id='{site_esc}'").first()["c"]
    assert n == d, f"visit_detail_id COLLISION {n} vs {d}"
    orphan = spark.sql(f"""SELECT count(*) c FROM {S}.visit_detail vd WHERE vd.site_id='{site_esc}'
      AND (vd.visit_occurrence_id IS NULL
        OR NOT EXISTS (SELECT 1 FROM {S}.visit_occurrence vo
             WHERE vo.visit_occurrence_id = vd.visit_occurrence_id))""").first()["c"]
    assert orphan == 0, f"{orphan} visit_detail rows with unresolved visit_occurrence_id"
    print("visit_detail", n, "collision-free | parent-visit resolves for all (Inpatient 9201)")


# === merge_l3 — pregnancy condition + final episode/event graph + relationships + ethnicity ===
# Runs after core and enrichment facts/visits are merged. It derives a non-null pregnancy anchor,
# publishes the L3 pregnancy-as-condition arm, rebuilds complete episode_event links from the final
# tables, copies eligible person relationships, and appends the dated NHS ethnicity observation.

def merge_l3(schema, site, stg):
    S = schema; S3 = stg + "_l3"
    site_esc = str(site).replace("'", "''")

    # Build one clinically defensible anchor per eligible pregnancy after core + enrichment facts
    # and visits have been merged. Prefer recorded LMP; then derive LMP from delivery+gestation;
    # then use the earliest dated event; finally use delivery date. Truly undated pregnancies
    # remain in the extension table but do not create invalid standard CDM rows.
    spark.sql(f"""
    CREATE OR REPLACE TABLE {stg}._pregnancy_anchor AS
    WITH ev AS (
      SELECT pregnancy_id, condition_start_date AS d
      FROM {S}.condition_occurrence WHERE site_id='{site_esc}'
      UNION ALL SELECT pregnancy_id, procedure_date
      FROM {S}.procedure_occurrence WHERE site_id='{site_esc}'
      UNION ALL SELECT pregnancy_id, measurement_date
      FROM {S}.measurement WHERE site_id='{site_esc}'
      UNION ALL SELECT pregnancy_id, observation_date
      FROM {S}.observation WHERE site_id='{site_esc}'
      UNION ALL SELECT pregnancy_id, visit_start_date
      FROM {S}.visit_occurrence WHERE site_id='{site_esc}'
      UNION ALL SELECT pregnancy_id, visit_end_date
      FROM {S}.visit_occurrence WHERE site_id='{site_esc}'
      UNION ALL SELECT pregnancy_id, visit_detail_start_date
      FROM {S}.visit_detail WHERE site_id='{site_esc}'
      UNION ALL SELECT pregnancy_id, visit_detail_end_date
      FROM {S}.visit_detail WHERE site_id='{site_esc}'
    ),
    agg AS (
      SELECT pregnancy_id, min(d) AS earliest_event
      FROM ev WHERE pregnancy_id IS NOT NULL AND d IS NOT NULL GROUP BY pregnancy_id
    ),
    anchored AS (
      SELECT pr.pregnancy_id, pr.person_id, pr.pregnancy_end_date,
        coalesce(
          pr.pregnancy_start_date,
          CASE WHEN pr.pregnancy_end_date IS NOT NULL
                    AND pr.gestational_length_in_day BETWEEN 1 AND 400
               THEN date_sub(pr.pregnancy_end_date, pr.gestational_length_in_day) END,
          a.earliest_event,
          pr.pregnancy_end_date
        ) AS episode_start_date
      FROM {stg}.pregnancy pr
      LEFT JOIN agg a ON a.pregnancy_id=pr.pregnancy_id
      WHERE pr.site_id='{site_esc}'
        AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id=pr.person_id)
    )
    SELECT pregnancy_id, person_id, episode_start_date,
      CASE WHEN pregnancy_end_date >= episode_start_date THEN pregnancy_end_date END AS episode_end_date
    FROM anchored
    """)

    spark.sql(f"""DELETE FROM {S}._quarantine
      WHERE site_id='{site_esc}' AND reason='undated_pregnancy_episode'""")
    spark.sql(f"""
    INSERT INTO {S}._quarantine (site_id, entity, entity_id, reason, detail)
    SELECT '{site_esc}', 'pregnancy', pregnancy_id, 'undated_pregnancy_episode',
           'no pregnancy start/end or dated clinical event'
    FROM {stg}._pregnancy_anchor WHERE episode_start_date IS NULL
    """)

    # Publish the staged L3 pregnancy-as-condition arm. This used to be built and discarded.
    spark.sql(f"""DELETE FROM {S}.condition_occurrence
      WHERE site_id='{site_esc}' AND source_table='l3_pregnancy'""")
    spark.sql(f"""
    INSERT INTO {S}.condition_occurrence (condition_occurrence_id, person_id, condition_concept_id,
      condition_start_date, condition_start_datetime, condition_end_date, condition_end_datetime,
      condition_type_concept_id, condition_status_concept_id, stop_reason, provider_id,
      visit_occurrence_id, visit_detail_id, condition_source_value, condition_source_concept_id,
      condition_status_source_value, site_id, pregnancy_id, source_table)
    SELECT c.condition_occurrence_id, c.person_id, c.condition_concept_id,
      a.episode_start_date, CAST(a.episode_start_date AS TIMESTAMP),
      a.episode_end_date, CAST(a.episode_end_date AS TIMESTAMP),
      c.condition_type_concept_id, CAST(NULL AS BIGINT), CAST(NULL AS STRING),
      CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      c.condition_source_value, 0 AS condition_source_concept_id, CAST(NULL AS STRING),
      '{site_esc}', c.pregnancy_id, 'l3_pregnancy'
    FROM {S3}.condition_occurrence c
    JOIN {stg}._pregnancy_anchor a ON a.pregnancy_id=c.pregnancy_id
    WHERE c.site_id='{site_esc}' AND a.episode_start_date IS NOT NULL
    """)
    l3c = spark.sql(f"""SELECT count(*) c FROM {S}.condition_occurrence
      WHERE site_id='{site_esc}' AND source_table='l3_pregnancy'""").first()["c"]

    # Final episode dates use the same anchor as the pregnancy-condition arm.
    spark.sql(f"DELETE FROM {S}.episode WHERE site_id='{site_esc}'")
    spark.sql(f"""
    INSERT INTO {S}.episode (episode_id, person_id, episode_concept_id, episode_start_date,
      episode_start_datetime, episode_end_date, episode_end_datetime, episode_parent_id,
      episode_number, episode_object_concept_id, episode_type_concept_id, episode_source_value,
      episode_source_concept_id, pregnancy_id, site_id)
    SELECT e.episode_id, e.person_id, e.episode_concept_id, a.episode_start_date,
      CAST(a.episode_start_date AS TIMESTAMP), a.episode_end_date,
      CAST(a.episode_end_date AS TIMESTAMP), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      e.episode_object_concept_id, 32817 AS episode_type_concept_id,
      'MSDS pregnancy' AS episode_source_value, 0 AS episode_source_concept_id,
      e.pregnancy_id, '{site_esc}'
    FROM {S3}.episode e
    JOIN {stg}._pregnancy_anchor a ON a.pregnancy_id=e.pregnancy_id
    WHERE e.site_id='{site_esc}' AND a.episode_start_date IS NOT NULL
    """)
    en = spark.sql(f"SELECT count(*) c FROM {S}.episode WHERE site_id='{site_esc}'").first()["c"]
    ed = spark.sql(f"SELECT count(DISTINCT episode_id) c FROM {S}.episode WHERE site_id='{site_esc}'").first()["c"]
    assert en == ed, f"episode_id COLLISION {en} vs {ed}"

    # Rebuild episode_event from the FINAL merged tables so core, L3 and enrichment rows
    # receive complete pregnancy-episode links.
    FC_COND, FC_MEAS, FC_OBS = 1147127, 1147138, 1147165
    FC_PROC, FC_VIS, FC_VD = 1147082, 1147070, 1147624
    spark.sql(f"DELETE FROM {S}.episode_event WHERE site_id='{site_esc}'")

    def link(src_tbl, id_col, field_concept):
        spark.sql(f"""
        INSERT INTO {S}.episode_event (episode_id, event_id, episode_event_field_concept_id,
          source_table, site_id)
        SELECT e.episode_id, f.{id_col}, {field_concept}, '{src_tbl}', '{site_esc}'
        FROM {S}.{src_tbl} f
        JOIN {S}.episode e
          ON e.pregnancy_id=f.pregnancy_id AND e.site_id=f.site_id
        WHERE f.site_id='{site_esc}' AND f.pregnancy_id IS NOT NULL
        """)

    link("condition_occurrence", "condition_occurrence_id", FC_COND)
    link("procedure_occurrence", "procedure_occurrence_id", FC_PROC)
    link("measurement", "measurement_id", FC_MEAS)
    link("observation", "observation_id", FC_OBS)
    link("visit_occurrence", "visit_occurrence_id", FC_VIS)
    link("visit_detail", "visit_detail_id", FC_VD)
    een = spark.sql(f"SELECT count(*) c FROM {S}.episode_event WHERE site_id='{site_esc}'").first()["c"]

    # fact_relationship (person<->person; BOTH endpoints must be eligible)
    spark.sql(f"DELETE FROM {S}.fact_relationship WHERE site_id='{site_esc}'")
    spark.sql(f"""
    INSERT INTO {S}.fact_relationship (domain_concept_id_1, fact_id_1, domain_concept_id_2,
      fact_id_2, relationship_concept_id, site_id)
    SELECT fr.domain_concept_id_1, fr.fact_id_1, fr.domain_concept_id_2, fr.fact_id_2,
      fr.relationship_concept_id, '{site_esc}'
    FROM {S3}.fact_relationship fr
    WHERE fr.site_id='{site_esc}'
      AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id=fr.fact_id_1)
      AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id=fr.fact_id_2)
    """)
    frn = spark.sql(f"SELECT count(*) c FROM {S}.fact_relationship WHERE site_id='{site_esc}'").first()["c"]
    print("L3 pregnancy-condition", l3c, "| episode", en,
          "| episode_event", een, "| fact_relationship", frn)

    # L3 ethnicity observation -> observation (APPEND; NON-NULL date; distinct l3 id)
    spark.sql(f"""
    INSERT INTO {S}.observation (observation_id, person_id, observation_concept_id, observation_date,
      observation_datetime, observation_type_concept_id, value_as_number, value_as_string,
      value_as_concept_id, qualifier_concept_id, unit_concept_id, provider_id, visit_occurrence_id,
      visit_detail_id, observation_source_value, observation_source_concept_id, unit_source_value,
      qualifier_source_value, value_source_value, observation_event_id, obs_event_field_concept_id,
      site_id, pregnancy_id, enrichment_tag, source_table)
    WITH pf AS (
      SELECT person_id, min(pregnancy_start_date) AS ps, min(pregnancy_end_date) AS pe
      FROM {stg}.pregnancy WHERE site_id='{site_esc}' GROUP BY person_id
    ),
    clin AS (
      SELECT person_id, min(d) AS earliest_clin FROM (
        SELECT person_id, condition_start_date AS d FROM {S}.condition_occurrence WHERE site_id='{site_esc}'
        UNION ALL SELECT person_id, measurement_date FROM {S}.measurement WHERE site_id='{site_esc}'
        UNION ALL SELECT person_id, observation_date FROM {S}.observation
          WHERE site_id='{site_esc}' AND (source_table <> 'l3_ethnicity' OR source_table IS NULL)
        UNION ALL SELECT person_id, procedure_date FROM {S}.procedure_occurrence WHERE site_id='{site_esc}'
        UNION ALL SELECT person_id, visit_start_date FROM {S}.visit_occurrence WHERE site_id='{site_esc}'
      ) WHERE d IS NOT NULL GROUP BY person_id
    )
    SELECT o.observation_id, o.person_id, o.observation_concept_id,
      coalesce(cl.earliest_clin, pf.ps, pf.pe, CAST(p.birth_datetime AS DATE)),
      CAST(coalesce(cl.earliest_clin, pf.ps, pf.pe, CAST(p.birth_datetime AS DATE)) AS TIMESTAMP),
      32817, CAST(NULL AS DOUBLE), CAST(NULL AS STRING),
      o.value_as_concept_id, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), o.observation_source_value,
      CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
      CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), '{site_esc}', CAST(NULL AS BIGINT),
      CAST(NULL AS STRING), 'l3_ethnicity'
    FROM {S3}.observation o
    JOIN {S}.person p ON p.person_id=o.person_id AND p.site_id='{site_esc}'
    LEFT JOIN pf ON pf.person_id=o.person_id
    LEFT JOIN clin cl ON cl.person_id=o.person_id
    WHERE o.site_id='{site_esc}'
      AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id=o.person_id)
    """)
    ln = spark.sql(f"""SELECT count(*) c FROM {S}.observation
      WHERE site_id='{site_esc}' AND source_table='l3_ethnicity'""").first()["c"]
    nnd = spark.sql(f"""SELECT count(*) c FROM {S}.observation
      WHERE site_id='{site_esc}' AND source_table='l3_ethnicity' AND observation_date IS NULL""").first()["c"]
    assert nnd == 0, f"{nnd} L3 ethnicity observations with NULL date"
    print("L3 ethnicity observation appended", ln, "| null-date", nnd)


# === merge_extensions — pregnancy / infant extension merge (Task 7 Step 5, finding #9) ===
# DELETE-by-site + INSERT {stg}.pregnancy/.infant -> {schema}.pregnancy/.infant, eligibility-filtered.
# EXPLICIT column projection (NOT SELECT *): in LIMITED mode ensure_enrich_schema never ran, so staging
# lacks the enrich fill/*_source columns that {schema} has (pre_pregnancy_bmi_source /
# pregnancy_marital_status_source_value on pregnancy; birth_weight_source / birth_apgar_source on
# infant). Each {schema} column is projected by name, selecting the staging column where present else
# CAST(NULL AS <type>) — so the same merge works with or without enrichment. pregnancy filters its
# person_id (mother); infant filters infant_id (== person_id) to {stg}._eligible.

def merge_extensions(schema, site, stg):
    S = schema
    site_esc = str(site).replace("'", "''")

    def _proj(tbl, cols, elig_col):
        # cols: list of (name, sqltype). Select the staging column if it exists, else CAST(NULL AS type).
        have = {f.name.lower() for f in spark.table(f"{stg}.{tbl}").schema.fields}
        sel = []
        for name, typ in cols:
            if name == 'site_id':
                sel.append(f"'{site_esc}' AS site_id")
            elif name.lower() in have:
                sel.append(f"s.{name}")
            else:
                sel.append(f"CAST(NULL AS {typ}) AS {name}")
        collist = ", ".join(n for n, _ in cols)
        spark.sql(f"DELETE FROM {S}.{tbl} WHERE site_id='{site_esc}'")
        spark.sql(f"""INSERT INTO {S}.{tbl} ({collist})
          SELECT {", ".join(sel)}
          FROM {stg}.{tbl} s
          WHERE s.site_id='{site_esc}'
            AND EXISTS (SELECT 1 FROM {stg}._eligible el WHERE el.person_id = s.{elig_col})""")
        n = spark.sql(f"SELECT count(*) c FROM {S}.{tbl} WHERE site_id='{site_esc}'").first()["c"]
        print(tbl, n, "(explicit projection, eligibility-filtered)")

    PREG_COLS = [
      ("person_id","BIGINT"), ("pregnancy_id","BIGINT"), ("pregnancy_start_date","DATE"),
      ("pregnancy_end_date","DATE"), ("gestational_length_in_day","INT"),
      ("pregnancy_outcome_concept_id","BIGINT"), ("pregnancy_mode_delivery_concept_id","BIGINT"),
      ("pregnancy_single_concept_id","BIGINT"), ("pregnancy_marital_status_concept_id","BIGINT"),
      ("pregnancy_number_fetuses","INT"), ("pregnancy_number_liveborn","INT"),
      ("prev_pregnancy_parity_concept_id","BIGINT"), ("prev_pregnancy_gravidity","INT"),
      ("prev_livebirth_number","INT"), ("prev_still_misc_number","INT"),
      ("pre_pregnancy_bmi","DOUBLE"), ("pregnancy_folic_concept_id","BIGINT"),
      ("pregnancy_outcome_source_value","STRING"), ("pregnancy_mode_delivery_source_value","STRING"),
      ("site_id","STRING"), ("pre_pregnancy_bmi_source","STRING"),
      ("pregnancy_marital_status_source_value","STRING"),
    ]
    INFANT_COLS = [
      ("pregnancy_id","BIGINT"), ("infant_id","BIGINT"), ("birth_outcome_concept_id","BIGINT"),
      ("birth_weight","DOUBLE"), ("birth_con_malformation_concept_id","BIGINT"),
      ("birth_apgar","INT"), ("labour_delivery_id","STRING"), ("site_id","STRING"),
      ("birth_weight_source","STRING"), ("birth_apgar_source","STRING"),
    ]
    _proj("pregnancy", PREG_COLS, "person_id")
    _proj("infant", INFANT_COLS, "infant_id")


# === build_observation_period — one row per published person (Task 6 Step 4) ===
# start = min / end = max event date across the person's published {schema} facts (condition /
# measurement / observation / procedure). Coalesce to the person's pregnancy dates, then to
# birth_datetime, when no dated fact exists (birth_datetime is guaranteed non-null: eligibles only).
# observation_period_id minted on grain 'obs-period' from person_id; period_type_concept_id 32817.

def build_observation_period(schema, site):
    S = schema
    site_esc = str(site).replace("'", "''")
    def m(grain, *p):
        return mid(grain, site, *p)

    spark.sql(f"DELETE FROM {S}.observation_period WHERE site_id='{site_esc}'")
    spark.sql(f"""
    INSERT INTO {S}.observation_period (observation_period_id, person_id,
      observation_period_start_date, observation_period_end_date, period_type_concept_id, site_id)
    WITH ev AS (
      SELECT person_id, condition_start_date AS d FROM {S}.condition_occurrence WHERE site_id='{site_esc}'
      UNION ALL SELECT person_id, measurement_date FROM {S}.measurement          WHERE site_id='{site_esc}'
      UNION ALL SELECT person_id, observation_date FROM {S}.observation          WHERE site_id='{site_esc}'
      UNION ALL SELECT person_id, procedure_date   FROM {S}.procedure_occurrence WHERE site_id='{site_esc}'
      -- T6-carry-B: obs_period must also see visits + episodes so a person whose earliest/latest
      -- activity is a visit or episode gets correct bounds (build_cdm runs obs_period AFTER Task 7).
      UNION ALL SELECT person_id, visit_start_date   FROM {S}.visit_occurrence WHERE site_id='{site_esc}'
      UNION ALL SELECT person_id, visit_end_date     FROM {S}.visit_occurrence WHERE site_id='{site_esc}'
      UNION ALL SELECT person_id, episode_start_date FROM {S}.episode          WHERE site_id='{site_esc}'
      UNION ALL SELECT person_id, episode_end_date   FROM {S}.episode          WHERE site_id='{site_esc}'
    ),
    agg AS (
      SELECT person_id, min(d) AS mn, max(d) AS mx FROM ev WHERE d IS NOT NULL GROUP BY person_id
    ),
    pf AS (
      SELECT person_id, min(pregnancy_start_date) AS ps,
             max(coalesce(pregnancy_end_date, pregnancy_start_date)) AS pe
      FROM {S}.pregnancy WHERE site_id='{site_esc}' GROUP BY person_id
    )
    SELECT {m('obs-period', 'p.person_id')} AS observation_period_id, p.person_id,
      coalesce(a.mn, pf.ps, CAST(p.birth_datetime AS DATE)) AS observation_period_start_date,
      coalesce(a.mx, pf.pe, CAST(p.birth_datetime AS DATE)) AS observation_period_end_date,
      32817 AS period_type_concept_id, '{site_esc}' AS site_id
    FROM {S}.person p
    LEFT JOIN agg a ON a.person_id = p.person_id
    LEFT JOIN pf     ON pf.person_id = p.person_id
    WHERE p.site_id='{site_esc}'
    """)
    n = spark.sql(f"SELECT count(*) c FROM {S}.observation_period WHERE site_id='{site_esc}'").first()["c"]
    pc = spark.sql(f"SELECT count(*) c FROM {S}.person WHERE site_id='{site_esc}'").first()["c"]
    nn = spark.sql(f"""SELECT count(*) c FROM {S}.observation_period
      WHERE site_id='{site_esc}' AND (observation_period_start_date IS NULL OR observation_period_end_date IS NULL)""").first()["c"]
    assert nn == 0, f"{nn} observation_period rows with NULL dates"
    assert n == pc, f"observation_period coverage {n} != persons {pc}"
    print(f"observation_period rows {n} | persons {pc} | null-date {nn}")


# === write_cdm_source (Task 8) — one deterministic cdm_source row (OMOP CDM v5.4) ===
# RELEASE_DATE is a module-level constant so source_release_date/cdm_release_date are DETERMINISTIC
# and reproducible (NO current_date/now). A driver may set globals()["RELEASE_DATE"] before build.
RELEASE_DATE = "2026-07-08"

def write_cdm_source(schema):  # TASK 8
    """Write generic, multi-site PREBORN metadata into cdm_source."""
    rel = globals().get("RELEASE_DATE", "2026-07-08")
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(rel)):
        raise ValueError(f"invalid RELEASE_DATE: {rel}")
    spark.sql(f"""
      INSERT INTO {schema}.cdm_source BY NAME
      REPLACE WHERE true
      SELECT
        'PREBORN multi-site maternity OMOP subset' AS cdm_source_name,
        'PREBORN' AS cdm_source_abbreviation,
        'PREBORN participating NHS organisations' AS cdm_holder,
        'Multi-site maternity-focused OMOP CDM v5.4 subset: MSDS common core with optional site-specific enrichment'
          AS source_description,
        CAST(NULL AS STRING) AS source_documentation_reference,
        CAST(NULL AS STRING) AS cdm_etl_reference,
        DATE'{rel}' AS source_release_date, DATE'{rel}' AS cdm_release_date,
        'v5.4' AS cdm_version, 756265 AS cdm_version_concept_id,
        (SELECT vocabulary_version FROM 4_prod.omop.vocabulary WHERE vocabulary_id='None')
          AS vocabulary_version
    """)
    print("write_cdm_source ->", schema, "| release", rel)


# === publish_site (Task 8) — locked per-table replacement with durable rollback/recovery journal ===
_PUBLISH_TABLES = [
    "person", "observation_period",
    "condition_occurrence", "procedure_occurrence", "measurement", "observation",
    "visit_occurrence", "visit_detail",
    "episode", "episode_event", "fact_relationship",
    "pregnancy", "infant", "_quarantine",
]
_VOCAB_TABLES = [
    "concept", "concept_relationship", "concept_ancestor", "concept_synonym",
    "vocabulary", "domain", "concept_class", "relationship",
]

def _table_columns(schema, table):
    cat, sch = schema.split(".", 1)
    rows = spark.sql(f"""
      SELECT column_name FROM {cat}.information_schema.columns
      WHERE table_schema='{sch}' AND table_name='{table}'
      ORDER BY ordinal_position""").collect()
    cols = [r["column_name"] for r in rows]
    if not cols:
        raise RuntimeError(f"no columns found for {schema}.{table}")
    return cols

def _ensure_publish_controls(target):
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {target}._publish_lock (
      lock_name STRING, token STRING, purpose STRING, acquired_at TIMESTAMP
    ) USING DELTA""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {target}._publish_journal (
      publish_token STRING, site_id STRING, table_name STRING, version_before BIGINT,
      state STRING, started_at TIMESTAMP, finished_at TIMESTAMP, detail STRING
    ) USING DELTA""")
    apply_preborn_comments(target, tables=["_publish_lock", "_publish_journal"])


def _acquire_write_lock(target, purpose):
    import uuid
    _ensure_publish_controls(target)
    token = uuid.uuid4().hex
    purpose_esc = str(purpose).replace("'", "''")
    # The publisher heartbeats after each table. A stale lock can therefore be reclaimed safely.
    spark.sql(f"""DELETE FROM {target}._publish_lock
      WHERE lock_name='exclusive' AND acquired_at < current_timestamp() - INTERVAL 2 HOURS""")
    spark.sql(f"""INSERT INTO {target}._publish_lock
      VALUES ('exclusive','{token}','{purpose_esc}',current_timestamp())""")
    rows = spark.sql(f"""SELECT token FROM {target}._publish_lock
      WHERE lock_name='exclusive'""").collect()
    if len(rows) != 1 or rows[0]["token"] != token:
        spark.sql(f"DELETE FROM {target}._publish_lock WHERE token='{token}'")
        raise RuntimeError(f"another PREBORN writer holds the lock on {target}")
    return token


def _heartbeat_write_lock(target, token):
    spark.sql(f"""UPDATE {target}._publish_lock SET acquired_at=current_timestamp()
      WHERE lock_name='exclusive' AND token='{token}'""")


def _release_write_lock(target, token):
    spark.sql(f"DELETE FROM {target}._publish_lock WHERE token='{token}'")


def _latest_version(full_table):
    row = spark.sql(f"DESCRIBE HISTORY {full_table} LIMIT 1").first()
    return int(row["version"])


def _restore_versions(target, versions):
    errors = []
    for table, version in reversed(list(versions.items())):
        try:
            spark.sql(f"RESTORE TABLE {target}.{table} TO VERSION AS OF {version}")
        except Exception as ex:
            errors.append(f"{table}: {type(ex).__name__}: {ex}")
    return errors


def _recover_incomplete_publishes(target):
    """Restore any publish interrupted before COMMITTED was durably recorded."""
    _ensure_publish_controls(target)
    rows = spark.sql(f"""SELECT publish_token, table_name, version_before
      FROM {target}._publish_journal
      WHERE state IN ('IN_PROGRESS','RECOVERY_FAILED')
      ORDER BY started_at, publish_token, table_name""").collect()
    tokens = []
    for row in rows:
        if row["publish_token"] not in tokens:
            tokens.append(row["publish_token"])
    for old_token in tokens:
        versions = {
            row["table_name"]: int(row["version_before"])
            for row in rows if row["publish_token"] == old_token
        }
        errors = _restore_versions(target, versions)
        tok = str(old_token).replace("'", "''")
        if errors:
            detail = "; ".join(errors).replace("'", "''")[:8000]
            spark.sql(f"""UPDATE {target}._publish_journal
              SET state='RECOVERY_FAILED', finished_at=current_timestamp(), detail='{detail}'
              WHERE publish_token='{tok}'""")
            raise RuntimeError(f"failed to recover interrupted PREBORN publish {old_token}: {errors}")
        spark.sql(f"""UPDATE {target}._publish_journal
          SET state='RECOVERED', finished_at=current_timestamp(),
              detail='restored automatically before next writer'
          WHERE publish_token='{tok}'""")
        print("recovered interrupted PREBORN publish", old_token)


def recover_incomplete_publish(target):
    """Manually recover any interrupted publish under the exclusive writer lock."""
    token = _acquire_write_lock(target, "recover interrupted publish")
    try:
        _recover_incomplete_publishes(target)
    finally:
        _release_write_lock(target, token)


def _record_publish_journal(target, token, site, versions):
    tok = str(token).replace("'", "''")
    site_esc = str(site).replace("'", "''")
    values = ",\n".join(
        f"('{tok}','{site_esc}','{table}',{version},'IN_PROGRESS',current_timestamp(),NULL,NULL)"
        for table, version in versions.items()
    )
    spark.sql(f"""INSERT INTO {target}._publish_journal
      (publish_token,site_id,table_name,version_before,state,started_at,finished_at,detail)
      VALUES {values}""")


def _mark_publish_journal(target, token, state, detail):
    tok = str(token).replace("'", "''")
    state_esc = str(state).replace("'", "''")
    detail_esc = str(detail).replace("'", "''")[:8000]
    spark.sql(f"""UPDATE {target}._publish_journal
      SET state='{state_esc}', finished_at=current_timestamp(), detail='{detail_esc}'
      WHERE publish_token='{tok}'""")


def publish_site(target, pub, site, post_validate=None):  # TASK 8
    """Publish one site with per-table atomic replacement and durable crash recovery."""
    ensure_preborn_schema(target)
    apply_preborn_comments(target)
    site_esc = str(site).replace("'", "''")
    token = _acquire_write_lock(target, f"publish {site}")
    versioned = _PUBLISH_TABLES + ["cdm_source"] + _VOCAB_TABLES
    versions = {}
    journaled = False
    try:
        _recover_incomplete_publishes(target)
        versions = {t: _latest_version(f"{target}.{t}") for t in versioned}
        _record_publish_journal(target, token, site, versions)
        journaled = True

        for t in _PUBLISH_TABLES:
            cols = ", ".join(_table_columns(target, t))
            spark.sql(f"""INSERT INTO {target}.{t} BY NAME
              REPLACE WHERE site_id='{site_esc}'
              SELECT {cols} FROM {pub}.{t} WHERE site_id='{site_esc}'""")
            expected = spark.sql(f"""SELECT count(*) c FROM {pub}.{t}
              WHERE site_id='{site_esc}'""").first()["c"]
            actual = spark.sql(f"""SELECT count(*) c FROM {target}.{t}
              WHERE site_id='{site_esc}'""").first()["c"]
            if actual != expected:
                raise RuntimeError(f"publish count mismatch {t}: target={actual}, pub={expected}")
            _heartbeat_write_lock(target, token)
            print(f"  published {t}: {actual} rows (site {site})")

        spark.sql(f"""INSERT INTO {target}.cdm_source BY NAME
          REPLACE WHERE true SELECT * FROM {pub}.cdm_source""")
        _heartbeat_write_lock(target, token)

        build_used_vocabulary(target)
        _heartbeat_write_lock(target, token)
        if post_validate is not None:
            post_validate()
        _mark_publish_journal(target, token, "COMMITTED", "publish and post-validation succeeded")
        print("publish_site DONE:", pub, "->", target, "| site", site)
    except Exception as ex:
        rollback_errors = _restore_versions(target, versions) if versions else []
        if journaled:
            if rollback_errors:
                _mark_publish_journal(target, token, "RECOVERY_FAILED",
                                      f"publish failed: {type(ex).__name__}: {ex}; rollback errors: {rollback_errors}")
            else:
                _mark_publish_journal(target, token, "ROLLED_BACK",
                                      f"publish failed and was restored: {type(ex).__name__}: {ex}")
        if rollback_errors:
            raise RuntimeError(
                f"publish failed ({type(ex).__name__}: {ex}); rollback errors: {rollback_errors}"
            ) from ex
        raise
    finally:
        try:
            _release_write_lock(target, token)
        except Exception as lock_ex:
            print("WARNING: publish lock release failed; stale-lock recovery will handle it:", lock_ex)


def build_used_vocabulary(schema):  # TASK 9
    """Build the filtered, FK-intact bundled vocabulary in `schema` from 4_prod.omop.*.
    Used set = DISTINCT non-zero concept_ids across every concept-carrying column of the populated
    clinical + extension tables (+ cdm_source.cdm_version_concept_id). Closure (iterated to a fixed
    point) adds: ancestors of the used set (concept_ancestor, a transitive table), the metadata-table
    FK concepts (vocabulary/domain/concept_class/relationship *_concept_id) and concept_synonym
    language concepts. concept_relationship is TWO-SIDED filtered (both endpoints in the built concept
    -> no dangling endpoint, and not a growth source); concept_ancestor keeps full ancestry of the
    used set with ancestor also in concept; vocabulary/domain/concept_class/relationship are FULL
    copies. Prints used/concept counts + a full FK-closure self-check and ASSERTS 0 dangling."""
    S = schema
    P = "4_prod.omop"
    # concept-carrying columns per populated table (matches ensure_preborn_schema / canonical v5.4 DDL)
    COLS = {
      "person": ["gender_concept_id", "race_concept_id", "ethnicity_concept_id",
                 "gender_source_concept_id", "race_source_concept_id", "ethnicity_source_concept_id"],
      "observation_period": ["period_type_concept_id"],
      "condition_occurrence": ["condition_concept_id", "condition_type_concept_id",
                               "condition_status_concept_id", "condition_source_concept_id"],
      "procedure_occurrence": ["procedure_concept_id", "procedure_type_concept_id",
                               "modifier_concept_id", "procedure_source_concept_id"],
      "measurement": ["measurement_concept_id", "measurement_type_concept_id", "operator_concept_id",
                      "value_as_concept_id", "unit_concept_id", "measurement_source_concept_id",
                      "unit_source_concept_id", "meas_event_field_concept_id"],
      "observation": ["observation_concept_id", "observation_type_concept_id", "value_as_concept_id",
                      "qualifier_concept_id", "unit_concept_id", "observation_source_concept_id",
                      "obs_event_field_concept_id"],
      "visit_occurrence": ["visit_concept_id", "visit_type_concept_id", "visit_source_concept_id",
                           "admitted_from_concept_id", "discharged_to_concept_id"],
      "visit_detail": ["visit_detail_concept_id", "visit_detail_type_concept_id",
                       "visit_detail_source_concept_id", "admitted_from_concept_id",
                       "discharged_to_concept_id"],
      "episode": ["episode_concept_id", "episode_object_concept_id", "episode_type_concept_id",
                  "episode_source_concept_id"],
      "episode_event": ["episode_event_field_concept_id"],
      "fact_relationship": ["domain_concept_id_1", "domain_concept_id_2", "relationship_concept_id"],
      "pregnancy": ["pregnancy_outcome_concept_id", "pregnancy_mode_delivery_concept_id",
                    "pregnancy_single_concept_id", "pregnancy_marital_status_concept_id",
                    "prev_pregnancy_parity_concept_id", "pregnancy_folic_concept_id"],
      "infant": ["birth_outcome_concept_id", "birth_con_malformation_concept_id"],
      "cdm_source": ["cdm_version_concept_id"],
    }
    sel = []
    for t, cols in COLS.items():
        for c in cols:
            sel.append(f"SELECT CAST({c} AS BIGINT) AS cid FROM {S}.{t}")
    union_sql = "\nUNION ALL\n".join(sel)
    spark.sql(f"""CREATE OR REPLACE TABLE {S}._vocab_used AS
      SELECT DISTINCT cid FROM ({union_sql}) WHERE cid IS NOT NULL AND cid > 0""")
    used_n = spark.table(f"{S}._vocab_used").count()

    # seed closure: used  UNION  ancestors(used)  UNION  metadata-FK concepts
    spark.sql(f"""CREATE OR REPLACE TABLE {S}._vocab_ids AS
      SELECT cid FROM {S}._vocab_used
      UNION
      SELECT CAST(ca.ancestor_concept_id AS BIGINT) FROM {P}.concept_ancestor ca
        JOIN {S}._vocab_used u ON ca.descendant_concept_id = u.cid
      UNION
      SELECT CAST(vocabulary_concept_id AS BIGINT) FROM {P}.vocabulary WHERE vocabulary_concept_id > 0
      UNION
      SELECT CAST(domain_concept_id AS BIGINT) FROM {P}.domain WHERE domain_concept_id > 0
      UNION
      SELECT CAST(concept_class_concept_id AS BIGINT) FROM {P}.concept_class WHERE concept_class_concept_id > 0
      UNION
      SELECT CAST(relationship_concept_id AS BIGINT) FROM {P}.relationship WHERE relationship_concept_id > 0""")

    # iterate to fixed point: add concept_synonym language concepts for included concepts
    iters = 0
    while iters < 5:
        iters += 1
        prev = spark.table(f"{S}._vocab_ids").count()
        spark.sql(f"""CREATE OR REPLACE TABLE {S}._vocab_ids_next AS
          SELECT cid FROM {S}._vocab_ids
          UNION
          SELECT CAST(cs.language_concept_id AS BIGINT)
          FROM {P}.concept_synonym cs
          JOIN {S}._vocab_ids v ON cs.concept_id = v.cid
          WHERE cs.language_concept_id IS NOT NULL AND cs.language_concept_id > 0""")
        spark.sql(f"CREATE OR REPLACE TABLE {S}._vocab_ids AS SELECT cid FROM {S}._vocab_ids_next")
        now = spark.table(f"{S}._vocab_ids").count()
        if now == prev:
            break
    spark.sql(f"DROP TABLE IF EXISTS {S}._vocab_ids_next")

    # === WRITE the bundled vocab (column lists + order per ensure_preborn_schema / v5.4 DDL) ===
    spark.sql(f"""CREATE OR REPLACE TABLE {S}.concept AS
      SELECT CAST(c.concept_id AS BIGINT) AS concept_id, c.concept_name, c.domain_id, c.vocabulary_id,
             c.concept_class_id, c.standard_concept, c.concept_code,
             c.valid_start_date, c.valid_end_date, c.invalid_reason
      FROM {P}.concept c JOIN {S}._vocab_ids v ON c.concept_id = v.cid""")

    spark.sql(f"""CREATE OR REPLACE TABLE {S}.concept_relationship AS
      SELECT CAST(cr.concept_id_1 AS BIGINT) AS concept_id_1, CAST(cr.concept_id_2 AS BIGINT) AS concept_id_2,
             cr.relationship_id, cr.valid_start_date, cr.valid_end_date, cr.invalid_reason
      FROM {P}.concept_relationship cr
      WHERE cr.concept_id_1 IN (SELECT concept_id FROM {S}.concept)
        AND cr.concept_id_2 IN (SELECT concept_id FROM {S}.concept)""")

    spark.sql(f"""CREATE OR REPLACE TABLE {S}.concept_ancestor AS
      SELECT CAST(ca.ancestor_concept_id AS BIGINT) AS ancestor_concept_id,
             CAST(ca.descendant_concept_id AS BIGINT) AS descendant_concept_id,
             CAST(ca.min_levels_of_separation AS BIGINT) AS min_levels_of_separation,
             CAST(ca.max_levels_of_separation AS BIGINT) AS max_levels_of_separation
      FROM {P}.concept_ancestor ca
      WHERE ca.descendant_concept_id IN (SELECT cid FROM {S}._vocab_used)
        AND ca.ancestor_concept_id IN (SELECT concept_id FROM {S}.concept)""")

    spark.sql(f"""CREATE OR REPLACE TABLE {S}.concept_synonym AS
      SELECT CAST(cs.concept_id AS BIGINT) AS concept_id, cs.concept_synonym_name,
             CAST(cs.language_concept_id AS BIGINT) AS language_concept_id
      FROM {P}.concept_synonym cs
      WHERE cs.concept_id IN (SELECT concept_id FROM {S}.concept)""")

    spark.sql(f"""CREATE OR REPLACE TABLE {S}.vocabulary AS
      SELECT vocabulary_id, vocabulary_name, vocabulary_reference, vocabulary_version,
             CAST(vocabulary_concept_id AS BIGINT) AS vocabulary_concept_id
      FROM {P}.vocabulary""")

    spark.sql(f"""CREATE OR REPLACE TABLE {S}.domain AS
      SELECT domain_id, domain_name, CAST(domain_concept_id AS BIGINT) AS domain_concept_id
      FROM {P}.domain""")

    spark.sql(f"""CREATE OR REPLACE TABLE {S}.concept_class AS
      SELECT concept_class_id, concept_class_name,
             CAST(concept_class_concept_id AS BIGINT) AS concept_class_concept_id
      FROM {P}.concept_class""")

    spark.sql(f"""CREATE OR REPLACE TABLE {S}.relationship AS
      SELECT relationship_id, relationship_name, is_hierarchical, defines_ancestry,
             reverse_relationship_id, CAST(relationship_concept_id AS BIGINT) AS relationship_concept_id
      FROM {P}.relationship""")

    concept_n = spark.table(f"{S}.concept").count()

    # === FK-closure self-check (all counts must be 0) ===
    checks = [
      ("concept_relationship.concept_id_1 in concept",
       f"SELECT count(*) c FROM {S}.concept_relationship r LEFT JOIN {S}.concept c ON c.concept_id=r.concept_id_1 WHERE c.concept_id IS NULL"),
      ("concept_relationship.concept_id_2 in concept",
       f"SELECT count(*) c FROM {S}.concept_relationship r LEFT JOIN {S}.concept c ON c.concept_id=r.concept_id_2 WHERE c.concept_id IS NULL"),
      ("concept_ancestor.ancestor in concept",
       f"SELECT count(*) c FROM {S}.concept_ancestor a LEFT JOIN {S}.concept c ON c.concept_id=a.ancestor_concept_id WHERE c.concept_id IS NULL"),
      ("concept_ancestor.descendant in concept",
       f"SELECT count(*) c FROM {S}.concept_ancestor a LEFT JOIN {S}.concept c ON c.concept_id=a.descendant_concept_id WHERE c.concept_id IS NULL"),
      ("vocabulary.vocabulary_concept_id in concept",
       f"SELECT count(*) c FROM {S}.vocabulary v LEFT JOIN {S}.concept c ON c.concept_id=v.vocabulary_concept_id WHERE v.vocabulary_concept_id>0 AND c.concept_id IS NULL"),
      ("domain.domain_concept_id in concept",
       f"SELECT count(*) c FROM {S}.domain d LEFT JOIN {S}.concept c ON c.concept_id=d.domain_concept_id WHERE d.domain_concept_id>0 AND c.concept_id IS NULL"),
      ("concept_class.concept_class_concept_id in concept",
       f"SELECT count(*) c FROM {S}.concept_class k LEFT JOIN {S}.concept c ON c.concept_id=k.concept_class_concept_id WHERE k.concept_class_concept_id>0 AND c.concept_id IS NULL"),
      ("relationship.relationship_concept_id in concept",
       f"SELECT count(*) c FROM {S}.relationship rl LEFT JOIN {S}.concept c ON c.concept_id=rl.relationship_concept_id WHERE rl.relationship_concept_id>0 AND c.concept_id IS NULL"),
      ("concept_synonym.concept_id in concept",
       f"SELECT count(*) c FROM {S}.concept_synonym s LEFT JOIN {S}.concept c ON c.concept_id=s.concept_id WHERE c.concept_id IS NULL"),
      ("concept_synonym.language_concept_id in concept",
       f"SELECT count(*) c FROM {S}.concept_synonym s LEFT JOIN {S}.concept c ON c.concept_id=s.language_concept_id WHERE s.language_concept_id>0 AND c.concept_id IS NULL"),
      ("cdm_source.cdm_version_concept_id in concept",
       f"SELECT count(*) c FROM {S}.cdm_source cs LEFT JOIN {S}.concept c ON c.concept_id=cs.cdm_version_concept_id WHERE cs.cdm_version_concept_id>0 AND c.concept_id IS NULL"),
      ("concept.domain_id in domain",
       f"SELECT count(*) c FROM {S}.concept c LEFT JOIN {S}.domain d ON d.domain_id=c.domain_id WHERE d.domain_id IS NULL"),
      ("concept.vocabulary_id in vocabulary",
       f"SELECT count(*) c FROM {S}.concept c LEFT JOIN {S}.vocabulary v ON v.vocabulary_id=c.vocabulary_id WHERE v.vocabulary_id IS NULL"),
      ("concept.concept_class_id in concept_class",
       f"SELECT count(*) c FROM {S}.concept c LEFT JOIN {S}.concept_class k ON k.concept_class_id=c.concept_class_id WHERE k.concept_class_id IS NULL"),
    ]
    dangling = 0
    print(f"build_used_vocabulary: schema={S} used_concepts={used_n} concept_rows={concept_n} closure_iters={iters}")
    for name, q in checks:
        n = spark.sql(q).first()["c"]
        dangling += n
        print(f"  FK-check [{'OK ' if n == 0 else 'FAIL'}] {name}: {n} dangling")
    spark.sql(f"DROP TABLE IF EXISTS {S}._vocab_used")
    spark.sql(f"DROP TABLE IF EXISTS {S}._vocab_ids")
    assert dangling == 0, f"build_used_vocabulary: {dangling} dangling FK(s) in bundled vocab for {S}"
    apply_preborn_comments(S, tables=_VOCAB_TABLES)
    print(f"build_used_vocabulary DONE: {S} | vocab FK closure 0-dangling")


def rebuild_used_vocabulary_safely(schema):
    """Rebuild live vocabulary tables under the exclusive writer lock with rollback."""
    token = _acquire_write_lock(schema, "vocabulary rebuild")
    versions = {}
    try:
        _recover_incomplete_publishes(schema)
        versions = {t: _latest_version(f"{schema}.{t}") for t in _VOCAB_TABLES}
        build_used_vocabulary(schema)
    except Exception as ex:
        rollback_errors = _restore_versions(schema, versions) if versions else []
        if rollback_errors:
            raise RuntimeError(
                f"vocabulary rebuild failed ({type(ex).__name__}: {ex}); "
                f"rollback errors: {rollback_errors}"
            ) from ex
        raise
    finally:
        _release_write_lock(schema, token)


In [0]:
# === build_cdm — staging orchestrator (Task 4) ===
# Takes the FINAL `target` (e.g. 8_dev.preborn) and derives a run-unique publish-staging schema
# pub = f"{target}_pub_{runtag}" with the full PREBORN shape. All layer-merges (Tasks 6-7) write
# into `pub`, NEVER directly into `target`; build_cdm RETURNS pub; validate runs against pub; only
# publish_site(target, pub, site) (Task 8) moves validated rows into target — the validate-before-
# publish guarantee. The transform working schemas ({pub}_stg, _stg_l3, _stg_enrich) are internal
# to the build and dropped by drop_working after the merge. Call order matters:
#   - conform_person establishes {stg}._eligible BEFORE the fact merge (finding #4)


def _schema_exists(schema):
    cat, sch = schema.split(".", 1)
    sch_esc = sch.replace("'", "''")
    return spark.sql(f"SHOW SCHEMAS IN `{cat}` LIKE '{sch_esc}'").count() > 0

def build_cdm(target, site, src, include_enrichment, cerner=None, race_case=None, runtag="r"):
    runtag = str(runtag).lower()
    if not re.fullmatch(r"[a-z0-9_]+", runtag):
        raise ValueError(f"invalid runtag {runtag!r}; use lowercase letters, digits, and underscores")
    pub = f"{target}_pub_{runtag}"
    stg = f"{pub}_stg"
    if _schema_exists(pub) or _schema_exists(stg) or _schema_exists(stg + "_l3") or _schema_exists(stg + "_enrich"):
        raise RuntimeError(f"runtag collision: staging already exists for {runtag}")
    ensure_preborn_schema(pub)
    msds_core_transform(site, src, stg, race_case=race_case)
    if include_enrichment:
        if cerner is None:
            raise ValueError("enrichment requires cerner sources")
        _build_barts_enrichment(site, src, cerner, stg)
    conform_person(pub, site, stg)
    _merge_to_cdm(pub, site, stg, include_enrichment)
    build_observation_period(pub, site)
    write_cdm_source(pub)
    build_used_vocabulary(pub)
    print("build_cdm staged into", pub, "for", site)
    return pub


In [0]:
# === drop_working — remove the complete run-scoped staging estate after successful publish ===

def drop_working(target, runtag):
    """Drop all run-scoped staging, including the validated publish schema, after success."""
    pub = f"{target}_pub_{str(runtag).lower()}"
    for schema in [f"{pub}_stg_enrich", f"{pub}_stg_l3", f"{pub}_stg", pub]:
        spark.sql(f"DROP SCHEMA IF EXISTS {schema} CASCADE")